In [0]:
# Databricks notebook source
# MAGIC %pip install geopy
# MAGIC dbutils.library.restartPython()

# COMMAND ----------

# MAGIC %md
# MAGIC # 02 — Silver Layer: Clean, Parse & Enrich (v12 — Full Fix + Semantic Hardening)
# MAGIC
# MAGIC **Input**: `virtue_foundation.ghana.bronze_facilities_raw`
# MAGIC **Output**: `virtue_foundation.ghana.silver_facilities_cleaned`
# MAGIC
# MAGIC ## What changed in v12 (vs v11):
# MAGIC
# MAGIC | Fix | Description |
# MAGIC |-----|-------------|
# MAGIC | v12-1  | `_MEANINGFUL_ADDR_RE` — abbreviated road types (Rd, St, Ave, Ln, Dr, Ct) now recognised; `Liberation Rd. 37` no longer False |
# MAGIC | v12-2  | `address_type` — new tri-state field: `physical`, `postal`, `mixed` |
# MAGIC | v12-3  | Org category — 50+ patterns; government/faith/ngo all dramatically improved |
# MAGIC | v12-4  | Phone — post-normalisation E.164 length gate; bad digits rejected |
# MAGIC | v12-5  | Operational status — negation handling; "accepting patients" / "24 hours" patterns |
# MAGIC | v12-6  | Numeric extraction — `organizationdescription` added as explicit source; year regex expanded |
# MAGIC | v12-7  | Geocoding — geopy Nominatim batch fallback for country-centroid rows (Section 16.5) |
# MAGIC | v12-8  | Campus-aware dedup — sub-facility detection regex; canonical override only on exact name match |
# MAGIC | v12-9  | Canonical rows get `is_generated_canonical` + `canonical_source_group` provenance flags |
# MAGIC | v12-10 | Contradiction flags — city/region mismatch, org/operator conflict, address/type conflict |
# MAGIC | v12-11 | Quality flags null → `[]`; dedup survivor score weights `source_trust` |
# MAGIC | v12-12 | `city_clean` backfill from address fragments before region resolution |
# MAGIC | v12-13 | Dedup survivor ordering: `source_trust=structured` beats `social_metadata` |
# MAGIC | v12-14 | `LOW:weak_facility_type` flag threshold moved to confidence < 0.45 |

# COMMAND ----------
# MAGIC %md ## 0. Imports & Config

# COMMAND ----------

import json
import re
import html
import time
from typing import Optional, List, Dict, Tuple
from datetime import datetime, timezone

import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StringType, IntegerType, FloatType, BooleanType,
    ArrayType, DoubleType, StructType, StructField,
)
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf

spark = SparkSession.builder.getOrCreate()
print(f"Run: {datetime.now(timezone.utc).isoformat()}")
print(f"Spark: {spark.version}")

# COMMAND ----------

class Config:
    BRONZE         = "virtue_foundation.ghana.bronze_facilities_raw"
    SILVER         = "virtue_foundation.ghana.silver_facilities_cleaned"
    SILVER_AUDIT   = "virtue_foundation.ghana.silver_facilities_audit"
    EXTRACTION_VER = "silver_v13"

    DEFAULT_LAT = 7.9465
    DEFAULT_LON = -1.0232

    MAX_DOCTORS = 1000
    MAX_BEDS    = 5000
    MAX_AREA    = 100_000
    MIN_YEAR    = 1500
    MAX_YEAR    = 2026

cfg = Config()
spark.sql("CREATE DATABASE IF NOT EXISTS virtue_foundation.ghana")

PDF_FACILITY_TYPE_ENUM = frozenset({"hospital", "pharmacy", "doctor", "clinic", "dentist"})

EXTENDED_TO_PDF_ENUM: Dict[str, str] = {
    "hospital":             "hospital",
    "clinic":               "clinic",
    "pharmacy":             "pharmacy",
    "dentist":              "dentist",
    "doctor":               "doctor",
    "chps":                 "clinic",
    "maternity_home":       "clinic",
    "eye_clinic":           "clinic",
    "diagnostic_center":    "clinic",
    "alternative_medicine": "clinic",
}

FDR_SPECIALTY_VOCAB = frozenset({
    "internalMedicine", "familyMedicine", "pediatrics", "cardiology",
    "generalSurgery", "emergencyMedicine", "gynecologyAndObstetrics",
    "orthopedicSurgery", "dentistry", "ophthalmology", "radiology",
    "pathology", "infectiousDiseases", "nephrology", "criticalCareMedicine",
    "cardiacSurgery", "plasticSurgery", "neurology", "psychiatry",
    "anesthesia", "dermatology", "urology", "gastroenterology",
    "pulmonology", "endocrinologyAndDiabetesAndMetabolism",
    "neonatologyPerinatalMedicine", "medicalOncology",
    "physicalMedicineAndRehabilitation", "otolaryngology",
    "geriatricsInternalMedicine", "hospiceAndPalliativeInternalMedicine",
    "publicHealth", "globalHealthAndInternationalHealth",
    "clinicalPathology", "obstetricsAndMaternityCare",
    "reproductiveEndocrinologyAndInfertility",
    "maternalFetalMedicineOrPerinatology", "socialAndBehavioralSciences",
    "orthodontics", "familyPlanningAndComplexContraception",
    "sportsMedicine", "craniofacialSurgery", "oralAndMaxillofacialSurgery",
})

_VALID_ISO2: frozenset = frozenset({
    "GH","NG","CI","SN","ML","BF","TG","BJ","NE","GN","LR","SL",
    "GM","GW","CM","GA","CG","CD","AO","ZA","KE","TZ","UG","RW",
    "ET","SD","SO","MZ","ZW","ZM","MW","MG","MR","MA","TN","EG",
    "LY","DZ","GB","US","DE","FR","IT","NL","SE","NO","DK","CH",
    "CA","AU","IN","CN","JP","BR","AR","MX",
})

# COMMAND ----------
# MAGIC %md ## 1. Read Bronze + Preserve Raw Fields

# COMMAND ----------

bronze = spark.table(cfg.BRONZE)
print(f"Bronze rows    : {bronze.count():,}")
print(f"Bronze columns : {len(bronze.columns)}")

bronze = bronze \
    .withColumn("facility_type_raw", F.col("facilitytypeid")) \
    .withColumn("operator_type_raw", F.col("operatortypeid"))
# COMMAND ----------
# MAGIC %md ## 1.5. Name Cleaning

# COMMAND ----------

_NAME_PLAIN_DELIMITER = re.compile(r"\s+[-–—]\s+")
_NAME_IS_ADDRESS_RE = re.compile(
    r"(?i)^("
    r"(?:\d+/)?No\.?\s*\d+"
    r"|#\d+"
    r"|\d+\s+[\w\s]+\s+(?:Rd|Road|Street|St|Ave|Avenue|Lane|Drive|Blvd|Close)"
    r"|P\.?\s*O\.?\s*Box\s+\d+"
    r"|PMB\s+\d+"
    r"|GPS\s*[:\-]?\s*[A-Z]{2,3}-\d{3,4}"
    r")"
)

@pandas_udf(ArrayType(StringType()))
def clean_name_and_extract_address_udf(name: pd.Series) -> pd.Series:
    result = []
    for raw in name:
        s = str(raw or "").strip()
        if not s or s.lower() in ("null", "none", "nan"):
            result.append(["", ""]); continue
        m = _NAME_PLAIN_DELIMITER.search(s)
        if m:
            left  = s[:m.start()].strip()
            right = s[m.end():].strip()
            if re.search(r"\d|(?:Street|Road|Lane|Drive|Ave|Blvd|Rd|St|Close|Link|Ghana|Accra|Kumasi|Takoradi)", right, re.I):
                result.append([left, right]); continue
        if _NAME_IS_ADDRESS_RE.match(s):
            result.append(["UNKNOWN_FACILITY", s]); continue
        result.append([s, ""])
    return pd.Series(result)

_np = clean_name_and_extract_address_udf(F.col("name"))
bronze = bronze \
    .withColumn("_np", _np) \
    .withColumn("name",              F.col("_np").getItem(0)) \
    .withColumn("_name_addr_suffix", F.col("_np").getItem(1)) \
    .drop("_np") \
    .withColumn(
        "address_line1",
        F.when(
            (F.col("_name_addr_suffix") != "") &
            (F.col("address_line1").isNull() | (F.trim(F.col("address_line1")) == "")),
            F.col("_name_addr_suffix")
        ).otherwise(F.col("address_line1"))
    ).drop("_name_addr_suffix")

print(f"Name cleaning done. Rows: {bronze.count():,}")

# COMMAND ----------
# MAGIC %md ## 2. JSON Array Parser

# COMMAND ----------

def _safe_json_parse(text: Optional[str]) -> List[str]:
    if text is None: return []
    text = str(text).strip()
    if text in ("", "null", "[]", "None", "nan"): return []
    if text.replace('""', '"').strip() in ("", "[]", "null", "None", '"[]"', "'[]'"): return []

    attempts = []
    if '""' in text:         attempts.append(text.replace('""', '"'))
    if text.startswith('"[') and text.endswith(']"'): attempts.append(text[1:-1])
    if text.startswith('"[\\"'):
        try:
            inner = json.loads(text)
            if isinstance(inner, str): attempts.insert(0, inner)
        except Exception: pass
    attempts.append(text)

    for attempt in attempts:
        attempt = attempt.strip()
        if attempt in ("", "[]", "null", "None", '"[]"'): return []
        try:
            result = json.loads(attempt)
            if isinstance(result, list):
                flat = []
                for x in result:
                    if isinstance(x, list):
                        flat.extend([str(i).strip() for i in x if str(i).strip() not in ("","null","None")])
                    elif x is not None and str(x).strip() not in ("","null","None"):
                        flat.append(str(x).strip())
                return flat
            elif isinstance(result, dict): return [json.dumps(result)]
            else:
                s = str(result).strip()
                if s.startswith("[") and s != "[]":
                    try:
                        inner = json.loads(s)
                        if isinstance(inner, list):
                            return [str(x).strip() for x in inner if x is not None and str(x).strip() not in ("","null","None")]
                    except Exception: pass
                return [s] if s and s not in ("null","None","[]") else []
        except (json.JSONDecodeError, ValueError): pass

    cleaned = text.strip('"').strip("'").strip()
    if not cleaned or cleaned in ("null","None","[]"): return []
    if "," in cleaned and len(cleaned) < 500 and cleaned.count(".") <= 2 and "\n" not in cleaned:
        items = [x.strip().strip('"').strip("'") for x in cleaned.split(",")]
        items = [i for i in items if i and i not in ("null","None","[]")]
        if items and all(len(i) < 120 for i in items) and len(items) <= 20: return items
    if cleaned and cleaned not in ("null","None","[]") and len(cleaned) < 200: return [cleaned]
    return []

@pandas_udf(ArrayType(StringType()))
def safe_json_udf(col: pd.Series) -> pd.Series:
    return col.apply(_safe_json_parse)

print("JSON parser defined.")

# COMMAND ----------
# MAGIC %md ## 3. Phone Normalization (v12-4: strict post-validation)

# COMMAND ----------

_NOT_PHONE_RE = re.compile(r"(?i)(BP\s*[-–]|P\.O\.?\s*Box|box\s+\d|po\s+box)")

# v12-4: Known bad patterns — too short, too long, or obviously wrong Ghana numbers
_BAD_PHONE_RE = re.compile(
    r"(?:^\+2330{5,}|^\+233(?:0000|1234|9999)|^\+\d{16,})"  # impossible lengths / dummy numbers
)

def _normalize_e164(phone: Optional[str]) -> Optional[str]:
    if not phone: return None
    s = str(phone).strip().strip('"').strip("'")
    if not s or s.lower() in ("null","none","nan",""): return None
    if _NOT_PHONE_RE.search(s): return None
    s = re.sub(r'(?<!\w)[Oo](?=\d)', '0', s)
    s = re.sub(r'(?<=\d)[Oo](?!\w)', '0', s)
    s_clean = re.sub(r"[\s\-\.\(\)\/]", "", s)
    if s_clean.startswith("+"):
        candidate = s_clean
        if not re.fullmatch(r"\d{7,15}", s_clean[1:]): return None
    elif not re.fullmatch(r"\d+", s_clean): return None
    else:
        d = s_clean
        if len(d) == 10 and d.startswith("0") and d[1] in "23456789": candidate = "+233" + d[1:]
        elif len(d) == 9  and d[0] in "23456789":                      candidate = "+233" + d
        elif len(d) == 12 and d.startswith("233"):                      candidate = "+" + d
        elif 7 <= len(d) <= 15:                                         candidate = "+" + d
        else: return None

    # v12-4: Reject obvious dummy/bad numbers
    if _BAD_PHONE_RE.search(candidate): return None
    # Ghana numbers: +233 followed by 9 digits
    if candidate.startswith("+233") and len(candidate) != 13: return None
    return candidate

def _normalize_phone_list(raw_items: Optional[List]) -> List[str]:
    if not raw_items: return []
    seen: Dict[str, str] = {}
    for raw in raw_items:
        norm = _normalize_e164(str(raw) if raw is not None else None)
        if norm and norm not in seen: seen[norm] = norm
    return list(seen.keys())

@pandas_udf(ArrayType(StringType()))
def normalize_phone_list_udf(phones: pd.Series) -> pd.Series:
    return pd.Series([_normalize_phone_list(list(items) if items is not None else []) for items in phones])

@pandas_udf(StringType())
def extract_official_phone_udf(phones: pd.Series) -> pd.Series:
    result = []
    for items in phones:
        lst = list(items) if items is not None else []
        if not lst: result.append(None); continue
        gh = [p for p in lst if str(p).startswith("+233")]
        result.append(gh[0] if gh else lst[0])
    return pd.Series(result)

@pandas_udf(StringType())
def normalized_phone_numbers_string_udf(phones: pd.Series) -> pd.Series:
    return pd.Series([json.dumps(_normalize_phone_list(list(items) if items is not None else [])) or None for items in phones])

print("Phone normalization defined.")

# COMMAND ----------
# MAGIC %md ## 4. Parse All Array Columns + Apply Phone Normalization

# COMMAND ----------

ARRAY_COLUMNS = ["specialties","procedure","equipment","capability","phone_numbers","affiliationtypeids","countries","websites"]

df = bronze
for col in ARRAY_COLUMNS:
    df = df.withColumn(f"{col}_parsed", safe_json_udf(F.col(col)))

df = df.withColumn("phone_numbers_parsed", normalize_phone_list_udf(F.col("phone_numbers_parsed")))
print("Array parsing complete.")

# COMMAND ----------
# MAGIC %md ## 5. Renames + Country + Email + URL Protocol

# COMMAND ----------

BRONZE_TO_SILVER_RENAMES = {
    "officialwebsite":       "officialWebsite",
    "address_stateorregion": "address_stateOrRegion",
    "address_ziporpostcode": "address_zipOrPostcode",
    "address_countrycode":   "address_countryCode",
    "facilitytypeid":        "facilityTypeId",
    "operatortypeid":        "operatorTypeId",
}
for old, new in BRONZE_TO_SILVER_RENAMES.items():
    df = df.withColumnRenamed(old, new)

df = df \
    .withColumn("address_country",
        F.when(F.lower(F.trim(F.col("address_country"))).isin("gh","ghana"), F.lit("Ghana"))
         .otherwise(F.col("address_country"))) \
    .withColumn("address_countryCode",
        F.when(F.col("address_countryCode").isNull() & (F.col("address_country") == "Ghana"), F.lit("GH"))
         .when(F.col("address_countryCode").isNotNull(), F.upper(F.col("address_countryCode")))
         .otherwise(F.col("address_countryCode"))) \
    .withColumn("country_scope",
        F.coalesce(F.col("address_country"), F.col("country_scope"), F.lit("Ghana")))

@pandas_udf(StringType())
def decode_email_udf(email: pd.Series) -> pd.Series:
    def _decode(e):
        if not e or str(e).strip() in ("","null","None","nan","#ERROR!","#N/A","#REF!"): return None
        decoded = html.unescape(str(e).strip())
        return decoded.lower() if "@" in decoded and re.search(r"@\w+\.\w+", decoded) else None
    return email.apply(_decode)

df = df.withColumn("email", decode_email_udf(F.col("email")))

_EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}$")
_GENERIC_EMAIL_LOCAL = {"info", "contact", "admin", "support", "hello", "office", "mail", "enquiries"}

@pandas_udf(BooleanType())
def email_validity_udf(email: pd.Series) -> pd.Series:
    return email.apply(lambda e: True if e and _EMAIL_RE.match(str(e).strip()) else False)

@pandas_udf(BooleanType())
def email_is_generic_udf(email: pd.Series) -> pd.Series:
    def _is_generic(e):
        if not e: return False
        local = str(e).split("@", 1)[0].lower()
        return local in _GENERIC_EMAIL_LOCAL
    return email.apply(_is_generic)

@pandas_udf(DoubleType())
def email_quality_score_udf(email: pd.Series) -> pd.Series:
    def _score(e):
        if not e or not _EMAIL_RE.match(str(e).strip()): return 0.0
        local = str(e).split("@", 1)[0].lower()
        return 0.7 if local in _GENERIC_EMAIL_LOCAL else 1.0
    return email.apply(_score)

def _fix_url(u: Optional[str]) -> Optional[str]:
    if not u or str(u).strip() in ("","null","None","nan"): return None
    u = str(u).strip().rstrip("/")
    if u.startswith("https://"): return u
    if u.startswith("http://"):  return "https://" + u[7:]
    if re.match(r"^(?:www\.)?[\w\-]+\.[a-z]{2,}", u, re.I): return "https://" + u
    return u

@pandas_udf(StringType())
def normalize_url_udf(url: pd.Series) -> pd.Series:
    return url.apply(_fix_url)

@pandas_udf(StringType())
def normalize_twitter_udf(link: pd.Series) -> pd.Series:
    def _fix(u):
        if not u or str(u).strip() in ("","null","None","nan"): return None
        u = str(u).strip()
        m = re.match(r"(https?://(?:twitter\.com|x\.com)/[^/]+)(?:/status/.+)?", u)
        if m:
            base = m.group(1)
            return ("https://" + base[7:]) if base.startswith("http://") else base
        return _fix_url(u)
    return link.apply(_fix)

@pandas_udf(StringType())
def normalize_instagram_udf(link: pd.Series) -> pd.Series:
    def _fix(u):
        if not u or str(u).strip() in ("","null","None","nan"): return None
        u = str(u).strip()
        if not u.startswith("http"):
            handle = u.strip("@").strip("/")
            return f"https://instagram.com/{handle}" if "/" not in handle else f"https://{handle}"
        return _fix_url(u)
    return link.apply(_fix)

_SOCIAL_DOMAINS = ("facebook.com", "fb.com", "instagram.com", "twitter.com", "x.com", "linkedin.com")

def _is_social_url(u: Optional[str]) -> bool:
    if not u: return False
    s = str(u).lower()
    return any(d in s for d in _SOCIAL_DOMAINS)

@pandas_udf(ArrayType(StringType()))
def clean_websites_list_udf(items: pd.Series) -> pd.Series:
    out = []
    for lst in items:
        urls = list(lst) if lst is not None else []
        cleaned = []
        for u in urls:
            cu = _fix_url(u)
            if cu and not _is_social_url(cu):
                cleaned.append(cu)
        # de-dupe while preserving order
        seen = set()
        deduped = []
        for u in cleaned:
            if u not in seen:
                seen.add(u); deduped.append(u)
        out.append(deduped)
    return pd.Series(out)

@pandas_udf(StringType())
def primary_website_udf(official: pd.Series, websites: pd.Series) -> pd.Series:
    result = []
    for off, lst in zip(official, websites):
        off_u = _fix_url(off)
        if off_u and not _is_social_url(off_u):
            result.append(off_u); continue
        urls = list(lst) if lst is not None else []
        pick = None
        for u in urls:
            cu = _fix_url(u)
            if cu and not _is_social_url(cu):
                pick = cu; break
        result.append(pick)
    return pd.Series(result)

@pandas_udf(BooleanType())
def website_is_social_only_udf(official: pd.Series, websites: pd.Series) -> pd.Series:
    result = []
    for off, lst in zip(official, websites):
        off_u = _fix_url(off)
        urls = [off_u] if off_u else []
        urls += list(lst) if lst is not None else []
        urls = [u for u in urls if u]
        if not urls:
            result.append(False); continue
        result.append(all(_is_social_url(u) for u in urls))
    return pd.Series(result)

_GH_PREFIX_RE = re.compile(r"^\+233(2[0-9]|3[0-9]|5[0-9]|8[0-9]|9[0-9])\d{7}$")

@pandas_udf(StringType())
def phone_country_code_udf(phone: pd.Series) -> pd.Series:
    def _cc(p):
        if not p or not str(p).startswith("+"): return None
        m = re.match(r"^\+(\d{1,3})", str(p))
        return m.group(1) if m else None
    return phone.apply(_cc)

@pandas_udf(DoubleType())
def phone_quality_score_udf(phone: pd.Series) -> pd.Series:
    def _score(p):
        if not p: return 0.0
        s = str(p).strip()
        if _GH_PREFIX_RE.match(s): return 1.0
        if s.startswith("+233") and len(s) == 13: return 0.8
        if re.fullmatch(r"\+\d{8,15}", s): return 0.5
        return 0.0
    return phone.apply(_score)

df = df \
    .withColumn("officialWebsite", normalize_url_udf(F.col("officialWebsite"))) \
    .withColumn("twitterlink",     normalize_twitter_udf(F.col("twitterlink"))) \
    .withColumn("instagramlink",   normalize_instagram_udf(F.col("instagramlink"))) \
    .withColumn("phone_numbers",   normalized_phone_numbers_string_udf(F.col("phone_numbers_parsed"))) \
    .withColumn("websites_parsed", clean_websites_list_udf(F.col("websites_parsed")))

df = df \
    .withColumn("email_validity", email_validity_udf(F.col("email"))) \
    .withColumn("email_is_generic", email_is_generic_udf(F.col("email"))) \
    .withColumn("email_quality_score", email_quality_score_udf(F.col("email"))) \
    .withColumn("website_primary", primary_website_udf(F.col("officialWebsite"), F.col("websites_parsed"))) \
    .withColumn("website_is_social_only", website_is_social_only_udf(F.col("officialWebsite"), F.col("websites_parsed")))

print("Renames + country + email + URL protocol done.")

# COMMAND ----------
# MAGIC %md ## 5.5. Organization Category (v12-3: 50+ patterns, dramatically improved coverage)

# COMMAND ----------

# COMMAND ----------
# MAGIC %md ## 5.5. Organization Category (v13 — Triple Fallback, ~0% null)
# MAGIC
# MAGIC v13-1: After text patterns and URL hints, add a third tier:
# MAGIC   Tier 3a: facility_type_clean + operatorTypeId as signal
# MAGIC   Tier 3b: default to private_company (low confidence)
# MAGIC This ensures org_category is never null for any row with a known facility type.
 
# COMMAND ----------
 
# v13: Order matters — more specific first
_ORG_TYPE_PATTERNS: List[Tuple[re.Pattern, str, float]] = [
    # Government / public sector
    (re.compile(
        r"\b(ghana\s+health\s+service|ghs|ministry\s+of\s+health|district\s+(?:health|hospital|assembly)"
        r"|regional\s+(?:health|hospital)|municipal\s+(?:health|hospital|assembly)"
        r"|government[\s\-](?:owned|run|funded|hospital|clinic)"
        r"|public\s+(?:hospital|clinic|health|sector)\b"
        r"|police\s+(?:hospital|clinic|service)\b|military\s+hospital|ghana\s+armed\s+forces"
        r"|national\s+(?:health|hospital|blood|insurance)|ghana\s+national"
        r"|teaching\s+hospital|referral\s+hospital|district\s+hospital\b"
        r"|ghana\s+prisons?\s+service|korle\s+bu|komfo\s+anokye|ridge\s+hospital"
        r"|37\s+military|tamale\s+teaching|cape\s+coast\s+teaching|ho\s+teaching"
        r"|effia\s+nkwanta|bolgatanga\s+regional|wa\s+regional|koforidua\s+regional"
        # v13: more government patterns
        r"|accra\s+psychiatric|psychiatric\s+hospital|government\s+hospital"
        r"|\.gov\.gh|moh\.gov|ghanahealthservice\.gov"
        r"|community\s+health\s+(?:compound|planning|post)|chps\s+compound"
        r"|sub.?district\s+(?:hospital|health)|polyclinic\s+(?:no|#|\d)"
        r")\b",
        re.I), "government", 0.92),
    # NGO / non-profit
    (re.compile(
        r"\b(ngo|non[\s\-]?governmental|non[\s\-]?profit|nonprofit|foundation(?!\s+hospital)"
        r"|charitable\s+trust|charity\b|relief\s+(?:fund|organisation|organization)"
        r"|humanitarian|development\s+organisation|aid\s+organisation"
        r"|partners\s+in\s+health|msf|médecins|doctors\s+without\s+borders"
        r"|world\s+health\s+organisation|who\s+accredited|red\s+cross|red\s+crescent"
        r"|marie\s+stopes|breast\s+care\s+international|health\s+for\s+all"
        r"|international\s+health\s+care\s+center|focos|west\s+africa\s+aids"
        r"|freedom\s+aid|volunteer\s+(?:health|medical|program)\b)\b",
        re.I), "ngo", 0.88),
    # Faith-based
    (re.compile(
        r"\b(mission\s+(?:hospital|clinic|health)|diocese|catholic|presbyterian|methodist"
        r"|adventist|seventh[\s\-]day\s+adventist|sda\b|church\s+of\s+(christ|god|pentecost)"
        r"|salvation\s+army|ahmadiyya|islamic\s+(?:hospital|clinic|medical)|muslim\s+(?:hospital|clinic)"
        r"|quaker|lutheran|anglican|baptist\s+(?:hospital|clinic)"
        r"|christian\s+(?:health\s+association|hospital|clinic|mission)"
        r"|chag\b|st[\s\.]\s*(?:luke|mary|joseph|john|peter|michael|francis|george|martin|dominic|anne|theresa|augustine)"
        r"|saint\s+(?:luke|mary|joseph|john|peter|michael|francis|george)"
        r"|holy\s+(?:family|cross|spirit|trinity|rosary|ghost)\s+(?:hospital|clinic|health)"
        r"|sacred\s+heart|our\s+lady|our\s+lord|immaculate"
        r"|church\s+hospital|convent\s+(?:hospital|clinic))\b",
        re.I), "faith_based", 0.88),
    # Private company
    (re.compile(
        r"\b(private(?:ly)?\s*(?:owned|run|operated|limited|clinic|hospital)"
        r"|limited\s+liability|ltd\.?|llc\.?|plc\.?\b|inc\.?\b"
        r"|enterprise[s]?\b|company\b(?!\s+(?:hospital|clinic))"
        r"|specialist\s+(?:hospital|clinic|centre)(?!\s+(?:government|public|ngo))"
        r"|(?:dr|doctor)[\.\s]+[a-z]+['s]*\s+(?:clinic|hospital|practice)"
        r"|family\s+(?:clinic|hospital|practice)\b)\b",
        re.I), "private", 0.75),
]
 
_ORG_URL_HINTS: List[Tuple[re.Pattern, str, float]] = [
    (re.compile(r"\b(ghanahealthservice|moh\.gov|health\.gov|gov\.gh)\b", re.I), "government", 0.85),
    (re.compile(r"\b(church|catholic|presbyterian|methodist|adventist|sda|islamic|mosque)\b", re.I), "faith_based", 0.75),
    (re.compile(r"\b(ngo|foundation|charity|nonprofit)\b", re.I), "ngo", 0.70),
]
 
# v13: Tier 3 — facility-type + operator signals for remaining nulls
# These map (facility_type_lower, operator_lower) → (category, confidence)
_FTYPE_OP_CATEGORY_MAP: Dict[Tuple[str, str], Tuple[str, float]] = {
    ("hospital",   "public"):  ("government",     0.65),
    ("clinic",     "public"):  ("government",     0.60),
    ("chps",       "public"):  ("government",     0.75),
    ("hospital",   "private"): ("private",0.55),
    ("clinic",     "private"): ("private",0.55),
    ("pharmacy",   "private"): ("private",0.60),
    ("dentist",    "private"): ("private",0.60),
    ("eye_clinic", "private"): ("private",0.55),
    ("maternity_home", "private"): ("private",0.55),
    ("diagnostic_center","private"): ("private",0.55),
}
 
@pandas_udf(ArrayType(StringType()))
def normalize_org_category_udf(
    org_raw: pd.Series, name: pd.Series, desc: pd.Series,
    mission: pd.Series, source_url: pd.Series,
    facility_type: pd.Series, operator_type: pd.Series
) -> pd.Series:
    result = []
    for o, n, d, ms, url, ft, op in zip(org_raw, name, desc, mission, source_url, facility_type, operator_type):
        text = " ".join(["" if pd.isna(x) else str(x) for x in [o, n, d, ms]])
        url_s  = "" if pd.isna(url) else str(url)
        ft_s   = str(ft or "").strip().lower()
        op_s   = str(op or "").strip().lower()
 
        assigned = None
        conf     = 0.0
        raw_sig  = None
 
        # Tier 0: direct org_type field
        org_s = str(o or "").strip().lower()
        if org_s in ("ngo", "non-profit", "nonprofit", "foundation"):
            assigned, conf, raw_sig = "ngo", 0.75, "org_type"
        elif org_s in ("government", "public"):
            assigned, conf, raw_sig = "government", 0.80, "org_type"
        elif org_s in ("faith", "faith-based", "mission"):
            assigned, conf, raw_sig = "faith_based", 0.75, "org_type"
 
        # Tier 1: text patterns
        if assigned is None:
            for pattern, label, p_conf in _ORG_TYPE_PATTERNS:
                if pattern.search(text):
                    assigned, conf, raw_sig = label, p_conf, "text_pattern"
                    break
 
        # Tier 2: URL hints
        if assigned is None and url_s:
            for pattern, label, p_conf in _ORG_URL_HINTS:
                if pattern.search(url_s):
                    assigned, conf, raw_sig = label, p_conf, "source_url"
                    break
 
        # v13 Tier 3a: facility_type + operator_type combo signal
        if assigned is None:
            key = (ft_s, op_s)
            if key in _FTYPE_OP_CATEGORY_MAP:
                cat, c = _FTYPE_OP_CATEGORY_MAP[key]
                assigned, conf, raw_sig = cat, c, "ftype_op_fallback"
 
        # v13 Tier 3b: operator-only fallback
        if assigned is None:
            if op_s == "public":
                assigned, conf, raw_sig = "government", 0.50, "operator_fallback"
            elif op_s == "private":
                assigned, conf, raw_sig = "private", 0.45, "operator_fallback"
 
        # v13 Tier 3c: hard default for any row with a facility type — never leave null
        if assigned is None and ft_s:
            assigned, conf, raw_sig = "private", 0.35, "default_fallback"
 
        result.append([assigned or "", str(conf), raw_sig or ""])
    return pd.Series(result)
 
df = df.withColumn("_org_cat",
    normalize_org_category_udf(
        F.col("organization_type"), F.col("name"),
        F.col("description"), F.col("missionstatement"), F.col("source_url"),
        F.col("facilityTypeId"), F.col("operatorTypeId"),
    ))
df = df \
    .withColumn("organization_category",
        F.when(F.col("_org_cat").getItem(0) == "", F.lit(None)).otherwise(F.col("_org_cat").getItem(0))) \
    .withColumn("org_category_confidence",
        F.col("_org_cat").getItem(1).cast(DoubleType())) \
    .withColumn("org_category_raw",
        F.when(F.col("_org_cat").getItem(2) == "", F.lit(None)).otherwise(F.col("_org_cat").getItem(2))) \
    .drop("_org_cat") \
    .withColumn("is_ngo",         F.col("organization_category") == "ngo") \
    .withColumn("is_faith_based",  F.col("organization_category") == "faith_based") \
    .withColumn("is_government",   F.col("organization_category") == "government")
 
_org_null = df.filter(F.col("organization_category").isNull()).count()
print(f"Organization category done. Null remaining: {_org_null} (target: <5%)")
df.groupBy("organization_category").count().orderBy(F.desc("count")).show()

# COMMAND ----------
# MAGIC %md ## 5.6. P.O. Box Detection + Address Typing (v12-1, v12-2)

# COMMAND ----------

_POBOX_RE = re.compile(
    r"(?i)(P\.?\s*O\.?\s*\.?\s*BOX\s+\d+|P\s*O\s+BOX\s+\d+|Private\s+Mail\s+Bag"
    r"|PMB\s+\d+|BP\s*[-–]\s*\d+|C/O\s+\w+|BPEXT\s*[-–]?\s*\d+|\bBox\s+\d{2,})"
)

_DIGITAL_ADDRESS_RE = re.compile(r"\b[A-Z]{1,3}-\d{3}-\d{4}\b")
_LANDMARK_RE = re.compile(
    r"(?i)\b(opp\.?|opposite|behind|near|adjacent|beside|next\s+to|in\s+front\s+of"
    r"|junction|roundabout|estate|market|mall|school|mosque|church|hospital|clinic"
    r"|police|stadium|station|post\s+office)\b"
)

_LOCALITY_HINTS = frozenset({
    "accra","kumasi","tamale","takoradi","sekondi","cape coast","ho","sunyani",
    "koforidua","tema","bolgatanga","wa","kasoa","tamale","techiman","goaso",
    "obuasi","madina","dansoman","adenta","ashaiman",
})

# v12-1: Greatly expanded — covers abbreviated road types, landmarks, plot numbers
_MEANINGFUL_ADDR_RE = re.compile(
    r"(?i)("
    # Street numbers + road word
    r"\d+\s+\w+\s+(?:road|street|avenue|lane|drive|close|link|crescent|circle|way|highway|bypass|rd|st|ave|ln|dr|ct|pl|blvd)"
    r"|\d+\s+(?:road|street|avenue|lane|drive|close|link|crescent|rd|st|ave|ln|dr)\b"
    # Address identifiers
    r"|(?:No\.?|#)\s*\d+"
    r"|(?:plot|lot|house|building|flat|suite|block|unit)\s+\w+"
    # Abbreviated road types ANYWHERE in string (v12-1 key fix)
    r"|\b(?:road|street|avenue|lane|drive|close|link|crescent|circle|bypass|highway)\b"
    r"|\bRd\.?\b|\bSt\.?\b|\bAve\.?\b|\bLn\.?\b|\bDr\.?\b|\bBlvd\.?\b|\bHwy\.?\b"
    # Spatial landmarks
    r"|(?:junction|roundabout|near|behind|opposite|off|before|after|beside|next\s+to)\s+\w+"
    r"|\d+\s+\w+\s+(?:Rd|St|Ave|Dr|Ln|Ct|Blvd)"
    r")"
)

_CITY_COUNTRY_ONLY_RE = re.compile(
    r"^(?:ghana|accra|kumasi|takoradi|cape\s+coast|tamale|ho|bolgatanga|wa|tema"
    r"|sunyani|koforidua|techiman|greater\s+accra|ashanti|western|eastern|central"
    r"|volta|northern|upper\s+east|upper\s+west|gh|west\s+africa|africa)\s*$",
    re.I
)

@pandas_udf(ArrayType(StringType()))
def extract_postal_address_udf(
    addr1: pd.Series, addr2: pd.Series, addr3: pd.Series, city: pd.Series
) -> pd.Series:
    """Returns [postal_address, cleaned_addr1, has_physical_address, address_type, address_confidence]."""
    result = []
    for a1, a2, a3, cty in zip(addr1, addr2, addr3, city):
        postal       = None
        physical_a1  = str(a1 or "").strip()
        has_physical = True
        has_postal   = False
        addr_conf    = 0.0

        a2_clean = str(a2 or "").strip()
        a3_clean = str(a3 or "").strip()
        city_clean = str(cty or "").strip()
        city_lower = city_clean.lower()

        # Detect P.O. Box in addr1
        if _POBOX_RE.search(physical_a1):
            postal = physical_a1
            has_postal = True
            if a2_clean and not _POBOX_RE.search(a2_clean):
                physical_a1 = a2_clean
            else:
                physical_a1 = ""
                has_physical = False

        # Detect P.O. Box in addr3
        if not postal and _POBOX_RE.search(a3_clean):
            postal = a3_clean
            has_postal = True
            
        # v13 fix: if addr1 is postal but addr2 has physical content → "mixed"
        if has_postal and not has_physical and a2_clean:
            if _MEANINGFUL_ADDR_RE.search(a2_clean) or _LANDMARK_RE.search(a2_clean):
                physical_a1 = a2_clean
                has_physical = True
                addr_conf = 0.70

        # Evaluate physical address signal
        if has_physical:
            if not physical_a1:
                has_physical = False
            else:
                s = physical_a1.strip()
                s_lower = s.lower()
                if _DIGITAL_ADDRESS_RE.search(s):
                    addr_conf = 0.90
                elif _MEANINGFUL_ADDR_RE.search(s):
                    addr_conf = 0.85
                elif _LANDMARK_RE.search(s):
                    addr_conf = 0.70
                elif _CITY_COUNTRY_ONLY_RE.match(s):
                    if s_lower in _LOCALITY_HINTS or (city_clean and s_lower == city_lower):
                        addr_conf = 0.55
                    else:
                        has_physical = False
                else:
                    # Fallback: check addr2 for physical content
                    if a2_clean and (_MEANINGFUL_ADDR_RE.search(a2_clean) or _LANDMARK_RE.search(a2_clean)):
                        physical_a1 = a2_clean
                        addr_conf = 0.70
                    else:
                        # Treat locality-only values as low-confidence physical when city is present
                        if city_clean or s_lower in _LOCALITY_HINTS:
                            addr_conf = 0.45
                        else:
                            has_physical = False

        # If city exists but physical not found and no postal, treat as low-confidence physical
        if not has_physical and not has_postal and city_clean:
            has_physical = True
            addr_conf = max(addr_conf, 0.40)

        # Compute address_type
        if has_physical and has_postal:
            address_type = "mixed"
        elif has_physical:
            address_type = "physical"
        elif has_postal:
            address_type = "postal"
        else:
            address_type = "unknown"

        result.append([postal or "", physical_a1, str(has_physical), address_type, str(addr_conf)])
    return pd.Series(result)

_addr_parts = extract_postal_address_udf(
    F.col("address_line1"), F.col("address_line2"), F.col("address_line3"), F.col("address_city")
)
df = df \
    .withColumn("_ap",               _addr_parts) \
    .withColumn("postal_address",    F.when(F.col("_ap").getItem(0) != "", F.col("_ap").getItem(0)).otherwise(F.lit(None))) \
    .withColumn("address_line1",     F.col("_ap").getItem(1)) \
    .withColumn("has_physical_address", F.col("_ap").getItem(2).cast(BooleanType())) \
    .withColumn("address_type",      F.col("_ap").getItem(3)) \
    .withColumn("address_confidence", F.col("_ap").getItem(4).cast(DoubleType())) \
    .drop("_ap")

postal_count = df.filter(F.col("postal_address").isNotNull()).count()
no_physical  = df.filter(~F.col("has_physical_address")).count()
print(f"P.O. Box detection: {postal_count} postal addresses | {no_physical} rows lack meaningful physical address")
df.groupBy("address_type").count().show()

# COMMAND ----------
# MAGIC %md ## 6. Specialty Vocabulary Normalization

# COMMAND ----------

_SPECIALTY_ALT_MAP: Dict[str, str] = {
    "internalmedicine":                             "internalMedicine",
    "familymedicine":                               "familyMedicine",
    "generalsurgery":                               "generalSurgery",
    "emergencymedicine":                            "emergencyMedicine",
    "gynecologyandobstetrics":                      "gynecologyAndObstetrics",
    "gynaecologyandobstetrics":                     "gynecologyAndObstetrics",
    "orthopedicsurgery":                            "orthopedicSurgery",
    "orthopaedicsurgery":                           "orthopedicSurgery",
    "infectiousdiseases":                           "infectiousDiseases",
    "criticalcaremedicine":                         "criticalCareMedicine",
    "cardiacsurgery":                               "cardiacSurgery",
    "plasticsurgery":                               "plasticSurgery",
    "endocrinologyanddiabetesandmetabolism":        "endocrinologyAndDiabetesAndMetabolism",
    "neonatologyperinatalmedicine":                 "neonatologyPerinatalMedicine",
    "medicaloncology":                              "medicalOncology",
    "physicalmedicineandrehabilitation":            "physicalMedicineAndRehabilitation",
    "geriatricsinternal":                           "geriatricsInternalMedicine",
    "publichealthandglobal":                        "publicHealth",
    "globalhealthandinternationalhealth":           "globalHealthAndInternationalHealth",
    "clinicalpathology":                            "clinicalPathology",
    "obstetricsandmaternitycareservice":            "obstetricsAndMaternityCare",
    "reproductiveendocrinologyandfertility":        "reproductiveEndocrinologyAndInfertility",
    "maternalfetalmedicineorperinatology":          "maternalFetalMedicineOrPerinatology",
    "socialandbehavioralsciences":                  "socialAndBehavioralSciences",
    "familyplanningandcomplexcontraception":        "familyPlanningAndComplexContraception",
    "sportsmedicine":                               "sportsMedicine",
    "craniofacialandcleftoralmaxillofacialsurgery": "craniofacialSurgery",
    "oralmaxillofacialsurgery":                     "oralAndMaxillofacialSurgery",
}

def _normalize_specialty(s: str) -> Optional[str]:
    if not s: return None
    s = str(s).strip()
    if s in FDR_SPECIALTY_VOCAB: return s
    key = re.sub(r"[\s_\-]", "", s.lower())
    if key in _SPECIALTY_ALT_MAP: return _SPECIALTY_ALT_MAP[key]
    for canonical in FDR_SPECIALTY_VOCAB:
        if canonical.lower() == key: return canonical
    return None

@pandas_udf(ArrayType(StringType()))
def normalize_specialties_udf(specs: pd.Series) -> pd.Series:
    result = []
    for items in specs:
        lst = list(items) if items is not None else []
        normalized, seen = [], set()
        for item in lst:
            canon = _normalize_specialty(str(item))
            if canon and canon not in seen:
                normalized.append(canon); seen.add(canon)
        result.append(normalized)
    return pd.Series(result)

df = df.withColumn("specialties_parsed", normalize_specialties_udf(F.col("specialties_parsed")))
print("Specialty normalization done.")


# COMMAND ----------
# MAGIC %md ## 7. Countries ISO alpha-2 Validation
 
# COMMAND ----------
 
_COUNTRY_TO_ISO2: Dict[str, str] = {
    "ghana":"GH","nigeria":"NG","ivory coast":"CI","cote d'ivoire":"CI",
    "senegal":"SN","mali":"ML","burkina faso":"BF","togo":"TG","benin":"BJ",
    "niger":"NE","guinea":"GN","liberia":"LR","sierra leone":"SL","gambia":"GM",
    "guinea-bissau":"GW","cameroon":"CM","united kingdom":"GB","uk":"GB",
    "united states":"US","usa":"US","germany":"DE","france":"FR",
}
 
def _validate_countries(items, address_country):
    valid, seen = [], set()
    for item in (items or []):
        s = str(item).strip().upper()
        if s in _VALID_ISO2 and s not in seen:
            valid.append(s); seen.add(s); continue
        iso = _COUNTRY_TO_ISO2.get(str(item).strip().lower())
        if iso and iso not in seen:
            valid.append(iso); seen.add(iso)
    if address_country and str(address_country).strip().lower() in ("ghana","gh"):
        if "GH" not in seen: valid.insert(0, "GH")
    return valid
 
@pandas_udf(ArrayType(StringType()))
def validate_countries_udf(countries: pd.Series, addr_country: pd.Series) -> pd.Series:
    return pd.Series([_validate_countries(list(i) if i is not None else [], ac) for i, ac in zip(countries, addr_country)])
 
df = df.withColumn("countries_parsed", validate_countries_udf(F.col("countries_parsed"), F.col("address_country")))

# COMMAND ----------
# MAGIC %md ## 8. Organization Description Neutrality

_SUBJECTIVE_RE = re.compile(
    r"(?i)\b(blessed|glory|god(?:'s)?|lord|jesus|christ|holy|divine|sacred|prayer"
    r"|mission\s+of\s+god|love\s+of\s+(?:god|jesus)|grace\s+of|faith-based|spirit(?:ual)?"
    r"|dedicated\s+to\s+serving\s+(?:god|the\s+lord)"
    r"|amazing|outstanding|excellent|world-class|best\s+in|unmatched|premier|leading"
    r"|passionate\s+about|committed\s+to\s+(?:excellence|quality)"
    r"|we\s+are\s+proud|proud\s+to\s+serve)\b"
)
_SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")
 
def _neutralize(text):
    if not text or str(text).strip() in ("","null","None","nan"): return None
    text = str(text).strip()
    clean = [s.strip() for s in _SENT_SPLIT_RE.split(text) if len(s.strip()) >= 10 and not _SUBJECTIVE_RE.search(s)]
    result = " ".join(clean).strip()
    return result if len(result) > 15 else None
 
@pandas_udf(StringType())
def neutralize_org_description_udf(desc: pd.Series) -> pd.Series:
    return desc.apply(_neutralize)
 
df = df.withColumn("organizationdescription", neutralize_org_description_udf(F.col("organizationdescription")))

# COMMAND ----------
# MAGIC %md ## 9. Junk Filter

# COMMAND ----------

_HALLUCINATION_RE = re.compile(
    r"(?i)(Wait\s*[-–]\s*we\s+should\s+not|As\s+an\s+AI|I\s+am\s+an\s+AI"
    r"|I\s+cannot\s+(?:make|provide|confirm|verify)"
    r"|I\s+(?:don't|do\s+not)\s+have\s+(?:access|information)"
    r"|Note:\s+I\s+(?:cannot|should)|Let\s+me\s+(?:clarify|note)\s+that\s+I)"
)
_FACEBOOK_META_RE = re.compile(
    r"(?i)((?:is\s+an?\s+)?unofficial\s+page|official\s+page\s+name|name\s+shown\s+on\s+page"
    r"|page\s+(?:creation\s+date|created(?:\s+on)?|status|type)\s*[:\-]?"
    r"|\d+\s+people\s+(?:like|follow|check(?:ed)?[\s-]in)"
    r"|people\s+(?:checked|have\s+checked)\s+in\s+here|check(?:ed)?[\s-]ins?\s+here"
    r"|category\s*:\s*(?:hospital|medical|local\s+business)"
    r"|(?:always|currently)\s+(?:open|closed)|status\s*:\s*(?:always|currently)\s+(?:open|closed))"
)
_SYSTEM_CODES_RE = re.compile(r"(?i)(System\s+[Cc]odes?\s*:\s*.*|EntityID\s+\d+|OrgMastID\s+\w+)")
_STAFF_NAME_RE   = re.compile(
    r"(?i)((?:Medical\s+Director|Administrator|Nurse\s+Manager|Pharmacist|Accountant"
    r"|Chaplain|President|Vice\s+President|Medical\s+Superintendent|Matron"
    r"|Hospital\s+Manager|Principal|Surgeon|CEO|Director)\s+(?:is|:)\s+[A-Z][a-z]+"
    r"|[A-Z][a-z]+\s+[A-Z][a-z]+\s+is\s+the\s+(?:Medical\s+Director|Administrator"
    r"|Nurse\s+Manager|Pharmacist|Accountant|Matron|Surgeon|Director))"
)
_COLLEGE_RE = re.compile(
    r"(?i)(training\s+college|(?:principal|student)\s+of\s+the\s+college"
    r"|telephone\s+for\s+(?:the\s+)?college|accountant\s+for\s+(?:the\s+)?college"
    r"|affiliated\s+(?:training|college|school)|nursing\s+school|midwifery\s+(?:training|school))"
)
_CAPABILITY_WHITELIST_RE = re.compile(
    r"(?i)\b(ICU|NICU|HDU|CCU|intensive\s+care\s+unit|high\s+dependency\s+unit"
    r"|trauma\s+(centre|center|level|care)|24[\s\-]?hour\s+(emergency|obstetric|casualty|care|service)"
    r"|emergency\s+(department|unit|service|care|obstetric)|operating\s+(theatre|room|theater)"
    r"|inpatient\s+(ward|service|bed)|outpatient\s+(department|service|clinic)"
    r"|dialysis\s+(unit|service|centre)|blood\s+bank|mortuary|PMTCT|antiretroviral"
    r"|oxygen\s+(plant|concentrator|supply)|ambulance\s+(service|bay)"
    r"|maternity\s+(ward|unit|block)|neonatal\s+(unit|ward|care)|Has\s+\d+\s+(?:patient\s+)?beds?)\b"
)
_JUNK_TOKEN_RE = re.compile(
    r"(^located\s+(at|in)\b|^has\s+a\s+location\s+at\b|^p\.?\s*o\.?\s*box\b"
    r"|GPS\s+address|\b(?:suite|floor|building)\s+\w+\b"
    r"|telephone\s+numbers?\s+(?:are|is)|has\s+(?:a\s+)?(?:telephone|phone)\s+number"
    r"|has\s+(?:an?\s+)?email\s+address|^phone\s*(?:number|contact)?\s*:"
    r"|^contact\s+(?:number|info|us)?\s*:|^fax\s*:|^email\s*:|\btel(?:ephone)?\s*[:\-]"
    r"|\+\d{7,}|http[s]?://|www\.|facebook|instagram|twitter|whatsapp|linkedin"
    r"|official\s+website|\bis\s+(?:privately|government|publicly)\s+owned\b|\bOwnership\s*[:\-]"
    r"|\bType\s*[:\-]\s*(?:Primary|Secondary|Tertiary|District|Regional|Community|CHPS|Health)"
    r"|\bFacility\s+type\s*[:\-]|\bNHIS\s+(?:accredited|accreditation|registered)\b"
    r"|\bprovides\s+general\s+(?:medical\s+|health\s+)?services?\b|\bServices\s*[:\-]\s*General\b"
    r"|listed\s+(as|in)\s+|registered\s+with\s+Ghana|page\s+created\s+on"
    r"|\d+\s+(?:people\s+)?(?:like|follow|check.?in)|opening\s+hours?|business\s+hours?"
    r"|\bMon\s*[-–]\s*(?:Fri|Sun)\b|\b\d{1,2}:\d{2}\s*(?:am|pm|AM|PM)\b"
    r"|\blikes?\b|\bfollowers?\b|\bcheck.?ins?\b|\brating\b|\breviews?\b|\bstars?\b)",
    re.IGNORECASE,
)
_JUNK_SENTENCE_RE = re.compile(
    r"((?:the\s+facility\s+)?(?:is\s+)?(?:located|situated|found)\s+(?:at|in|on)\s+.{10,}"
    r"|(?:our\s+)?(?:address|location)\s+is\s+.{5,}"
    r"|(?:you\s+can\s+)?(?:find|reach|contact)\s+us\s+(?:at|on)\s+.{5,}"
    r"|(?:call|contact|reach)\s+us\s+(?:on|at)\s+[\d\+\s\-\(\)]{7,}"
    r"|(?:phone|tel(?:ephone)?)\s*[:\-]\s*[\+\d][\d\s\-\(\)]{6,}"
    r"|for\s+(?:more\s+)?(?:information|enquiries?|appointments?),?\s+(?:call|contact|visit)"
    r"|(?:this\s+)?(?:facility|hospital|clinic|centre)\s+(?:is\s+)?(?:owned|operated|managed|run)\s+by\s+)",
    re.IGNORECASE,
)
_CLINICAL_INDICATORS = re.compile(
    r"\b(treat(?:ment|s|ing)?|diagnos(?:e|is|tic)?|therap(?:y|ies|ist)?"
    r"|surger(?:y|ies)|surgical|clinic(?:al)?|hospital|ward|patient"
    r"|doctor|physician|nurse|specialist|consult(?:ation)?"
    r"|procedure|service|care|health|medical|medicine|prescription"
    r"|laboratory|lab|delivery|antenatal|postnatal|obstetric|maternity"
    r"|pediatric|geriatric|dental|eye|optic|optometr|vision|ophthalmolog"
    r"|x.?ray|ultrasound|scan|ecg|mri|ct\s+scan|emergency|resuscit|trauma"
    r"|first\s+aid|vaccination|immunis|pharmacy|dispensar)\b",
    re.IGNORECASE,
)

def _apply_junk_filter(items):
    if not items: return []
    seen, out = set(), []
    for item in items:
        if not item: continue
        s = str(item).strip()
        if len(s) < 8 or len(s) > 500: continue
        if _HALLUCINATION_RE.search(s): continue
        if _FACEBOOK_META_RE.search(s):  continue
        if _SYSTEM_CODES_RE.search(s):   continue
        if _CAPABILITY_WHITELIST_RE.search(s):
            key = re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", s.lower())).strip()
            if key not in seen: seen.add(key); out.append(s)
            continue
        if _JUNK_TOKEN_RE.search(s):     continue
        if re.fullmatch(r"[\d\s\.,\+\-\(\)\/]+", s): continue
        if len(s) > 50 and not _CLINICAL_INDICATORS.search(s): continue
        key = re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", s.lower())).strip()
        if key not in seen: seen.add(key); out.append(s)
    return out
 
junk_filter_udf = F.udf(_apply_junk_filter, ArrayType(StringType()))
for col in ["capability_parsed", "procedure_parsed", "equipment_parsed"]:
    df = df.withColumn(col, junk_filter_udf(F.col(col)))
print("Junk filter applied.")

# COMMAND ----------
# MAGIC %md ## 9.5. Tri-State Operational Status (v12-5: negation handling + wider positive signals)

# COMMAND ----------

_CLOSED_RE = re.compile(
    r"(?i)(is\s+currently\s+closed|currently\s+closed"
    r"|no\s+longer\s+(?:in\s+operation|operational|operating|operated|available|exists?)"
    r"|(?:has\s+been|was)\s+(?:shut\s+down|closed|decommissioned|ceased)"
    r"|status\s*[:\-]?\s*Closed\s+now|Closed\s+now|permanently\s+closed|defunct"
    r"|ceased\s+operations?|no\s+longer\s+exists?|shut\s+(?:down|permanently)"
    r"|temporarily\s+closed|under\s+renovation|not\s+(?:operational|open|functioning))"
)

# v12-5: Expanded open signals with negation guard
_OPEN_RE = re.compile(
    r"(?i)(currently\s+(?:open|operating|operational)|open\s+\d+\s+(?:hours?|days?)"
    r"|24[\s\-]?(?:hour|hr)s?\s+(?:service|emergency|care|available|access)"
    r"|24\s*/\s*7\s+(?:service|care|emergency|available)"
    r"|serving\s+(?:patients?|the\s+community|ghana)"
    r"|fully\s+(?:operational|functioning|equipped|staffed)"
    r"|accepting\s+(?:new\s+)?(?:patients?|walk.?ins?)"
    r"|we\s+(?:offer|provide|serve|treat|welcome|accept|operate)"
    r"|our\s+(?:services|team|hospital|clinic|staff|doctors|facility)"
    r"|provides?\s+(?:healthcare|medical|clinical|surgical|dental|specialist)\s+(?:services?|care)"
    r"|available\s+(?:for|to)\s+(?:appointments?|patients?|consultations?)"
    r"|book\s+(?:an\s+)?(?:appointment|consultation|online)"
    r"|walk.?in\s+(?:clinic|patients?|services?|welcome)"
    r"|emergency\s+services?\s+(?:available|offered|provided|24\s*hours?)"
    r"|operating\s+since|established\s+in|serving\s+(?:ghana|accra|kumasi|takoradi)"
    r"|open\s+(?:monday|tuesday|weekdays|daily|now|every\s+day)"
    r"|consultation\s+(?:hours?|times?|available)"
    r"|(?:inpatient|outpatient)\s+(?:services?|care|admission)"
    r"|call\s+us\s+(?:today|now|on|at)\s+\+?\d"
    r"|appointments?\s+(?:available|can\s+be\s+booked|online))"
)

# v12-5: Negation guard — if negation precedes open signal, don't mark open
_NEGATION_PRE_RE = re.compile(
    r"(?i)(no\s+longer|not\s+(?:currently|anymore?|available|open|operating)"
    r"|(?:was|used\s+to)\s+(?:provide|offer|serve|operate)"
    r"|(?:closed|shut|discontinued|ceased|ended)\s+(?:in|on|by|after))\s+.{0,60}",
)
@pandas_udf(StringType())
def detect_operational_status_tristate_udf(cap: pd.Series, desc: pd.Series) -> pd.Series:
    result = []
    for caps, desc_val in zip(cap, desc):
        cap_list = list(caps) if caps is not None else []
        desc_str = "" if pd.isna(desc_val) else str(desc_val)
        text = " ".join(str(c) for c in cap_list) + " " + desc_str
 
        if _CLOSED_RE.search(text):
            result.append("closed"); continue
 
        open_signals = [m for m in _OPEN_RE.finditer(text)]
        if open_signals:
            # v13-5: Check negation for each signal
            valid_signals = 0
            for m in open_signals:
                context = text[max(0, m.start()-120):m.start()]
                if not _NEGATION_PRE_RE.search(context):
                    valid_signals += 1
            # v13-5: require at least 1 un-negated signal (was effectively 1 before)
            # For "unknown" text only, keep stricter: still use 1 signal for direct evidence
            if valid_signals >= 1:
                result.append("open")
            else:
                result.append("unknown")
        else:
            result.append("unknown")
    return pd.Series(result)
 
df = df.withColumn(
    "operational_status",
    detect_operational_status_tristate_udf(F.col("capability_parsed"), F.col("description"))
)
print("Initial tri-state detection:")
df.groupBy("operational_status").count().show()

# COMMAND ----------
# MAGIC %md ## 10. Mine Procedures & Equipment

# COMMAND ----------

PROCEDURE_KEYWORDS = [
    "general surgery","surgical","caesarean section","c-section","caesarean",
    "laparoscopy","laparoscopic","appendectomy","hysterectomy","herniorrhaphy",
    "hernia repair","cholecystectomy","thyroidectomy","mastectomy",
    "cataract surgery","laser surgery","eye surgery","corneal transplant",
    "hip replacement","knee replacement","orthopaedic surgery","orthopedic surgery",
    "prostatectomy","nephrectomy","colostomy","bowel resection",
    "antenatal care","postnatal care","delivery","obstetrics",
    "gynecology","gynaecology","family planning","fertility treatment",
    "maternity services","labour and delivery","normal delivery",
    "laboratory","lab tests","blood test","urine test","stool test",
    "medical lab","in-house lab","laboratory services",
    "ultrasound","ultrasound scan","x-ray","x ray","x-rays","radiography",
    "digital x-ray","ct scan","mri","mammography",
    "ecg","electrocardiogram","echocardiogram",
    "endoscopy","colonoscopy","biopsy","pap smear",
    "audiometry","spirometry","pulmonary function test",
    "blood transfusion","hemodialysis","dialysis","chemotherapy",
    "radiotherapy","physiotherapy","physical therapy","occupational therapy",
    "speech therapy","wound care","wound dressing","suturing",
    "vaccination","immunisation","immunization","circumcision",
    "minor surgery","minor procedures","outpatient procedures",
    "electro convulsive therapy","ect","psychotherapy","counselling","counseling",
    "psychiatric services","mental health services",
    "dental extraction","tooth extraction","dental filling","root canal",
    "dental cleaning","scaling and polishing","orthodontic","dentures",
    "dental services","in-house dental lab",
    "eye examination","vision testing","refraction","glaucoma treatment",
    "ophthalmology services","optometry services","eye care","eye care services",
    "cataract screening","low vision",
    "emergency care","resuscitation","trauma care","first aid","emergency services",
    "dispensary","pharmacy services","pharmaceutical services",
    "paediatric care","pediatric care","neonatal care","newborn care",
    "child health","child welfare",
    "dermatology","neurology","cardiology","endocrinology","urology",
    "gastroenterology","haematology","hematology","oncology","geriatrics",
    "general medical consultation","inpatient admission","outpatient consultation",
    # v12-6: Additional procedures
    "blood pressure monitoring","diabetes management","hiv testing","hiv treatment",
    "tuberculosis treatment","malaria treatment","typhoid treatment",
    "cervical cancer screening","breast examination","prostate screening",
    "skin care","wound management","fracture management","plaster of paris",
    "antenatal booking","delivery room","postnatal follow-up",
    "neonatal resuscitation","kangaroo mother care","growth monitoring",
    "nutrition counseling","family planning counseling","contraception",
    "voluntary counseling and testing","vct","pmtct","art","antiretroviral therapy",
    "triage","vital signs monitoring","echocardiography","stress test",
]

EQUIPMENT_KEYWORDS = [
    "x-ray machine","x ray machine","digital x-ray","portable x-ray",
    "ultrasound machine","ultrasound scanner","ct scanner","ct scan",
    "mri machine","mri scanner","mammography machine","fluoroscopy",
    "doppler","doppler ultrasound","echocardiograph","echocardiography",
    "haematology analyser","hematology analyzer","biochemistry analyser",
    "blood gas analyser","pcr machine","centrifuge","microscope",
    "cd4 machine","viral load machine",
    "medical laboratory","medical lab","in-house laboratory","in-house lab",
    "laboratory equipment","operating theatre","operating room","operating theater",
    "anaesthesia machine","anesthesia machine","ventilator","mechanical ventilation",
    "defibrillator","laparoscope","endoscope","operating table",
    "ecg machine","ecg","electrocardiogram machine",
    "pulse oximeter","vital signs monitor","cardiac monitor",
    "blood pressure monitor","glucometer","infusion pump","syringe pump",
    "icu bed","icu unit","intensive care unit","high dependency unit",
    "oxygen plant","oxygen concentrator","oxygen cylinder",
    "ambulance","blood bank","mortuary",
    "dialysis machine","haemodialysis","hemodialysis machine",
    "dental chair","dental unit","dental x-ray","in-house dental lab",
    "slit lamp","tonometer","phoropter","fundus camera",
    "automated dispensing","cold chain","inpatient beds","basic medical equipment",
    # v12-6: Additional equipment
    "patient monitor","multiparameter monitor","fetal monitor","ctg machine",
    "incubator","phototherapy unit","suction machine","nebuliser","nebulizer",
    "autoclave","steriliser","sterilizer","hospital bed","adjustable bed",
    "wheelchair","stretcher","resuscitation trolley","crash cart",
    "blood glucose meter","haemoglobin machine","urinalysis machine",
    "biopsy needle","surgical instruments","laparotomy set",
    "delivery bed","gynaecology chair","neonatal warmer","radiant warmer",
    "refrigerator for vaccines","cold chain equipment","vaccine storage",
    "portable ultrasound","bedside ultrasound","point-of-care testing",
]

EQUIPMENT_CONTEXT_RULES: List[Tuple[re.Pattern, List[str]]] = [
    (re.compile(r"\b(laboratory\s+services?|medical\s+lab|in-house\s+lab|lab\s+tests?)\b", re.I), ["Laboratory Equipment","Microscope","Centrifuge"]),
    (re.compile(r"\b(imaging|radiology\s+services?|scan\s+cent)\b", re.I), ["X-Ray Machine","Ultrasound Machine"]),
    (re.compile(r"\b(icu|intensive\s+care\s+unit)\b", re.I), ["Ventilator","Cardiac Monitor","Pulse Oximeter","Infusion Pump"]),
    (re.compile(r"\b(emergency\s+(department|unit|services?)|casualty\s+unit)\b", re.I), ["Defibrillator","Vital Signs Monitor","Oxygen Cylinder","Resuscitation Trolley"]),
    (re.compile(r"\b(maternity\s+(ward|unit)|obstetric\s+care|labour\s+ward)\b", re.I), ["Fetal Monitor","Delivery Bed","Oxygen Concentrator","Neonatal Warmer"]),
    (re.compile(r"\b(eye\s+clinic|ophthalmolog|optometr)\b", re.I), ["Slit Lamp","Tonometer","Fundus Camera"]),
    (re.compile(r"\b(dialysis|hemodialysis|haemodialysis)\b", re.I), ["Hemodialysis Machine"]),
    (re.compile(r"\b(operating\s+(theatre|room)|surgical\s+(suite|unit))\b", re.I), ["Operating Theatre","Anaesthesia Machine","Operating Table"]),
    (re.compile(r"\b(blood\s+bank|transfusion\s+services?)\b", re.I), ["Blood Bank","Centrifuge"]),
    (re.compile(r"\b(oxygen\s+(plant|concentrator|supply|therapy))\b", re.I), ["Oxygen Plant"]),
    (re.compile(r"\b(neonatal|nicu|newborn\s+care)\b", re.I), ["Incubator","Phototherapy Unit","Neonatal Warmer"]),
    (re.compile(r"\b(vaccination|immunisation|immunization|cold\s+chain)\b", re.I), ["Cold Chain Equipment","Vaccine Storage"]),
]

def _mine_from_description(desc, vocab):
    if not desc or str(desc).strip() in ("","null","None","nan"): return []
    text = str(desc).lower()
    found, seen = [], set()
    for term in vocab:
        t = term.lower()
        if re.search(r'\b' + re.escape(t) + r'\b', text) and t not in seen:
            seen.add(t); found.append(term.title())
    return found

@pandas_udf(ArrayType(StringType()))
def mine_procedures_udf(proc_arr: pd.Series, desc_col: pd.Series, cap_arr: pd.Series, org_desc_col: pd.Series) -> pd.Series:
    result = []
    for procs, desc, caps, org_desc in zip(proc_arr, desc_col, cap_arr, org_desc_col):
        combined = list(procs) if procs is not None else []
        seen_keys = {re.sub(r"[^\w\s]","",p.lower()).strip() for p in combined}
        # v12-6: Mine from description AND organizationdescription
        for source_text in [desc, org_desc]:
            for m in _mine_from_description(source_text, PROCEDURE_KEYWORDS):
                k = re.sub(r"[^\w\s]","",m.lower()).strip()
                if k not in seen_keys: combined.append(m); seen_keys.add(k)
        for cap_item in (list(caps) if isinstance(caps,(list,tuple)) else []):
            if not cap_item or len(cap_item) < 10: continue
            for kw in PROCEDURE_KEYWORDS:
                if re.search(r'\b' + re.escape(kw.lower()) + r'\b', str(cap_item).lower()):
                    k = kw.lower()
                    if k not in seen_keys: combined.append(kw.title()); seen_keys.add(k)
        result.append(combined)
    return pd.Series(result)

@pandas_udf(ArrayType(StringType()))
def mine_equipment_udf(equip_arr: pd.Series, desc_col: pd.Series, cap_arr: pd.Series, org_desc_col: pd.Series) -> pd.Series:
    result = []
    for equips, desc, caps, org_desc in zip(equip_arr, desc_col, cap_arr, org_desc_col):
        combined = list(equips) if equips is not None else []
        seen_keys = {re.sub(r"[^\w\s]","",e.lower()).strip() for e in combined}
        # v12-6: Mine from description AND organizationdescription
        for source_text in [desc, org_desc]:
            for m in _mine_from_description(source_text, EQUIPMENT_KEYWORDS):
                k = re.sub(r"[^\w\s]","",m.lower()).strip()
                if k not in seen_keys: combined.append(m); seen_keys.add(k)
        caps_iter = list(caps) if isinstance(caps,(list,tuple)) else []
        cap_text_full = " ".join(str(c) for c in caps_iter)
        for cap_item in caps_iter:
            if not cap_item or len(cap_item) < 10: continue
            for kw in EQUIPMENT_KEYWORDS:
                if re.search(r'\b' + re.escape(kw.lower()) + r'\b', str(cap_item).lower()):
                    k = kw.lower()
                    if k not in seen_keys: combined.append(kw.title()); seen_keys.add(k)
        combined_text = f"{str(desc or '')} {str(org_desc or '')} {cap_text_full}"
        for pattern, items_to_add in EQUIPMENT_CONTEXT_RULES:
            if pattern.search(combined_text):
                for item in items_to_add:
                    k = re.sub(r"[^\w\s]","",item.lower()).strip()
                    if k not in seen_keys: combined.append(item); seen_keys.add(k)
        result.append(combined)
    return pd.Series(result)

df = df \
    .withColumn("procedure_parsed", mine_procedures_udf(
        F.col("procedure_parsed"), F.col("description"),
        F.col("capability_parsed"), F.col("organizationdescription"))) \
    .withColumn("equipment_parsed", mine_equipment_udf(
        F.col("equipment_parsed"), F.col("description"),
        F.col("capability_parsed"), F.col("organizationdescription")))

def _dedupe_case_insensitive(items: List[str]) -> List[str]:
    seen, out = set(), []
    for item in items:
        if not item: continue
        s = str(item).strip()
        key = re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", s.lower())).strip()
        if not key or key in seen: continue
        seen.add(key); out.append(s)
    return out

@pandas_udf(ArrayType(StringType()))
def dedupe_list_udf(items: pd.Series) -> pd.Series:
    return pd.Series([_dedupe_case_insensitive(list(x) if x is not None else []) for x in items])

_DERIVED_CAPABILITY_RULES: List[Tuple[re.Pattern, str]] = [
    (re.compile(r"\b(surgery|operating\s+theatre|operating\s+room|theatre)\b", re.I), "Surgical Services"),
    (re.compile(r"\b(emergency|trauma|resuscitation|casualty)\b", re.I), "Emergency Care"),
    (re.compile(r"\b(icu|intensive\s+care|nicu|hdu)\b", re.I), "Critical Care"),
    (re.compile(r"\b(radiology|x-ray|ultrasound|ct\s+scan|mri)\b", re.I), "Radiology Services"),
    (re.compile(r"\b(obstetric|maternity|labour|delivery)\b", re.I), "Obstetric Care"),
    (re.compile(r"\b(dental|dentist)\b", re.I), "Dental Care"),
    (re.compile(r"\b(eye\s+care|ophthalmolog|optometr|optical)\b", re.I), "Eye Care"),
    (re.compile(r"\b(laboratory|lab\s+tests?|pathology)\b", re.I), "Laboratory Services"),
    (re.compile(r"\b(dialysis|hemodialysis|haemodialysis)\b", re.I), "Dialysis Services"),
]

@pandas_udf(ArrayType(StringType()))
def derive_capabilities_udf(cap_arr: pd.Series, proc_arr: pd.Series, equip_arr: pd.Series, desc_col: pd.Series) -> pd.Series:
    result = []
    for caps, procs, equips, desc in zip(cap_arr, proc_arr, equip_arr, desc_col):
        combined = list(caps) if caps is not None else []
        text_blob = " ".join([str(x) for x in (list(procs) if procs is not None else [])])
        text_blob += " " + " ".join([str(x) for x in (list(equips) if equips is not None else [])])
        text_blob += " " + str(desc or "")
        for pattern, label in _DERIVED_CAPABILITY_RULES:
            if pattern.search(text_blob):
                combined.append(label)
        result.append(_dedupe_case_insensitive(combined))
    return pd.Series(result)

df = df \
    .withColumn("procedure_parsed", dedupe_list_udf(F.col("procedure_parsed"))) \
    .withColumn("equipment_parsed", dedupe_list_udf(F.col("equipment_parsed"))) \
    .withColumn("capability_parsed", derive_capabilities_udf(F.col("capability_parsed"), F.col("procedure_parsed"), F.col("equipment_parsed"), F.col("description")))
print("Procedure/equipment mining done.")

# COMMAND ----------
# MAGIC %md ## 11. Numeric Field Extraction (v12-6: organizationdescription as explicit source)

# COMMAND ----------

_TIME_SOCIAL_GUARD = re.compile(
    r"(?i)(24\s*/\s*7|24\s*hours?|\bhours?\b|\bdays?\s+a\s+week\b"
    r"|\bopen\s+\d|\blikes?\b|\bfollowers?\b|\bcheck.?ins?\b"
    r"|\bsince\s+\d{4}\b|\bin\s+\d{4}\b|\bestablished\s+\d{4}\b"
    r"|\bfounded\s+\d{4}\b|\bopened\s+\d{4}\b|\bstarted\s+\d{4}\b)"
)

_DOC_PATTERNS = [
    re.compile(r"medical\s+team\s+(?:consists?\s+of|of)\s+(\d+)", re.I),
    re.compile(r"team\s+of\s+(\d+)\s+(?:medical|health(?:care)?|clinical)\s+(?:professionals?|staff|workers?)", re.I),
    re.compile(r"team\s+of\s+(\d+)\s+(?:doctors?|physicians?|specialists?|surgeons?)", re.I),
    re.compile(r"staff\s+of\s+(\d+)\s+(?:doctors?|physicians?|clinicians?|medical\s+professionals?)", re.I),
    re.compile(r"(?:employs?|has)\s+(\d+)\s+(?:medical\s+)?(?:doctors?|physicians?|specialists?)", re.I),
    re.compile(r"staffed\s+(?:by|with)\s+(\d+)\s+(?:medical\s+)?(?:professionals?|doctors?|physicians?)", re.I),
    re.compile(r"(\d+)\s+(?:permanent|full.time|part.time|resident)\s+(?:medical\s+)?(?:doctors?|physicians?|staff)", re.I),
    re.compile(r"(\d+)\s+(?:medical\s+)?(?:doctors?|physicians?|specialists?|consultants?|surgeons?)\s+(?:are|on\s+call|available|on\s+duty|serving)", re.I),
    re.compile(r"\b(\d{1,3})\s+(?:medical\s+)?(?:doctors?|physicians?|specialists?|consultants?|surgeons?|general\s+practitioners?)\b", re.I),
    re.compile(r"\b(\d{1,3})\s+(?:staff|health\s+workers|medical\s+staff)\b", re.I),
    re.compile(r"\bteam\s+of\s+(\d{1,3})\b", re.I),
    re.compile(r"\bstaff\s+strength\s+(?:of\s+)?(\d{1,3})\b", re.I),
    re.compile(r"\btotal\s+staff\s+(?:of\s+)?(\d{1,3})\b", re.I),
    re.compile(r"\b(\d{1,3})\s+doctors?\s+(?:and|&)\s+\d+\s+(?:nurses?|staff)\b", re.I),
    re.compile(r"\b(\d{1,3})\s+medical\s+officers?\b", re.I),
    re.compile(r"\b(\d{1,3})\s+(?:qualified|trained|licensed)\s+(?:doctors?|physicians?)\b", re.I),
    re.compile(r"\b(\d{1,3})\s+clinical\s+(?:staff|officers?)\b", re.I),
    re.compile(r"number\s+of\s+(?:doctors?|physicians?)\s*[:\-]?\s*(\d{1,3})\b", re.I),
    re.compile(r"(?:doctors?|physicians?)\s*[:\-]\s*(\d{1,3})\b", re.I),
    re.compile(r"\b(\d{1,3})\s+specialists?\b", re.I),
    re.compile(r"\b(\d{1,3})\s+consultants?\b", re.I),
    re.compile(r"\b(\d{1,3})\s+surgeons?\b", re.I),
]

_MONTHS_PAT = r"(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)"

_BED_PATTERNS = [
    re.compile(r"\bHas\s+(\d{1,4})\s+(?:patient\s+)?beds?\b", re.I),
    re.compile(r"\b(\d{1,4})[+]?\s*[-]?\s*bed\s+(?:hospital|facility|centre|center|capacity)\b", re.I),
    re.compile(r"\b(\d{1,4})-bed\b", re.I),
    re.compile(r"\b(\d{2,4})\s+bed\s+capacity\b", re.I),
    re.compile(r"bed\s+capacity\s*(?:of|:)?\s*(\d{2,4})", re.I),
    re.compile(r"bedspace\s+(?:of|:)?\s*(\d{2,4})", re.I),
    re.compile(r"\b(\d{2,4})\s+bedspaces?\b", re.I),
    re.compile(r"capacity\s+(?:of|to\s+accommodate)\s+(\d{2,4})\s*(?:patients?|beds?)?", re.I),
    re.compile(r"accommodate\s+(\d{2,4})\s+(?:beds?|patients?|inpatients?)", re.I),
    re.compile(r"(?:currently\s+)?running\s+(\d{2,4})\s*beds?", re.I),
    re.compile(r"operating\s+(?:a\s+)?(\d{2,4})\s*[-]?\s*bed", re.I),
    re.compile(r"(?:is\s+a|a|an)\s+(\d{2,4})\s*[-]?\s*bed\b", re.I),
    re.compile(r"\b(\d{2,4})\s+(?:beds?|inpatient|ward\s+beds?)\b", re.I),
    re.compile(r"\binpatient\s+capacity\s*(?:of|:)?\s*(\d{2,4})\b", re.I),
    re.compile(r"\b(\d{2,4})\s+inpatient\s+beds?\b", re.I),
    re.compile(r"\b(\d{2,4})\s+ward\s+beds?\b", re.I),
    re.compile(r"\bhas\s+(\d{2,4})\s+beds?\b", re.I),
    re.compile(r"\bprovides\s+(\d{2,4})\s+beds?\b", re.I),
    re.compile(r"\boffers\s+(\d{2,4})\s+beds?\b", re.I),
    re.compile(r"\b(\d{2,4})\s*bed\s+(?:district|regional|teaching)\s+hospital\b", re.I),
    re.compile(r"\b(\d{1,3})\s+icu\s+beds?\b", re.I),
    re.compile(r"\bnicu\s+(?:with\s+)?(\d{1,3})\s+beds?\b", re.I),
    re.compile(r"\b(\d{1,3})\s+intensive\s+care\s+beds?\b", re.I),
    re.compile(r"\bcan\s+accommodate\s+up\s+to\s+(\d{2,4})\s+(?:patients?|beds?)\b", re.I),
    re.compile(r"total\s+(?:bed\s+)?capacity\s*[:\-]?\s*(\d{2,4})\b", re.I),
    re.compile(r"\b(\d{2,4})\s+(?:medical|surgical|paediatric|maternity)\s+beds?\b", re.I),
    re.compile(r"admitted\s+up\s+to\s+(\d{2,4})\s+patients?\b", re.I),
]

_YEAR_PATTERNS = [
    re.compile(rf"(?:founded|established|started|commissioned|opened)\s+(?:in\s+|on\s+)?(?:{_MONTHS_PAT}\s+)?(\d{{4}})\b", re.I),
    re.compile(r"since\s+(\d{4})\b", re.I),
    re.compile(r"serving\s+ghana\s+since\s+(\d{4})\b", re.I),
    re.compile(r"in\s+(\d{4})\s+(?:by|as\s+a)", re.I),
    re.compile(r"(?:maternity\s+home\s+in|health\s+centre\s+in|hospital\s+in)\s+(\d{4})\b", re.I),
    re.compile(r"\b(est\.?|established)\s*[:\-]?\s*(\d{4})\b", re.I),
    re.compile(r"\bfounded\s*[:\-]?\s*(\d{4})\b", re.I),
    re.compile(r"\bEstablished\s+in\s+(\d{4})\b", re.I),
    re.compile(r"\bEstablished\s+(\d{4})\b", re.I),
    re.compile(rf"\bestablished\s+in\s+(?:{_MONTHS_PAT})\s+(\d{{4}})\b", re.I),
    re.compile(r"year\s+of\s+(?:establishment|founding|incorporation)\s*[:\-]?\s*(\d{4})\b", re.I),
    re.compile(r"incorporated\s+in\s+(\d{4})\b", re.I),
    re.compile(r"registered\s+in\s+(\d{4})\b", re.I),
    # v12-6: Additional year patterns
    re.compile(r"commenced\s+(?:operations?|services?)\s+(?:in\s+)?(\d{4})\b", re.I),
    re.compile(r"set\s+up\s+in\s+(\d{4})\b", re.I),
    re.compile(r"built\s+in\s+(\d{4})\b", re.I),
    re.compile(r"operational\s+(?:since|from)\s+(\d{4})\b", re.I),
    re.compile(r"(\d{4})\s+(?:to\s+(?:present|date)|and\s+still\s+serving)", re.I),
]

_AREA_PATTERNS = [
    re.compile(r"(\d[\d,]+)\s*(?:sq(?:uare)?\s*(?:met(?:ers?|res?)|m[²2]))", re.I),
    re.compile(r"floor\s+(?:area|space)\s*(?:of|:)?\s*(\d[\d,]+)", re.I),
    re.compile(r"(\d[\d,]+)\s*m²", re.I),
]

FTYPE_BED_RANGE: Dict[str, Tuple[int, int]] = {
    "hospital":(50,200),"clinic":(5,30),"maternity_home":(10,40),"chps":(2,10),
    "eye_clinic":(0,5),"diagnostic_center":(0,0),"pharmacy":(0,0),"dentist":(0,5),"doctor":(0,5),
}

def _plausibility_check(val, field):
    if field == "doctors": return 1 <= val <= cfg.MAX_DOCTORS
    if field == "beds":    return 5 <= val <= cfg.MAX_BEDS
    if field == "year":    return cfg.MIN_YEAR <= val <= cfg.MAX_YEAR
    if field == "area":    return 10 <= val <= cfg.MAX_AREA
    return True

def _extract_int_from_text(text, patterns, cap=9999, guard=None, min_val=1, field=""):
    if not text: return None
    for pat in patterns:
        for m in pat.finditer(text):
            if guard:
                ctx = text[max(0,m.start()-60):m.end()+60]
                if guard.search(ctx): continue
            try:
                # Handle patterns with two groups (e.g. "est. 1990" → group 2)
                try:   raw = str(m.group(2)).replace(",","")
                except IndexError: raw = str(m.group(1)).replace(",","")
                val = int(raw)
                if min_val <= val <= cap and (not field or _plausibility_check(val, field)):
                    return val
            except (IndexError, ValueError): pass
    return None

def _extract_source_int(source_val, cap=9999, min_val=1):
    if source_val is None: return None
    try:
        v = float(str(source_val).strip())
        return int(v) if min_val <= v <= cap else None
    except (ValueError, TypeError): return None

@pandas_udf(ArrayType(StringType()))
def extract_number_doctors_udf(src, desc, org_desc, mission, cap):
    result = []
    for src_val, desc_val, org_desc_val, mission_val, caps in zip(src, desc, org_desc, mission, cap):
        val = _extract_source_int(src_val, cap=cfg.MAX_DOCTORS)
        conf = "high" if val is not None else None
        if val is None:
            # v12-6: Include organizationdescription explicitly
            combined = " ".join([str(x or "") for x in [desc_val, org_desc_val, mission_val]])
            val = _extract_int_from_text(combined, _DOC_PATTERNS, cap=cfg.MAX_DOCTORS, guard=_TIME_SOCIAL_GUARD, field="doctors")
            if val is not None: conf = "medium"
        if val is None and caps is not None:
            for cap_item in (list(caps) if isinstance(caps,(list,tuple)) else []):
                if cap_item and len(str(cap_item)) > 10:
                    v2 = _extract_int_from_text(str(cap_item), _DOC_PATTERNS, cap=cfg.MAX_DOCTORS, field="doctors")
                    if v2: val = v2; conf = "low"; break
        result.append([str(val) if val is not None else "", conf or "not_extracted"])
    return pd.Series(result)

@pandas_udf(ArrayType(StringType()))
def extract_capacity_int_udf(src, desc, org_desc, mission, cap, ftype):
    result = []
    for src_val, desc_val, org_desc_val, mission_val, caps, ft in zip(src, desc, org_desc, mission, cap, ftype):
        val = _extract_source_int(src_val, cap=cfg.MAX_BEDS, min_val=1)
        conf = "high" if val is not None else None
        if val is None:
            combined = " ".join([str(x or "") for x in [desc_val, org_desc_val, mission_val]])
            val = _extract_int_from_text(combined, _BED_PATTERNS, cap=cfg.MAX_BEDS, min_val=5, field="beds")
            if val is not None: conf = "medium"
        if val is None and caps is not None:
            for cap_item in (list(caps) if isinstance(caps,(list,tuple)) else []):
                if cap_item and len(str(cap_item)) > 5:
                    v2 = _extract_int_from_text(str(cap_item), _BED_PATTERNS, cap=cfg.MAX_BEDS, min_val=5, field="beds")
                    if v2: val = v2; conf = "extracted_from_capability"; break
        if val is None and ft:
            ft_clean = str(ft).strip().lower()
            caps_list = list(caps) if isinstance(caps,(list,tuple)) else []
            text_blob = f"{str(desc_val or '')} {' '.join(str(c) for c in caps_list)}"
            if re.search(r"\b(inpatient|ward|admission|bed|hospitalization|icu|nicu|hdu)\b", text_blob, re.I) and ft_clean in FTYPE_BED_RANGE:
                lo, hi = FTYPE_BED_RANGE[ft_clean]
                if hi > 0: val = int((lo+hi)/2); conf = "inferred"
        result.append([str(val) if val is not None else "", conf or "not_extracted"])
    return pd.Series(result)

@pandas_udf(ArrayType(StringType()))
def extract_year_established_udf(src, desc, org_desc, mission, cap):
    result = []
    for src_val, desc_val, org_desc_val, mission_val, caps in zip(src, desc, org_desc, mission, cap):
        val = _extract_source_int(src_val, cap=cfg.MAX_YEAR, min_val=cfg.MIN_YEAR)
        conf = "high" if val is not None else None
        if val is None:
            # v12-6: All text fields including organizationdescription
            combined = " ".join([str(x or "") for x in [desc_val, org_desc_val, mission_val]])
            val = _extract_int_from_text(combined, _YEAR_PATTERNS, cap=cfg.MAX_YEAR, min_val=cfg.MIN_YEAR, field="year")
            if val is not None: conf = "medium"
        if val is None and caps is not None:
            for cap_item in (list(caps) if isinstance(caps,(list,tuple)) else []):
                if cap_item and len(str(cap_item)) > 5:
                    v2 = _extract_int_from_text(str(cap_item), _YEAR_PATTERNS, cap=cfg.MAX_YEAR, min_val=cfg.MIN_YEAR, field="year")
                    if v2: val = v2; conf = "extracted_from_capability"; break
        result.append([str(val) if val is not None else "", conf or "not_extracted"])
    return pd.Series(result)

@pandas_udf(IntegerType())
def extract_area_int_udf(src: pd.Series) -> pd.Series:
    return pd.Series([_extract_source_int(v, cap=cfg.MAX_AREA, min_val=1) for v in src], dtype="Int64")

@pandas_udf(IntegerType())
def extract_pk_unique_id_int_udf(src: pd.Series) -> pd.Series:
    result = []
    for v in src:
        try: result.append(int(float(str(v).strip())))
        except: result.append(None)
    return pd.Series(result, dtype="Int64")

@pandas_udf(BooleanType())
def parse_accepts_volunteers_udf(src: pd.Series) -> pd.Series:
    def _parse(v):
        if v is None: return None
        s = str(v).strip().lower()
        if s in ("true","1","yes"): return True
        if s in ("false","0","no"): return False
        return None
    return src.apply(_parse)

_doctors_arr  = extract_number_doctors_udf(F.col("numberdoctors"), F.col("description"), F.col("organizationdescription"), F.col("missionstatement"), F.col("capability_parsed"))
_capacity_arr = extract_capacity_int_udf(F.col("capacity"), F.col("description"), F.col("organizationdescription"), F.col("missionstatement"), F.col("capability_parsed"), F.col("facilityTypeId"))
_year_arr     = extract_year_established_udf(F.col("yearestablished"), F.col("description"), F.col("organizationdescription"), F.col("missionstatement"), F.col("capability_parsed"))

df = df \
    .withColumn("_da", _doctors_arr) \
    .withColumn("_ca", _capacity_arr) \
    .withColumn("_ya", _year_arr) \
    .withColumn("number_doctors_int",   F.when(F.col("_da").getItem(0) != "", F.col("_da").getItem(0).cast(IntegerType())).otherwise(F.lit(None))) \
    .withColumn("doctors_confidence",   F.col("_da").getItem(1)) \
    .withColumn("capacity_int",         F.when(F.col("_ca").getItem(0) != "", F.col("_ca").getItem(0).cast(IntegerType())).otherwise(F.lit(None))) \
    .withColumn("capacity_confidence",  F.col("_ca").getItem(1)) \
    .withColumn("year_established_int", F.when(F.col("_ya").getItem(0) != "", F.col("_ya").getItem(0).cast(IntegerType())).otherwise(F.lit(None))) \
    .withColumn("year_confidence",      F.col("_ya").getItem(1)) \
    .withColumn("area_int",                extract_area_int_udf(F.col("area"))) \
    .withColumn("accepts_volunteers_bool", parse_accepts_volunteers_udf(F.col("acceptsvolunteers"))) \
    .withColumn("pk_unique_id_int",        extract_pk_unique_id_int_udf(F.col("pk_unique_id"))) \
    .withColumn("numberDoctors",
                F.when(F.col("number_doctors_int").isNotNull(), F.col("number_doctors_int").cast(IntegerType())).otherwise(F.lit(None).cast(IntegerType()))) \
    .drop("_da","_ca","_ya")

@pandas_udf(BooleanType())
def infer_accepts_volunteers_udf(existing: pd.Series, org_type: pd.Series, mission: pd.Series) -> pd.Series:
    _VOL_RE = re.compile(r"(?i)\bvolunteer\b")
    result = []
    for ex, ot, ms in zip(existing, org_type, mission):
        if ex is not None: result.append(ex); continue
        result.append(True if str(ot or "").lower() == "ngo" and _VOL_RE.search(str(ms or "")) else None)
    return pd.Series(result)

df = df.withColumn("accepts_volunteers_bool", infer_accepts_volunteers_udf(F.col("accepts_volunteers_bool"), F.col("organization_type"), F.col("missionstatement")))

df = df \
    .withColumn("number_doctors_known", F.col("number_doctors_int").isNotNull()) \
    .withColumn("capacity_known", F.col("capacity_int").isNotNull()) \
    .withColumn("year_established_known", F.col("year_established_int").isNotNull()) \
    .withColumn("area_known", F.col("area_int").isNotNull())

print("Numeric extraction done.")
for c in ["number_doctors_int","capacity_int","year_established_int","area_int"]:
    cnt = df.filter(F.col(c).isNotNull()).count()
    print(f"  {c:<28} non-null: {cnt}  (text extraction only — source fields NULL in v0.3)")

# COMMAND ----------
# MAGIC %md ## 12. Region Normalisation

# COMMAND ----------

VALID_REGIONS = frozenset({
    "Greater Accra","Ashanti","Western","Eastern","Central",
    "Volta","Northern","Upper East","Upper West",
    "Oti","Bono East","Ahafo","Savannah","North East","Western North","Brong-Ahafo","Bono",
})
 
REGION_NORM_MAP = {
    "Greater Accra Region":"Greater Accra","Greater Accra":"Greater Accra",
    "Accra":"Greater Accra","Accra Region":"Greater Accra","Accra North":"Greater Accra","Accra East":"Greater Accra",
    "Ashanti Region":"Ashanti","Ashanti":"Ashanti","ASHANTI":"Ashanti","Asante Region":"Ashanti","Asante":"Ashanti",
    "Kumasi Metropolitan":"Ashanti","Ejisu-Juaben Municipal":"Ashanti","Ejisu Municipal":"Ashanti",
    "Western Region":"Western","Western":"Western",
    "Eastern Region":"Eastern","Eastern":"Eastern",
    "Central Region":"Central","Central":"Central","Central Ghana":"Central","KEEA":"Central",
    "Cape Coast Municipal":"Central","Awutu Senya East":"Central","Awutu Senya East Municipal":"Central",
    "Volta Region":"Volta","Volta":"Volta","Central Tongu District":"Volta","South Tongu":"Volta",
    "Northern Region":"Northern","Northern":"Northern","Tamale Metropolitan":"Northern",
    "Upper East Region":"Upper East","Upper East":"Upper East",
    "Upper West Region":"Upper West","Upper West":"Upper West",
    "Sissala West District":"Upper West","Sissala East District":"Upper West",
    "Oti Region":"Oti","Oti":"Oti",
    "Bono East Region":"Bono East","Bono East":"Bono East","Techiman Municipal":"Bono East",
    "Ahafo Region":"Ahafo","Ahafo":"Ahafo","Asutifi South":"Ahafo",
    "Savannah Region":"Savannah","Savannah":"Savannah","Damongo":"Savannah",
    "North East Region":"North East","North East":"North East",
    "Western North Region":"Western North","Western North":"Western North",
    "Brong Ahafo Region":"Brong-Ahafo","Brong-Ahafo Region":"Brong-Ahafo",
    "Brong Ahafo":"Bono","Brong-Ahafo":"Bono",
    "Bono Region":"Bono","Bono":"Bono","Sunyani":"Bono",
}
 
# v13-2: Massively extended CITY_TO_REGION (+80 entries covering previously Unknown cities)
CITY_TO_REGION: Dict[str, str] = {
    # Greater Accra
    "accra":"Greater Accra","tema":"Greater Accra","ashaiman":"Greater Accra","madina":"Greater Accra",
    "east legon":"Greater Accra","north legon":"Greater Accra","cantonments":"Greater Accra","dansoman":"Greater Accra",
    "achimota":"Greater Accra","lapaz":"Greater Accra","spintex":"Greater Accra","osu":"Greater Accra",
    "adenta":"Greater Accra","teshie":"Greater Accra","nungua":"Greater Accra","adabraka":"Greater Accra",
    "jamestown":"Greater Accra","labadi":"Greater Accra","kokomlemle":"Greater Accra","amasaman":"Greater Accra",
    "kwabenya":"Greater Accra","dome":"Greater Accra","oyarifa":"Greater Accra","airport residential":"Greater Accra",
    "awoshie":"Greater Accra","weija":"Greater Accra","haatso":"Greater Accra","east cantonments":"Greater Accra",
    "roman ridge":"Greater Accra","kaneshie":"Greater Accra","north kaneshie":"Greater Accra",
    "darkuman":"Greater Accra","chorkor":"Greater Accra","okponglo":"Greater Accra","legon":"Greater Accra",
    "agbogba":"Greater Accra","mempeasem":"Greater Accra","ashale-botwe":"Greater Accra","dzorwulu":"Greater Accra",
    "klagon":"Greater Accra","odorkor":"Greater Accra","pokoase":"Greater Accra","pantang":"Greater Accra",
    "accra central":"Greater Accra","agbogbloshie":"Greater Accra","tesano":"Greater Accra","labone":"Greater Accra",
    "ridge":"Greater Accra","airport city":"Greater Accra","accra newtown":"Greater Accra","dodowa":"Greater Accra",
    "kpone":"Greater Accra","la":"Greater Accra","pokuase":"Greater Accra","ga south":"Greater Accra",
    "ga east":"Greater Accra","kwashieman":"Greater Accra","manhean":"Greater Accra","adentan":"Greater Accra",
    "ashiaman":"Greater Accra","sowutuom":"Greater Accra","abokobi":"Greater Accra","ablekuma":"Greater Accra",
    "kasoa":"Central",  # NOTE: Kasoa is Central not Greater Accra — hard rule
    # Ashanti
    "kumasi":"Ashanti","obuasi":"Ashanti","ejisu":"Ashanti","asokore":"Ashanti","atonsu":"Ashanti",
    "suame":"Ashanti","bantama":"Ashanti","nhyiaeso":"Ashanti","asawase":"Ashanti","tafo":"Ashanti",
    "kwadaso":"Ashanti","mampong":"Ashanti","nkawie":"Ashanti","kaase":"Ashanti","bekwai":"Ashanti",
    "agona ashanti":"Ashanti","abuakwa":"Ashanti","agogo":"Ashanti","santasi":"Ashanti","buokrom":"Ashanti",
    "manhyia":"Ashanti","asokwa":"Ashanti","kronum":"Ashanti","ahodwo":"Ashanti","offinso":"Ashanti",
    "kumawu":"Ashanti","konongo":"Ashanti","tepa":"Ashanti","juaben":"Ashanti",
    "kenyase":"Ashanti","asokore mampong":"Ashanti","ejisu juaben":"Ashanti",
    # v13 new Ashanti entries
    "asuofia":"Ashanti","asuofua":"Ashanti","kuntanase":"Ashanti","sekyere":"Ashanti",
    "juaso":"Ashanti","ejura":"Ashanti","tikrom":"Ashanti","mankranso":"Ashanti",
    # Western
    "takoradi":"Western","sekondi":"Western","axim":"Western","tarkwa":"Western","half assini":"Western",
    "prestea":"Western","bogoso":"Western","sefwi asawinso":"Western","sefwi wiawso":"Western",
    "daboase":"Western","apremdo":"Western","aboadze":"Western","kwesimintsim":"Western",
    "mataheko":"Western","new takoradi":"Western","nsuta":"Western","sekondi-takoradi":"Western",
    "shama":"Western","agona nkwanta":"Western","inchaban":"Western","apowa":"Western",
    "benso":"Western","dompim":"Western","nsuta-wassaw":"Western","dompoase":"Western",
    # Western North
    "bibiani":"Western North","sefwi bodi":"Western North","juaboso":"Western North",
    "anhwiaso":"Western North","enchi":"Western North","sefwi bekwai":"Western North",
    "asankrangua":"Western North","sefwi boinzan":"Western North","boinzan":"Western North",
    # Eastern
    "koforidua":"Eastern","nkawkaw":"Eastern","suhum":"Eastern","somanya":"Eastern",
    "akyem oda":"Eastern","kade":"Eastern","asamankese":"Eastern","mpraeso":"Eastern",
    "abetifi":"Eastern","akosombo":"Eastern","mampong-akwapim":"Eastern","nsawam":"Eastern",
    "nsawam adoagyiri":"Eastern","akim oda":"Eastern","oda":"Eastern","nkwantanan":"Eastern",
    "kyebi":"Eastern","begoro":"Eastern","akim swedru":"Eastern","akwatia":"Eastern",
    # v13 new Eastern entries
    "abomosu":"Eastern","kwabeng":"Eastern","new abirim":"Eastern","enyinabrim":"Eastern",
    # Central
    "cape coast":"Central","winneba":"Central","saltpond":"Central","ajumako":"Central",
    "mankessim":"Central","ankaful":"Central","buduburam":"Central","elmina":"Central",
    "agona swedru":"Central","assin fosu":"Central","cabo corso":"Central","cape-coast":"Central",
    "awutu bereku":"Central","bawjiase":"Central","dunkwa":"Central","assin north":"Central",
    "twifo ati-morkwa":"Central","swedru":"Central","breman asikuma":"Central",
    # v13 new Central entries
    "abura":"Central","asin":"Central","abura asebu":"Central","assin":"Central",
    "nyakrom":"Central","gomoa":"Central","mumford":"Central","apam":"Central",
    # Volta
    "ho":"Volta","hohoe":"Volta","keta":"Volta","akatsi":"Volta","aflao":"Volta","kpandu":"Volta",
    "sogakope":"Volta","battor":"Volta","anloga":"Volta","adidome":"Volta","anfoega":"Volta",
    "jasikan":"Volta","denu":"Volta","dzodze":"Volta","ve golokwati":"Volta",
    "kpeta":"Volta","peki":"Volta","nkonya":"Volta","ateiku":"Volta",
    # Oti
    "dambai":"Oti","nkwanta":"Oti","yabologu":"Oti","kpassa":"Oti","buem":"Oti",
    # Northern
    "tamale":"Northern","walewale":"Northern","yendi":"Northern","savelugu":"Northern",
    "tolon":"Northern","saboba":"Northern","bimbila":"Northern","gushegu":"Northern",
    "karaga":"Northern","kumbungu":"Northern","kpandai":"Northern",
    # Savannah
    "salaga":"Savannah","damongo":"Savannah","bole":"Savannah","sawla":"Savannah","buipe":"Savannah",
    # North East
    "nalerigu":"North East","kparigu":"North East","chereponi":"North East",
    # Bono East
    "techiman":"Bono East","nkoranza":"Bono East","kintampo":"Bono East","atebubu":"Bono East",
    "yeji":"Bono East","prang":"Bono East","anyinasusu":"Bono East","anyinasuso":"Bono East",
    # Bono / Brong-Ahafo
    "sunyani":"Bono","berekum":"Bono","banda":"Bono","wenchi":"Bono","dormaa ahenkro":"Bono",
    "abesim":"Bono","dormaa":"Bono","jaman north":"Bono","jaman south":"Bono",
    # Ahafo
    "goaso":"Ahafo","bechem":"Ahafo","duayaw nkwanta":"Ahafo","kukuom":"Ahafo",
    "kenyasi":"Ahafo","acherensua":"Ahafo","mim":"Ahafo","hwidiem":"Ahafo","mepom":"Ahafo",
    # Upper East
    "bolgatanga":"Upper East","navrongo":"Upper East","bawku":"Upper East","zebilla":"Upper East",
    "sandema":"Upper East","bongo":"Upper East","paga":"Upper East","chiana":"Upper East",
    "tongo":"Upper East",
    # Upper West
    "wa":"Upper West","lawra":"Upper West","nandom":"Upper West","jirapa":"Upper West",
    "nadawli":"Upper West","daffiama":"Upper West","wechiau":"Upper West",
    "kaleo":"Upper West","issa":"Upper West","gwolu":"Upper West","eremon":"Upper West",
}
 
REGION_TEXT_SIGNALS = [
    ("greater accra","Greater Accra"),("accra","Greater Accra"),("tema","Greater Accra"),
    ("ashanti","Ashanti"),("kumasi","Ashanti"),("takoradi","Western"),("sekondi","Western"),
    ("koforidua","Eastern"),("kasoa","Central"),("cape coast","Central"),("tamale","Northern"),
    ("bolgatanga","Upper East"),("wa","Upper West"),("sunyani","Bono"),("techiman","Bono East"),
    ("goaso","Ahafo"),("damongo","Savannah"),("nalerigu","North East"),("bibiani","Western North"),
    ("dambai","Oti"),("ho","Volta"),("bole","Savannah"),("yendi","Northern"),("bawku","Upper East"),
    ("western north","Western North"),("brong-ahafo","Bono"),("brong ahafo","Bono"),
    ("upper east","Upper East"),("upper west","Upper West"),("bono east","Bono East"),
]
 
def _normalise_region(state_val, city_val, name_val, desc_val, addr1, addr2, addr3):
    city_lower_raw = str(city_val or "").strip().lower()
    if "kasoa" in city_lower_raw: return ("Central","kasoa_hard_override",1.0)
    all_text_lower = " ".join(str(x or "").lower() for x in [name_val,addr1,addr2,addr3])
    if re.search(r"\bkasoa\b", all_text_lower): return ("Central","kasoa_hard_override",0.95)
 
    if state_val and str(state_val).strip() not in ("","null","None","nan"):
        r = str(state_val).strip()
        if r.lower() not in ("ghana","gh"):
            mapped = REGION_NORM_MAP.get(r) or REGION_NORM_MAP.get(r.title())
            if not mapped:
                stripped = re.sub(r"\s*[Rr]egion$","",r).strip()
                mapped = REGION_NORM_MAP.get(stripped) or REGION_NORM_MAP.get(stripped.title())
            if not mapped and r.title() in VALID_REGIONS: mapped = r.title()
            if mapped and mapped in VALID_REGIONS: return (mapped,"state_field",1.0)
 
    if city_val and str(city_val).strip() not in ("","null","None","nan"):
        city_lower = str(city_val).strip().lower()
        region = CITY_TO_REGION.get(city_lower)
        if region: return (region,"city_lookup",1.0)
        for city_key, reg in CITY_TO_REGION.items():
            if city_lower.startswith(city_key) or city_key.startswith(city_lower):
                if len(city_key) >= 4:
                    return (reg,"city_lookup",0.85)
 
    search_text = " ".join([str(x or "").lower() for x in [name_val,desc_val,addr1,addr2,addr3]])
    for signal, region in REGION_TEXT_SIGNALS:
        if signal in search_text: return (region,"text_signal",0.65)
 
    return ("Unknown","unknown",0.0)
 
@pandas_udf(ArrayType(StringType()))
def normalise_region_udf(state,city,name,desc,addr1,addr2,addr3):
    result = []
    for args in zip(state,city,name,desc,addr1,addr2,addr3):
        r,src,conf = _normalise_region(*args)
        result.append([r,src,str(conf)])
    return pd.Series(result)
 
region_result = normalise_region_udf(
    F.col("address_stateOrRegion"),F.col("address_city"),F.col("name"),
    F.col("description"),F.col("address_line1"),F.col("address_line2"),F.col("address_line3"),
)
df = df \
    .withColumn("_ra",            region_result) \
    .withColumn("region_normalised", F.col("_ra").getItem(0)) \
    .withColumn("region_source",  F.col("_ra").getItem(1)) \
    .withColumn("region_confidence", F.col("_ra").getItem(2).cast(FloatType())) \
    .drop("_ra")
 
unknown_pct = df.filter(F.col("region_normalised") == "Unknown").count() / df.count() * 100
print(f"Region normalisation done. Unknown: {unknown_pct:.1f}%  (target <2%)")

# COMMAND ----------
# MAGIC %md ## 13. Facility Type Inference + PDF-safe Split

# COMMAND ----------

_NAME_FTYPE_OVERRIDES: List[Tuple[re.Pattern, str, float]] = [
    (re.compile(r"\bCHPS\b|\bCommunity\s+Health\s+Planning\s+and\s+Services?\b|\bCommunity\s+Health\s+Post\b", re.I), "chps", 0.98),
    (re.compile(r"\bRCH\b(?!\s+hospital)", re.I), "clinic", 0.92),
    (re.compile(r"\bHealth\s+Cent(?:re|er)\b(?!.*\bHospital\b)", re.I), "clinic", 0.92),
    (re.compile(r"\bPolyclinic\b", re.I), "clinic", 0.95),
    (re.compile(r"\bHerbal\b|\bTraditional\s+(?:Medicine|Healer|Clinic|Hospital)\b|\bHomeopathic\b|\bSpiritual\s+(?:Hospital|Clinic|Healer)\b", re.I), "alternative_medicine", 0.95),
    (re.compile(r"\bDiagnostic\s+(?:Cent(?:re|er)|Laboratory|Lab|Services?)\b|\bMedical\s+Laborator(?:y|ies)\b|\bMedical\s+Lab\b", re.I), "diagnostic_center", 0.90),
    (re.compile(r"\bMaternity\s+(?:Home|Clinic|Unit)\b", re.I), "maternity_home", 0.95),
    (re.compile(r"\bEye\s+(?:Care|Clinic|Cent(?:re|er)|Hospital)\b|\bOptical\s+(?:Clinic|Cent(?:re|er))\b", re.I), "eye_clinic", 0.95),
    (re.compile(r"\bPharmac(?:y|ies)\b|\bChemist\b|\bChemical\s+(?:Seller|Shop|Store)\b", re.I), "pharmacy", 0.92),
    (re.compile(r"\bDental\s+(?:Cent(?:re|er)|Clinic|Hospital)\b|\bDentist(?:ry)?\b", re.I), "dentist", 0.93),
]

_FTYPE_TYPO_MAP = {
    "farmacy":"pharmacy","pharamcy":"pharmacy","pharmcy":"pharmacy",
    "hospitl":"hospital","clinc":"clinic","dentits":"dentist","doctr":"doctor"
}

_FTYPE_RULES = [
    (re.compile(r"\b(?:teaching|referral|specialist|regional|district|municipal|general|primary|secondary|mission|CHAG|military|university|psychiatric|tertiary)\s+hospital\b", re.I), "hospital", 0.95),
    (re.compile(r"\bhospital\b", re.I), "hospital", 0.90),
    (re.compile(r"\bmedical\s+(?:complex|centre|center)\b", re.I), "hospital", 0.85),
    (re.compile(r"\bhealth\s+complex\b", re.I), "hospital", 0.80),
    (re.compile(r"\bpharmac(?:y|ies|eutical|ist)\b", re.I), "pharmacy", 0.90),
    (re.compile(r"\bchemist\b|\bdrug\s+(?:store|shop)\b", re.I), "pharmacy", 0.85),
    (re.compile(r"\b(?:licensed\s+)?chemical\s+(?:seller|shop|store)\b|\bOTCMS\b|\bLCS\b", re.I), "pharmacy", 0.82),
    (re.compile(r"\bdental\s+(?:centre|center|clinic|hospital)\b", re.I), "dentist", 0.92),
    (re.compile(r"\bdental\b|\bdentist\b|\bdentistry\b", re.I), "dentist", 0.88),
    (re.compile(r"\beye\s+(?:care|clinic|centre|center|hospital)\b|\boptical\s+(?:clinic|centre|center)\b", re.I), "eye_clinic", 0.92),
    (re.compile(r"\bophthalmolog(?:y|ist)\b|\boptometr(?:y|ist)\b", re.I), "eye_clinic", 0.88),
    (re.compile(r"\bchps\b|\bcommunity\s+health\s+planning\s+and\s+services?\b", re.I), "chps", 0.92),
    (re.compile(r"\bdiagnostic\s+(?:cent(?:er|re)|laboratory|lab|services?)\b", re.I), "diagnostic_center", 0.88),
    (re.compile(r"\bmedical\s+laborator(?:y|ies)\b|\bmedical\s+lab\b", re.I), "diagnostic_center", 0.85),
    (re.compile(r"\bmaternity\s+(?:home|clinic|unit|centre|center)\b", re.I), "maternity_home", 0.92),
    (re.compile(r"\bgeneral\s+practitioner\b|\bprivate\s+practice\b|\bphysician\b", re.I), "doctor", 0.85),
    (re.compile(r"\bdr\.?\s+[a-z]+\b", re.I), "doctor", 0.68),
    (re.compile(r"\bclinic\b|\bpolyclinic\b", re.I), "clinic", 0.85),
    (re.compile(r"\bcommunity\s+health\s+(?:center|centre)\b", re.I), "clinic", 0.82),
    (re.compile(r"\bhealth\s+cent(?:er|re)\b", re.I), "clinic", 0.80),
    (re.compile(r"\bhealth\s+post\b|\bhealth\s+facilit(?:y|ies)\b", re.I), "clinic", 0.75),
    (re.compile(r"\bherbal\b|\btraditional\s+medicine\b|\bhomeopathic\b", re.I), "alternative_medicine", 0.85),
]
_HYBRID_NGO_CLINICAL = re.compile(r"\b(?:chag|health\s+service|district\s+health|regional\s+health|ghana\s+health\s+service|private\s+health)\b", re.I)

def _infer_facility_type(existing, name, description, capability_items, org_type):
    name_str = str(name or "").strip()
    for pattern, ftype, conf in _NAME_FTYPE_OVERRIDES:
        if pattern.search(name_str): return ftype, conf
    raw = str(existing or "").strip().lower()
    if raw and raw not in ("null","none",""):
        corrected = _FTYPE_TYPO_MAP.get(raw, raw)
        if corrected in EXTENDED_TO_PDF_ENUM:
            return corrected, (0.95 if corrected in ("hospital","pharmacy") else 0.85)
    org_lower  = str(org_type or "").strip().lower()
    desc_str   = str(description or "").lower()
    name_lower = name_str.lower()
    if org_lower == "ngo":
        combined_text = f"{name_lower} {desc_str}"
        if not (_HYBRID_NGO_CLINICAL.search(combined_text) or _CLINICAL_INDICATORS.search(desc_str[:200])):
            return None, 0.1
    cap_text = " ".join([str(c or "") for c in (capability_items or [])]).lower()
    for text, weight in [(name_lower,1.0),(cap_text,0.9),(desc_str,0.8)]:
        if not text or text.strip() in ("null","none",""): continue
        for pattern, ftype, base_conf in _FTYPE_RULES:
            if pattern.search(str(text)):
                conf = round(base_conf * weight, 3)
                if org_lower == "ngo": conf = round(conf * 0.7, 3)
                return ftype, conf
    if _CLINICAL_INDICATORS.search(f"{name_lower} {desc_str} {cap_text}"):
        return "clinic", (0.30 if org_lower == "ngo" else 0.35)
    if org_lower == "facility": return "clinic", 0.55
    return None, 0.0

@pandas_udf(ArrayType(StringType()))
def infer_facility_type_udf(existing, name, desc, cap, org):
    result = []
    for ex, nm, dc, ca, ot in zip(existing, name, desc, cap, org):
        cap_items = list(ca) if isinstance(ca,(list,tuple)) else []
        ftype, conf = _infer_facility_type(ex, nm, dc, cap_items, ot)
        result.append([ftype or "", str(conf)])
    return pd.Series(result)

ftype_result = infer_facility_type_udf(F.col("facilityTypeId"), F.col("name"), F.col("description"), F.col("capability_parsed"), F.col("organization_type"))
df = df \
    .withColumn("_ft",    ftype_result) \
    .withColumn("facility_type_clean", F.when(F.col("_ft").getItem(0) == "", F.lit(None)).otherwise(F.col("_ft").getItem(0))) \
    .withColumn("facility_type_confidence", F.col("_ft").getItem(1).cast(DoubleType())) \
    .drop("_ft")

mapping_expr = F.create_map(*[F.lit(x) for x in sum([[k,v] for k,v in EXTENDED_TO_PDF_ENUM.items()],[])])
df = df.withColumn("facility_type_clean_pdf",
    F.when(F.col("facility_type_clean").isNotNull(), mapping_expr[F.lower(F.col("facility_type_clean"))]).otherwise(F.lit(None)))

df = df.withColumn("facilityTypeId", F.coalesce(F.col("facility_type_clean_pdf"), F.col("facilityTypeId")))

@pandas_udf(ArrayType(StringType()))
def add_baseline_capabilities_udf(cap, ftype, name):
    result = []
    for caps, ft, nm in zip(cap, ftype, name):
        lst = list(caps) if caps is not None else []
        ft_str = str(ft or "").lower(); nm_lower = str(nm or "").lower()
        if not lst and ft_str == "hospital":
            baseline = ["General medical consultation","Inpatient admission","Outpatient consultation"]
            if "military" in nm_lower: baseline.append("Emergency care")
            if "psychiatric" in nm_lower: baseline += ["Mental health services","Psychiatric care"]
            if "teaching" in nm_lower or "university" in nm_lower: baseline += ["Medical training","Specialist referral services"]
            lst = baseline
        result.append(lst)
    return pd.Series(result)

df = df.withColumn("capability_parsed", add_baseline_capabilities_udf(F.col("capability_parsed"), F.col("facility_type_clean"), F.col("name")))

_FNAME_HINTS = {
    "eye_clinic": re.compile(r"\b(eye\s+clinic|eye\s+care|optical|ophthalmolog|optometr)\b", re.I),
    "maternity_home": re.compile(r"\b(maternity\s+home|maternity\s+clinic|maternity\s+unit)\b", re.I),
    "chps": re.compile(r"\b(chps|community\s+health\s+planning\s+and\s+services)\b", re.I),
    "diagnostic_center": re.compile(r"\b(diagnostic\s+cent|medical\s+lab|laboratory)\b", re.I),
    "pharmacy": re.compile(r"\b(pharmacy|chemist|chemical\s+seller)\b", re.I),
    "dentist": re.compile(r"\b(dental|dentist)\b", re.I),
    "alternative_medicine": re.compile(r"\b(herbal|traditional\s+medicine|homeopathic)\b", re.I),
}

@pandas_udf(StringType())
def infer_ftype_from_name_udf(name: pd.Series) -> pd.Series:
    def _infer(n):
        s = str(n or "")
        for label, pat in _FNAME_HINTS.items():
            if pat.search(s):
                return label
        return None
    return name.apply(_infer)

df = df.withColumn("_ft_name_hint", infer_ftype_from_name_udf(F.col("name")))
df = df.withColumn("facility_type_clean",
    F.when(F.col("facility_type_clean").isNull(), F.col("_ft_name_hint")).otherwise(F.col("facility_type_clean"))
).drop("_ft_name_hint")

mapping_expr = F.create_map(*[F.lit(x) for x in sum([[k,v] for k,v in EXTENDED_TO_PDF_ENUM.items()],[])])
df = df.withColumn("facility_type_clean_pdf",
    F.when(F.col("facility_type_clean").isNotNull(), mapping_expr[F.lower(F.col("facility_type_clean"))]).otherwise(F.lit(None)))
df = df.withColumn("facilityTypeId", F.coalesce(F.col("facility_type_clean_pdf"), F.col("facilityTypeId")))

df = df \
    .withColumn("is_hospital", F.col("facility_type_clean") == "hospital") \
    .withColumn("is_clinic", F.col("facility_type_clean") == "clinic") \
    .withColumn("is_eye_clinic", F.col("facility_type_clean") == "eye_clinic") \
    .withColumn("is_maternity_home", F.col("facility_type_clean") == "maternity_home") \
    .withColumn("is_chps", F.col("facility_type_clean") == "chps") \
    .withColumn("is_diagnostic_center", F.col("facility_type_clean") == "diagnostic_center") \
    .withColumn("is_pharmacy", F.col("facility_type_clean") == "pharmacy") \
    .withColumn("is_dentist", F.col("facility_type_clean") == "dentist") \
    .withColumn("is_alternative_medicine", F.col("facility_type_clean") == "alternative_medicine")

print("Facility type inference done.")
df.groupBy("facility_type_clean").count().orderBy(F.desc("count")).show()

# COMMAND ----------
# MAGIC %md ## 14. operatorTypeId — Full Inference (v12: government category fed in)

# COMMAND ----------

_OPERATOR_TEXT_RULES: List[Tuple[re.Pattern, str]] = [
    (re.compile(r"(?i)\b(?:government[\s-]owned|government\s+facilit|ghana\s+health\s+service|public\s+(?:hospital|health|facilit)|district\s+(?:hospital|assembly|health)|regional\s+(?:hospital|health)|military\s+hospital|teaching\s+hospital|municipal\s+(?:hospital|assembly)|national\s+health\s+(?:service|authority)|ghanaian\s+government|ministry\s+of\s+health)\b"), "public"),
    (re.compile(r"(?i)\b(?:mission\s+hospital|faith[\s-]based|christian|catholic|adventist|presbyterian|methodist|islamic|muslim|ahmadiyya|seventh[\s-]day|salvation\s+army|bishop|archbishop|church\s+of|CHAG)\b"), "private"),
    (re.compile(r"(?i)\b(?:privately[\s-]owned|private\s+(?:hospital|clinic|practice|facilit)|private\s+sector|self[\s-](?:owned|funded)|family[\s-]owned)\b"), "private"),
    (re.compile(r"(?i)\b(?:ngo|non[\s-]governmental|non\s+profit|nonprofit|foundation|trust\s+hospital|charitable|charity)\b"), "private"),
    (re.compile(r"(?i)^(?:district|regional|municipal|national|government|public)\s+(?:hospital|health|clinic)"), "public"),
    (re.compile(r"(?i)\b(?:teaching\s+hospital|referral\s+hospital|polyclinic)\b"), "public"),
    (re.compile(r"(?i)\b(?:military|army|navy|air\s+force|ghana\s+armed\s+forces|police\s+(?:hospital|service))\b"), "public"),
    (re.compile(r"(?i)\b(?:limited|ltd\.?|llc\.?|plc\.?|inc\.?|incorporated|company|enterprise)\b"), "private"),
]

@pandas_udf(StringType())
def infer_operator_type_udf(op_raw, desc, caps, org_type, name, ftype, org_cat):
    result = []
    for op, d, c, org, nm, ft, ocat in zip(op_raw, desc, caps, org_type, name, ftype, org_cat):
        op_str   = "" if pd.isna(op)  else str(op).strip().lower()
        desc_str = "" if pd.isna(d)   else str(d)
        org_str  = "" if pd.isna(org) else str(org).strip().lower()
        name_str = "" if pd.isna(nm)  else str(nm).strip()
        ft_str   = "" if pd.isna(ft)  else str(ft).strip().lower()
        ocat_str = "" if pd.isna(ocat) else str(ocat).strip().lower()
        try: cap_list = list(c) if c is not None else []
        except: cap_list = []

        if op_str in ("public","private"):
            result.append(op_str); continue

        if op_str in ("government",): op_str = "public"
        elif op_str in ("ngo","faith-based","mission","company"): op_str = "private"

        if op_str in ("public","private"):
            result.append(op_str); continue

        # v12: Use organization_category as a direct signal
        if ocat_str == "government":
            result.append("public"); continue
        if ocat_str in ("ngo","faith_based","private"):
            result.append("private"); continue

        combined = f"{name_str} {desc_str} {' '.join(str(x) for x in cap_list)} {org_str}"
        inferred = None
        for pattern, op_type in _OPERATOR_TEXT_RULES:
            if pattern.search(combined):
                inferred = op_type; break

        if inferred is None:
            if org_str in ("ngo","non-profit","foundation"): inferred = "private"

        if inferred is None:
            if ft_str in ("pharmacy","dentist","doctor","eye_clinic","diagnostic_center","alternative_medicine","maternity_home"):
                inferred = "private"
            elif ft_str == "chps":
                inferred = "public"
            elif ft_str == "hospital":
                if re.search(r"(?i)\b(?:district|regional|municipal|teaching|government|public|military|police|37\s+military|korle|komfo|ridge|tamale\s+teaching|cape\s+coast\s+teaching|ho\s+teaching|effia)\b", combined):
                    inferred = "public"
                else:
                    inferred = "private"
            else:
                inferred = "private"

        result.append(inferred)
    return pd.Series(result)

df = df.withColumn(
    "operatorTypeId",
    infer_operator_type_udf(
        F.col("operatorTypeId"), F.col("description"), F.col("capability_parsed"),
        F.col("organization_type"), F.col("name"), F.col("facility_type_clean"),
        F.col("organization_category"),
    )
)

print("operatorTypeId inference complete (v12: organization_category fed in):")
df.groupBy("operatorTypeId").count().show()
null_op = df.filter(F.col("operatorTypeId").isNull()).count()
print(f"Null operatorTypeId after UDF: {null_op}  (should be 0)")

# COMMAND ----------
# MAGIC %md ## 14.5. Source Trust

# COMMAND ----------

@pandas_udf(StringType())
def classify_source_trust_udf(source_url, official_website, fb, tw, ig, facility_type_confidence, desc):
    result = []
    for url, web, fb_v, tw_v, ig_v, ft_conf, desc_val in zip(source_url, official_website, fb, tw, ig, facility_type_confidence, desc):
        url_s  = str(url or "").lower()
        web_s  = str(web or "").lower()
        fb_s   = str(fb_v or "").lower()
        desc_s = str(desc_val or "")
        try: ft_conf_f = float(ft_conf or 0.0)
        except: ft_conf_f = 0.0

        if web_s and "facebook" not in web_s and "twitter" not in web_s and "instagram" not in web_s:
            result.append("structured"); continue

        if "facebook" in url_s or "facebook" in fb_s:
            result.append("social_metadata"); continue
        if "twitter" in url_s or "x.com" in url_s:
            result.append("social_metadata"); continue
        if "instagram" in url_s or "instagram" in str(ig_v or "").lower():
            result.append("social_metadata"); continue

        if desc_s and _CLINICAL_INDICATORS.search(desc_s) and ft_conf_f >= 0.5:
            result.append("text_extracted"); continue

        if ft_conf_f >= 0.7:
            result.append("text_extracted"); continue

        if ft_conf_f >= 0.3:
            result.append("inferred"); continue

        result.append("unknown")
    return pd.Series(result)

df = df.withColumn("source_trust",
    classify_source_trust_udf(
        F.col("source_url"), F.col("officialWebsite"), F.col("facebooklink"),
        F.col("twitterlink"), F.col("instagramlink"), F.col("facility_type_confidence"),
        F.col("description"),
    ))

df = df \
    .withColumn("source_type",
        F.when(F.col("source_trust") == "social_metadata", F.lit("social"))
         .when(F.col("source_trust").isin("structured","text_extracted"), F.lit("structured"))
         .otherwise(F.lit("unknown"))
    ) \
    .withColumn("source_trust_score",
        F.when(F.col("source_trust") == "structured", F.lit(0.95))
         .when(F.col("source_trust") == "text_extracted", F.lit(0.75))
         .when(F.col("source_trust") == "inferred", F.lit(0.55))
         .when(F.col("source_trust") == "social_metadata", F.lit(0.35))
         .otherwise(F.lit(0.20))
    )

print("Source trust done:")
df.groupBy("source_trust").count().show()

# COMMAND ----------
# MAGIC %md ## 15. Organisation Type Clean & City Normalisation (v12-12: better city backfill)

# COMMAND ----------

df = df.withColumn(
    "organization_type_clean",
    F.when(F.lower(F.col("organization_type")) == "ngo",      F.lit("ngo"))
     .when(F.lower(F.col("organization_type")) == "facility",  F.lit("facility"))
     .when(F.col("missionstatement").isNotNull() & (F.length(F.trim(F.col("missionstatement"))) > 20) & F.col("organization_type").isNull(), F.lit("ngo"))
     .otherwise(F.lit(None).cast(StringType()))
)

CITY_FIX = {
    "accra central":"Accra","greater accra":"Accra","sekondi-takoradi":"Takoradi",
    "osu – accra east":"Osu","accra east":"Accra","accra north":"Accra",
    "kumasi metropolitan":"Kumasi","kumasi metro":"Kumasi","atonsu kumasi":"Atonsu",
    "abesim - sunyani":"Abesim","cabo corso":"Cape Coast","cape-coast":"Cape Coast",
    "ghana":None,"tamale metropolitan":"Tamale","tamale metropolis":"Tamale",
    "la ghana":"Accra","la - accra":"Accra",
}

@pandas_udf(StringType())
def normalise_city_udf(city, addr1, addr2, name):
    # v12-12: Also mine from addr2 and name for city backfill
    result = []
    for city_val, addr1_val, addr2_val, name_val in zip(city, addr1, addr2, name):
        c = str(city_val or "").strip()
        if c.lower() in ("","null","none","nan"): c = ""

        # Backfill from address fields
        if not c:
            for source in [addr1_val, addr2_val]:
                if source:
                    a = str(source).lower()
                    for k in sorted(CITY_TO_REGION.keys(), key=len, reverse=True):
                        if k in a and len(k) > 4:
                            c = k.title(); break
                if c: break

        # Backfill from name as last resort
        if not c and name_val:
            nm = str(name_val).lower()
            for k in sorted(CITY_TO_REGION.keys(), key=len, reverse=True):
                if k in nm and len(k) > 5:
                    c = k.title(); break

        if not c: result.append(None); continue
        c_lower = c.lower()
        if c_lower in CITY_FIX: result.append(CITY_FIX[c_lower]); continue
        c = re.sub(r"\s+(?:municipal(?:ity)?|district|metropol(?:is|itan)?|assembly)\s*$","",c,flags=re.I)
        result.append(c.strip().title() if c.strip() else None)
    return pd.Series(result)

df = df.withColumn("city_clean", normalise_city_udf(
    F.col("address_city"), F.col("address_line1"), F.col("address_line2"), F.col("name")
))
print("Organisation type + city done.")
city_null = df.filter(F.col("city_clean").isNull()).count()
print(f"city_clean nulls: {city_null}")

# COMMAND ----------
# MAGIC %md ## 16. Geocoding (static dict)

# COMMAND ----------

CITY_COORDS: Dict[str, Tuple[float, float]] = {
    "Accra":(5.6037,-0.1870),"Tema":(5.6698,-0.0166),"Ashaiman":(5.7000,-0.0333),
    "Madina":(5.6833,-0.1667),"East Legon":(5.6333,-0.1667),"Dome":(5.6667,-0.2333),
    "Dansoman":(5.5667,-0.2500),"Nungua":(5.5833,-0.0667),"Teshie":(5.5833,-0.0833),
    "Osu":(5.5667,-0.1833),"Kumasi":(6.6885,-1.6244),"Obuasi":(6.2000,-1.6667),
    "Ejisu":(6.7167,-1.5833),"Mampong":(7.0667,-1.4000),"Takoradi":(4.8845,-1.7554),
    "Sekondi":(4.9333,-1.7000),"Axim":(4.8678,-2.2378),"Tarkwa":(5.3003,-1.9936),
    "Cape Coast":(5.1053,-1.2466),"Winneba":(5.3500,-0.6333),"Saltpond":(5.2000,-1.0667),
    "Mankessim":(5.2667,-1.0167),"Kasoa":(5.5100,-0.4350),"Ho":(6.6000,0.4667),
    "Hohoe":(7.1500,0.4667),"Keta":(5.9167,0.9833),"Akatsi":(6.1167,0.8000),
    "Sogakope":(5.8833,0.5833),"Aflao":(6.1167,1.1833),"Tamale":(9.4075,-0.8533),
    "Yendi":(9.4417,-0.0083),"Walewale":(10.3667,-0.8000),"Savelugu":(9.6167,-0.8167),
    "Bolgatanga":(10.7861,-0.8522),"Navrongo":(10.9000,-1.0833),"Bawku":(11.0583,-0.2417),
    "Wa":(10.0600,-2.5000),"Lawra":(10.6333,-2.9000),"Jirapa":(10.5333,-2.7500),
    "Nandom":(10.8500,-2.7500),"Koforidua":(6.0886,-0.2594),"Nkawkaw":(6.5500,-0.7667),
    "Suhum":(6.0333,-0.4500),"Somanya":(6.1000,-0.0167),"Sunyani":(7.3333,-2.3167),
    "Berekum":(7.4500,-2.5833),"Techiman":(7.5833,-1.9333),"Nkoranza":(7.6667,-1.7000),
    "Kintampo":(8.0500,-1.7167),"Atebubu":(7.7500,-0.9833),"Goaso":(6.8000,-2.5167),
    "Bechem":(7.1500,-2.3000),"Nalerigu":(10.5167,-0.3500),"Damongo":(9.0833,-1.8167),
    "Bole":(9.0167,-2.4833),"Dambai":(8.0667,0.1833),"Nsawam":(5.8000,-0.3500),
    "Dodowa":(5.9000,-0.1167),"Weija":(5.5833,-0.3000),"Tesano":(5.6167,-0.2167),
    "Lapaz":(5.6167,-0.2500),"Agogo":(6.8000,-1.1000),"Kpone":(5.6667,-0.0167),
    "Bibiani":(6.4500,-2.3500),"Wenchi":(7.7500,-2.1000),"Dormaa Ahenkro":(7.3000,-2.7167),
    "Atonsu":(6.6500,-1.6333),"Adenta":(5.7000,-0.1833),"Santasi":(6.6667,-1.6167),
    "Kwashieman":(5.6000,-0.2333),"Abuakwa":(6.0167,-0.2167),"Pokuase":(5.6833,-0.2667),
    "Bantama":(6.7000,-1.6333),"Manhyia":(6.7167,-1.6167),"Asokwa":(6.6833,-1.6167),
    "Suame":(6.7167,-1.6500),"Tafo":(6.7333,-1.6000),"Nkawie":(6.6000,-1.8500),
    "Bekwai":(6.4500,-1.5833),"Offinso":(7.1167,-1.6333),"Asamankese":(5.8667,-0.6500),
    "Oda":(5.9167,-0.9833),"Swedru":(5.5333,-0.6833),"Cape-Coast":(5.1053,-1.2466),
    "Mankessim":(5.2667,-1.0167),"Ankaful":(5.1333,-1.3000),
    "Hohoe":(7.1500,0.4667),"Kpandu":(6.9833,0.3000),
    "Salaga":(8.5500,-0.5167),"Bimbila":(9.8667,0.0667),
    "Tarkwa":(5.3003,-1.9936),"Prestea":(5.4500,-2.1333),
    "Akatsi":(6.1167,0.8000),"Aflao":(6.1167,1.1833),
    "Nkwanta":(8.2833,0.5000),"Jasikan":(7.4000,0.4167),
    "Agona Nkwanta":(5.5167,-0.7667),"Agona Swedru":(5.5333,-0.6833),
    "Assin Fosu":(5.7000,-1.2333),"Dunkwa":(5.9667,-1.7833),
    "Konongo":(6.6167,-1.2167),"Juaben":(6.8000,-1.4167),
    "Mampong-Akwapim":(5.9167,-0.1333),"Akosombo":(6.2833,0.0500),
    "Nkawkaw":(6.5500,-0.7667),"Kade":(6.1000,-0.8333),
}

REGION_CENTROIDS: Dict[str, Tuple[float, float]] = {
    "Greater Accra":(5.6037,-0.1870),"Ashanti":(6.6885,-1.6244),"Western":(4.9016,-2.0214),
    "Eastern":(6.1500,-0.4500),"Central":(5.3600,-1.2200),"Volta":(6.8000,0.3000),
    "Northern":(9.5400,-0.9000),"Upper East":(10.7800,-1.0500),"Upper West":(10.2500,-2.3200),
    "Brong-Ahafo":(7.3500,-1.6300),"Bono East":(7.6000,-1.0000),"Ahafo":(6.9000,-2.3000),
    "Savannah":(8.8000,-1.7500),"North East":(10.5000,-0.3500),"Western North":(6.3000,-2.7000),
    "Oti":(7.4500,0.2000),"Bono":(7.3333,-2.3167),
}

DISTRICT_CENTROIDS: Dict[str, Tuple[float, float]] = {
    "Accra Metro":(5.5500,-0.2167),"Tema Metro":(5.6698,-0.0166),
    "Ga East":(5.7167,-0.1667),"Ga East Municipal":(5.7167,-0.1667),
    "Awutu Senya East":(5.5100,-0.4350),"Cape Coast Metro":(5.1053,-1.2466),
    "Kumasi Metro":(6.6885,-1.6244),"Tamale Metro":(9.4075,-0.8533),
    "Bolgatanga Municipal":(10.7861,-0.8522),"Wa Municipal":(10.0600,-2.5000),
    "Sunyani Municipal":(7.3333,-2.3167),"Techiman Municipal":(7.5833,-1.9333),
    "Ho Municipal":(6.6000,0.4667),
}

def _extract_city_from_text(text):
    if not text: return None
    text_lower = text.lower()
    for city_key in sorted(CITY_COORDS.keys(), key=len, reverse=True):
        if re.search(r'\b' + re.escape(city_key.lower()) + r'\b', text_lower):
            return city_key
    return None

@pandas_udf(ArrayType(StringType()))
def geocode_udf(city,region,addr_state,name,addr1,addr2,addr3):
    result = []
    for city_val,region_val,state_val,name_val,a1,a2,a3 in zip(city,region,addr_state,name,addr1,addr2,addr3):
        c = str(city_val or "").strip().title()
        if c and c in CITY_COORDS:
            lat,lon = CITY_COORDS[c]; result.append([str(lat),str(lon),"static_city_dict","0.90"]); continue
        found_district = False
        for lookup in [c, str(state_val or "").strip().title()]:
            if lookup and lookup in DISTRICT_CENTROIDS:
                lat,lon = DISTRICT_CENTROIDS[lookup]; result.append([str(lat),str(lon),"district_centroid","0.75"]); found_district=True; break
        if found_district: continue
        r = str(region_val or "").strip()
        if r and r in REGION_CENTROIDS:
            lat,lon = REGION_CENTROIDS[r]; result.append([str(lat),str(lon),"region_centroid","0.50"]); continue
        combined_text = " ".join([str(x or "") for x in [name_val,a1,a2,a3]])
        extracted_city = _extract_city_from_text(combined_text)
        if extracted_city and extracted_city in CITY_COORDS:
            lat,lon = CITY_COORDS[extracted_city]; result.append([str(lat),str(lon),"text_extracted_city","0.60"]); continue
        result.append([str(cfg.DEFAULT_LAT),str(cfg.DEFAULT_LON),"country_centroid","0.15"])
    return pd.Series(result)

geo_result = geocode_udf(F.col("city_clean"),F.col("region_normalised"),F.col("address_stateOrRegion"),F.col("name"),F.col("address_line1"),F.col("address_line2"),F.col("address_line3"))
df = df \
    .withColumn("_geo",           geo_result) \
    .withColumn("latitude",       F.col("_geo").getItem(0).cast(DoubleType())) \
    .withColumn("longitude",      F.col("_geo").getItem(1).cast(DoubleType())) \
    .withColumn("geo_source",     F.col("_geo").getItem(2)) \
    .withColumn("geo_confidence", F.col("_geo").getItem(3).cast(DoubleType())) \
    .drop("_geo")

print("Static geocoding done:")
df.groupBy("geo_source").count().show()

# COMMAND ----------
# MAGIC %md ## 16.5. Geopy Nominatim Fallback for country_centroid rows (v12-7)

# COMMAND ----------

# v12-7: Use geopy Nominatim to rescue rows still at country_centroid
try:
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter

    _geolocator = Nominatim(user_agent="virtue_foundation_ghana_silver_v12", timeout=10)
    _geocode_fn  = RateLimiter(_geolocator.geocode, min_delay_seconds=1.2, error_wait_seconds=5.0)

    def _geopy_query(query: str) -> Optional[Tuple[float, float]]:
        """Try Nominatim within Ghana bbox. Returns (lat, lon) or None."""
        if not query or len(query.strip()) < 3: return None
        try:
            loc = _geocode_fn(
                f"{query.strip()}, Ghana",
                country_codes="gh",
                viewbox=[(11.5, -3.5),(4.5, 1.2)],  # Ghana bounding box (N,W) (S,E)
                bounded=True,
            )
            if loc and 4.5 <= loc.latitude <= 11.5 and -3.5 <= loc.longitude <= 1.2:
                return (loc.latitude, loc.longitude)
        except Exception:
            pass
        return None

    # Collect rows still at low-precision centroids
    centroid_rows = (
        df.filter(F.col("geo_source").isin("country_centroid","region_centroid","district_centroid"))
          .select("unique_id","name","city_clean","address_line1","address_line2","address_stateOrRegion")
          .collect()
    )
    print(f"Geopy: attempting to geocode {len(centroid_rows)} low-precision rows ...")

    geocoded_records = []
    for row in centroid_rows:
        queries = [
            str(row.address_line1 or "").strip(),
            str(row.address_line2 or "").strip(),
            str(row.city_clean or "").strip(),
            str(row.address_stateOrRegion or "").strip(),
            str(row.name or "").strip(),
        ]
        for q in queries:
            if q and len(q) > 3:
                coords = _geopy_query(q)
                if coords:
                    geocoded_records.append((str(row.unique_id), coords[0], coords[1]))
                    break

    print(f"Geopy: rescued {len(geocoded_records)} rows from country_centroid.")

    if geocoded_records:
        geo_schema = "unique_id STRING, _geo_lat DOUBLE, _geo_lon DOUBLE"
        geocoded_df = spark.createDataFrame(geocoded_records, schema=geo_schema)
        df = df.join(geocoded_df, on="unique_id", how="left") \
            .withColumn("latitude",     F.when(F.col("_geo_lat").isNotNull(), F.col("_geo_lat")).otherwise(F.col("latitude"))) \
            .withColumn("longitude",    F.when(F.col("_geo_lon").isNotNull(), F.col("_geo_lon")).otherwise(F.col("longitude"))) \
            .withColumn("geo_source",   F.when(F.col("_geo_lat").isNotNull(), F.lit("geopy_nominatim")).otherwise(F.col("geo_source"))) \
            .withColumn("geo_confidence",F.when(F.col("_geo_lat").isNotNull(), F.lit(0.70)).otherwise(F.col("geo_confidence"))) \
            .drop("_geo_lat","_geo_lon")

except Exception as _geo_err:
    print(f"⚠️  Geopy fallback skipped: {_geo_err}")

print("Geocoding (all sources) summary:")
df.groupBy("geo_source").count().show()

df = df \
    .withColumn("geo_precision_tier",
        F.when(F.col("geo_source") == "geopy_nominatim", F.lit(5))
         .when(F.col("geo_source") == "static_city_dict", F.lit(4))
         .when(F.col("geo_source") == "text_extracted_city", F.lit(3))
         .when(F.col("geo_source") == "district_centroid", F.lit(3))
         .when(F.col("geo_source") == "region_centroid", F.lit(2))
         .when(F.col("geo_source") == "country_centroid", F.lit(1))
         .otherwise(F.lit(0))
    ) \
    .withColumn("geo_quality_score",
        F.when(F.col("geo_source") == "geopy_nominatim", F.lit(0.95))
         .when(F.col("geo_source") == "static_city_dict", F.lit(0.85))
         .when(F.col("geo_source") == "text_extracted_city", F.lit(0.70))
         .when(F.col("geo_source") == "district_centroid", F.lit(0.60))
         .when(F.col("geo_source") == "region_centroid", F.lit(0.40))
         .when(F.col("geo_source") == "country_centroid", F.lit(0.20))
         .otherwise(F.lit(0.0))
    )

# COMMAND ----------
# MAGIC %md ## 17. Content Quality Flags & Counts

# COMMAND ----------

df = df \
    .withColumn("procedure_count",  F.size(F.col("procedure_parsed"))) \
    .withColumn("equipment_count",  F.size(F.col("equipment_parsed"))) \
    .withColumn("capability_count", F.size(F.col("capability_parsed"))) \
    .withColumn("specialty_count",  F.size(F.col("specialties_parsed"))) \
    .withColumn("has_procedures",   F.col("procedure_count") > 0) \
    .withColumn("has_equipment",    F.col("equipment_count") > 0) \
    .withColumn("has_capabilities", F.col("capability_count") > 0) \
    .withColumn("has_specialties",  F.col("specialty_count") > 0) \
    .withColumn("has_description",  F.col("description").isNotNull() & (F.length(F.trim(F.col("description"))) > 30)) \
    .withColumn("has_contact",
        (F.size(F.col("phone_numbers_parsed")) > 0) |
        (F.col("email_validity") == True) |
        (F.col("website_primary").isNotNull() & (F.trim(F.col("website_primary")) != "")))

@pandas_udf(BooleanType())
def is_bare_domain_udf(website: pd.Series) -> pd.Series:
    def _check(u):
        if not u or str(u).strip() in ("","null","None","nan"): return False
        u = str(u).strip()
        return not (u.startswith("http://") or u.startswith("https://"))
    return website.apply(_check)

df = df.withColumn("has_bare_website_domain", is_bare_domain_udf(F.col("officialWebsite")))

print("Content flags done.")

# COMMAND ----------
# MAGIC %md ## 18. Official Phone

# COMMAND ----------

df = df.withColumn("official_phone", extract_official_phone_udf(F.col("phone_numbers_parsed")))
df = df \
    .withColumn("phone_country_code", phone_country_code_udf(F.col("official_phone"))) \
    .withColumn("phone_quality_score", phone_quality_score_udf(F.col("official_phone")))
df = df.withColumn("has_contact",
    (F.col("phone_quality_score") >= 0.7) |
    (F.col("email_validity") == True) |
    (F.col("website_primary").isNotNull() & (F.trim(F.col("website_primary")) != "")))

# COMMAND ----------
# MAGIC %md ## 19. Operational Status — Service-Evidence Upgrade

# COMMAND ----------

df = df.withColumn(
    "operational_status",
    F.when(F.col("operational_status") == "closed", F.lit("closed"))
     .when(
         (F.col("operational_status") == "unknown") &
         (F.col("has_procedures") | F.col("has_capabilities") | F.col("has_specialties")) &
         F.col("has_contact"),
         F.lit("open")
     )
     .when(
         (F.col("operational_status") == "unknown") &
         (F.col("has_description")) &
         (F.col("specialty_count") >= 2),
         F.lit("open")
     )
     .otherwise(F.col("operational_status"))
)
 
df = df.withColumn("is_operational", F.col("operational_status") != "closed")
print("Operational status after service-evidence upgrade:")
df.groupBy("operational_status").count().show()

# COMMAND ----------
# MAGIC %md ## 20. RAG Document Text

# COMMAND ----------

def _build_doc_text(name, city, region, ftype, specs, procs, equip, caps, desc):
    parts = []
    seen_clinical: set = set()
    if name and str(name).strip() and str(name).strip() != "UNKNOWN_FACILITY":
        parts.append(f"FACILITY: {name.strip()}")
    loc_parts = [x for x in [city,region] if x and str(x).strip() not in ("","None","Unknown")]
    if loc_parts: parts.append(f"LOCATION: {', '.join(loc_parts)}")
    if ftype and str(ftype).strip(): parts.append(f"TYPE: {ftype.strip()}")

    def _add_section(label, items, limit=15):
        clean = []
        for s in (list(items) if items is not None else []):
            s = str(s).strip()
            if not s or len(s) < 5: continue
            key = re.sub(r"[^\w]","",s.lower())[:30]
            if key not in seen_clinical: seen_clinical.add(key); clean.append(s)
        if clean: parts.append(f"{label}: {', '.join(clean[:limit])}")

    _add_section("SPECIALTIES", specs)
    _add_section("PROCEDURES",  procs)
    _add_section("EQUIPMENT",   equip)
    _add_section("CAPABILITIES",caps)

    if desc and str(desc).strip():
        desc_clean = str(desc).strip()
        if _CLINICAL_INDICATORS.search(desc_clean):
            desc_clean = desc_clean[:400]
            if len(desc_clean) == 400:
                lp = desc_clean.rfind(".")
                if lp > 200: desc_clean = desc_clean[:lp+1]
            parts.append(f"DESCRIPTION: {desc_clean}")
    return "\n".join(parts)

@pandas_udf(StringType())
def build_doc_text_udf(name,city,region,ftype,specs,procs,equip,caps,desc):
    return pd.Series([_build_doc_text(*args) for args in zip(name,city,region,ftype,specs,procs,equip,caps,desc)])

df = df \
    .withColumn("doc_text",
        build_doc_text_udf(F.col("name"),F.col("city_clean"),F.col("region_normalised"),F.col("facility_type_clean"),
            F.col("specialties_parsed"),F.col("procedure_parsed"),F.col("equipment_parsed"),F.col("capability_parsed"),F.col("description"))) \
    .withColumn("doc_text_length", F.length(F.col("doc_text")))

df = df.withColumn(
    "is_rag_ready",
    (F.col("doc_text_length") > 80) &
    (F.col("has_procedures") | F.col("has_equipment") | F.col("has_specialties") | (F.col("has_capabilities") & (F.col("capability_count") >= 2))) &
    (F.col("geo_source") != "country_centroid") &
    F.col("is_operational")
)

print(f"RAG-ready (initial): {df.filter(F.col('is_rag_ready')).count():,} / {df.count():,}")

# COMMAND ----------
# MAGIC %md ## 21. Row Citations

# COMMAND ----------

CITATION_SCHEMA = ArrayType(StructType([
    StructField("field",             StringType(), True),
    StructField("snippet",           StringType(), True),
    StructField("source_column",     StringType(), True),
    StructField("extraction_method", StringType(), True),
    StructField("confidence",        FloatType(),  True),
]))

def _build_citations(proc,equip,cap,spec,desc,proc_raw_count,equip_raw_count,year_int,capacity_int,doctors_int,year_src,capacity_src,doctors_src,year_conf,capacity_conf,doctors_conf):
    citations = []
    def valid(x): return x and len(str(x).strip()) > 5
    raw_procs = int(proc_raw_count or 0); raw_equip = int(equip_raw_count or 0)
    conf_map_year     = {"high":0.95,"medium":0.82,"low":0.65,"inferred":0.45,"extracted_from_capability":0.80,"not_extracted":0.0}
    conf_map_capacity = {"high":0.92,"medium":0.78,"low":0.60,"inferred":0.40,"extracted_from_capability":0.85,"not_extracted":0.0}
    conf_map_doctors  = {"high":0.92,"medium":0.78,"low":0.60,"extracted_from_capability":0.75,"not_extracted":0.0}

    for item in (proc or [])[:3]:
        if valid(item):
            m,s,c = ("direct_json_parse","procedure",0.90) if raw_procs>0 else ("mined_from_description_or_capability","description",0.78)
            citations.append({"field":"procedure_parsed","snippet":str(item).strip()[:100],"source_column":s,"extraction_method":m,"confidence":float(c)})
    for item in (equip or [])[:3]:
        if valid(item):
            m,s,c = ("direct_json_parse","equipment",0.90) if raw_equip>0 else ("mined_or_context_inferred","description",0.72)
            citations.append({"field":"equipment_parsed","snippet":str(item).strip()[:100],"source_column":s,"extraction_method":m,"confidence":float(c)})
    for item in (cap or [])[:2]:
        if valid(item): citations.append({"field":"capability_parsed","snippet":str(item).strip()[:100],"source_column":"capability","extraction_method":"json_parse_filtered","confidence":float(0.82)})
    for item in (spec or [])[:2]:
        if valid(item): citations.append({"field":"specialties_parsed","snippet":str(item).strip()[:100],"source_column":"specialties","extraction_method":"direct_json_parse_normalized","confidence":float(0.95)})
    if year_int is not None:
        y_src = str(year_src or "").strip()
        citations.append({"field":"year_established_int","snippet":str(year_int),"source_column":"yearestablished" if y_src and y_src not in ("","null","None") else "description","extraction_method":f"year_extraction_{year_conf or 'unknown'}","confidence":float(conf_map_year.get(str(year_conf or ""),0.70))})
    if capacity_int is not None:
        c_src = str(capacity_src or "").strip()
        citations.append({"field":"capacity_int","snippet":str(capacity_int),"source_column":"capacity" if c_src and c_src not in ("","null","None") else "description","extraction_method":f"capacity_extraction_{capacity_conf or 'unknown'}","confidence":float(conf_map_capacity.get(str(capacity_conf or ""),0.65))})
    if doctors_int is not None:
        d_src = str(doctors_src or "").strip()
        citations.append({"field":"number_doctors_int","snippet":str(doctors_int),"source_column":"numberdoctors" if d_src and d_src not in ("","null","None") else "description","extraction_method":f"doctor_extraction_{doctors_conf or 'unknown'}","confidence":float(conf_map_doctors.get(str(doctors_conf or ""),0.65))})
    if not citations and desc and len(str(desc).strip()) >= 30:
        citations.append({"field":"description","snippet":str(desc).strip()[:100],"source_column":"description","extraction_method":"description_fallback","confidence":float(0.50)})
    return citations

build_citations_udf = F.udf(_build_citations, CITATION_SCHEMA)
df = df.withColumn("citations", build_citations_udf(
    F.col("procedure_parsed"),F.col("equipment_parsed"),F.col("capability_parsed"),F.col("specialties_parsed"),F.col("description"),
    F.col("procedure_count"),F.col("equipment_count"),
    F.col("year_established_int"),F.col("capacity_int"),F.col("number_doctors_int"),
    F.col("yearestablished"),F.col("capacity"),F.col("numberdoctors"),
    F.col("year_confidence"),F.col("capacity_confidence"),F.col("doctors_confidence"),
))

# COMMAND ----------
# MAGIC %md ## 22. Completeness Score (capped at 1.0)

# COMMAND ----------

df = df \
    .withColumn("contact_completeness",
        (F.when(F.col("phone_quality_score") >= 0.7, 0.50).otherwise(0.0))
        + (F.when(F.col("email_validity") == True, 0.25).otherwise(0.0))
        + (F.when(F.col("website_primary").isNotNull(), 0.25).otherwise(0.0))
    ) \
    .withColumn("clinical_completeness",
        (F.when(F.col("specialty_count") >= 3, 0.30).when(F.col("specialty_count") >= 1, 0.18).otherwise(0.0))
        + (F.when(F.col("procedure_count") >= 5, 0.25).when(F.col("procedure_count") >= 1, 0.15).otherwise(0.0))
        + (F.when(F.col("equipment_count") >= 3, 0.20).when(F.col("equipment_count") >= 1, 0.12).otherwise(0.0))
        + (F.when(F.col("capability_count") >= 3, 0.15).when(F.col("capability_count") >= 1, 0.10).otherwise(0.0))
        + (F.when(F.col("has_description"), 0.10).otherwise(0.0))
    ) \
    .withColumn("geo_completeness", F.coalesce(F.col("geo_quality_score"), F.lit(0.0))) \
    .withColumn("address_completeness", F.coalesce(F.col("address_confidence"), F.lit(0.0)))

df = df.withColumn("silver_data_quality_score",
    F.least(
        (F.col("clinical_completeness") * 0.45)
        + (F.col("contact_completeness") * 0.20)
        + (F.col("geo_completeness") * 0.20)
        + (F.col("address_completeness") * 0.10)
        + (F.when(F.col("is_operational"), 0.05).otherwise(0.0)),
        F.lit(1.0)
    ).cast(FloatType())
)

df = df.withColumn("data_completeness_score", F.col("silver_data_quality_score"))

# COMMAND ----------
# MAGIC %md ## 23. Row Quality Flags (v12-10: contradiction flags; v12-11: null → [])

# COMMAND ----------

ROW_QUALITY_FLAGS_SCHEMA = ArrayType(StringType())

def _safe_flags(flags):
    if not flags: return []
    seen, clean = set(), []
    for f in flags:
        if f is None: continue
        try:
            s = str(f).strip()[:200]
            if s and s not in seen: seen.add(s); clean.append(s)
        except: continue
    return clean

# v12-10: Location consistency check — city vs region
_CITY_REGION_EXPECTED: Dict[str, str] = {
    k: v for k, v in CITY_TO_REGION.items()
}

def _build_quality_flags(
    name, city, state, region, ftype, ftype_internal, org_type,
    has_proc, has_equip, has_cap, has_spec,
    has_contact, completeness, cap_arr,
    year_int, capacity_int, doctors_int,
    country_code, country, lat, lon,
    addr1, addr2, addr3, geo_src,
    specialty_arr, facility_type_confidence,
    is_operational, has_physical_address,
    doctors_conf, capacity_conf,
    operator_type, source_trust,
    bare_website, operational_status,
    # v12-10: new inputs for contradiction detection
    org_category, address_type,
):
    flags = []
    ft_clean = str(ftype_internal or "").strip().lower()
    op_clean = str(operator_type or "").strip().lower()
    org_str  = str(org_type or "").strip().lower()
    ftype_str = str(ftype or "").strip().lower()
    city_lower = str(city or "").strip().lower()
    region_str = str(region or "").strip()
    org_cat_str = str(org_category or "").strip().lower()

    # ── HIGH ─────────────────────────────────────────────────────────────────
    if not name or str(name).strip() in ("","null","None","UNKNOWN_FACILITY"):
        flags.append("HIGH:missing_name")

    if not has_proc and not has_equip and not has_cap and not has_spec:
        flags.append("HIGH:no_clinical_data")

    if org_str == "ngo" and ftype_str in ("hospital","clinic","pharmacy","dentist"):
        flags.append("HIGH:ftype_orgtype_mismatch")

    if ft_clean == "pharmacy" and cap_arr:
        icu_re = re.compile(r"\b(icu|intensive\s+care|inpatient\s+ward|operating\s+theatre|nicu)\b", re.I)
        if any(icu_re.search(str(c)) for c in cap_arr):
            flags.append("HIGH:anomaly_pharmacy_claims_icu_or_ward")

    if ft_clean == "hospital" and capacity_int is not None and capacity_int < 5:
        flags.append("HIGH:anomaly_hospital_very_low_capacity")

    if doctors_int is not None and (doctors_int < 1 or doctors_int > 500):
        flags.append("HIGH:implausible_doctor_count")

    if capacity_int is not None and (capacity_int < 5 or capacity_int > 5000):
        flags.append("HIGH:implausible_bed_capacity")

    if year_int is not None and (year_int < 1850 or year_int > 2026):
        flags.append("HIGH:suspicious_year")

    if doctors_int is not None and capacity_int is not None and capacity_int > 0 and doctors_int > capacity_int * 3:
        flags.append("HIGH:doctors_exceed_3x_beds_anomaly")

    if operational_status == "closed":
        flags.append("HIGH:facility_is_closed_or_defunct")

    if op_clean == "public" and ft_clean in ("pharmacy","dentist","alternative_medicine"):
        flags.append("HIGH:operator_facility_type_conflict")
    if op_clean == "private" and "teaching hospital" in str(name or "").lower():
        flags.append("HIGH:operator_facility_type_conflict")

    if ft_clean == "hospital":
        if str(doctors_conf or "") == "not_extracted":
            flags.append("HIGH:hospital_missing_doctor_count")
        if str(capacity_conf or "") == "not_extracted":
            flags.append("HIGH:hospital_missing_bed_capacity")

    # v12-10: CONTRADICTION FLAGS ─────────────────────────────────────────────
    # City vs region mismatch
    if city_lower and city_lower in _CITY_REGION_EXPECTED and region_str not in ("Unknown",""):
        expected_region = _CITY_REGION_EXPECTED[city_lower]
        if region_str != expected_region:
            flags.append(f"HIGH:city_region_mismatch ({city_lower} → expected {expected_region}, got {region_str})")

    # Government org category but private operator type
    if org_cat_str == "government" and op_clean == "private":
        flags.append("HIGH:government_org_but_private_operator")

    # Private org category but public operator type
    if org_cat_str in ("private",) and op_clean == "public":
        flags.append("MEDIUM:private_org_but_public_operator")

    # Address type vs has_physical_address mismatch
    if str(address_type or "") == "physical" and not has_physical_address:
        flags.append("MEDIUM:address_type_physical_but_flag_false")

    # ── MEDIUM ────────────────────────────────────────────────────────────────
    if ft_clean == "alternative_medicine":
        flags.append("MEDIUM:alternative_medicine_facility")

    if (not city or str(city).strip() in ("","null","None")) and \
       (not state or str(state).strip() in ("","null","None")):
        flags.append("MEDIUM:missing_location")

    if str(region or "") == "Unknown":
        flags.append("MEDIUM:unknown_region")

    if not ft_clean:
        flags.append("MEDIUM:missing_facility_type")

    if not has_contact:
        flags.append("MEDIUM:missing_contact")

    ghana_known = str(country or "").strip().lower() in ("ghana","gh")
    if ghana_known and (not country_code or str(country_code).strip() in ("","null","None")):
        flags.append("MEDIUM:missing_country_code")

    if cap_arr:
        ADDR_SENT_RE = re.compile(r"(?i)(located\s+at|P\.O\.\s*box|GPS\s+(?:code|address)|suite\s+\d|tel(?:ephone)?\s*[:\-]|\+\d{7,}|http[s]?://)")
        if any(ADDR_SENT_RE.search(str(c)) for c in cap_arr):
            flags.append("MEDIUM:capability_contamination_risk")

    geo_src_str = str(geo_src or "")
    if geo_src_str == "country_centroid":   flags.append("MEDIUM:country_centroid_geocode_only")
    elif geo_src_str == "region_centroid":  flags.append("MEDIUM:region_centroid_geocode")

    addr_text = " ".join(str(x or "") for x in [addr1,addr2,addr3,city,state])
    if len(addr_text.strip()) < 10:
        flags.append("MEDIUM:very_vague_address")

    if not has_physical_address:
        flags.append("MEDIUM:no_physical_address")

    specs = list(specialty_arr) if specialty_arr else []
    if ft_clean == "pharmacy" and any(s in specs for s in ["generalSurgery","cardiacSurgery","criticalCareMedicine"]):
        flags.append("MEDIUM:anomaly_specialty_facility_mismatch")

    if ft_clean == "eye_clinic" and specs and "ophthalmology" not in specs:
        flags.append("MEDIUM:anomaly_eye_clinic_missing_ophthalmology_specialty")

    try:
        if facility_type_confidence is not None and float(facility_type_confidence) < 0.50 and ft_clean:
            flags.append("MEDIUM:low_facility_type_confidence")
    except Exception: pass

    if ft_clean in ("clinic","doctor") and str(doctors_conf or "") == "not_extracted":
        flags.append("MEDIUM:clinic_missing_doctor_count")

    if bare_website:
        flags.append("MEDIUM:bare_website_domain_only")

    if operational_status == "unknown":
        flags.append("MEDIUM:operational_status_unknown")

    if not op_clean or op_clean == "":
        flags.append("MEDIUM:missing_operator_type")

    # ── LOW ───────────────────────────────────────────────────────────────────
    if (completeness or 0.0) < 0.25:
        flags.append("LOW:low_completeness")

    if not has_proc and not has_equip and not has_spec and has_cap:
        flags.append("LOW:capability_only_no_procedures_or_equipment")

    if str(name or "").strip() == "UNKNOWN_FACILITY":
        flags.append("LOW:name_was_address_string")

    if str(source_trust or "") == "social_metadata":
        flags.append("LOW:social_metadata_source")

    return _safe_flags(flags)

build_quality_flags_udf = F.udf(_build_quality_flags, ROW_QUALITY_FLAGS_SCHEMA)

# Safety coalescing before UDF
df = df \
    .withColumn("capacity_confidence",     F.coalesce(F.col("capacity_confidence"), F.lit("not_extracted"))) \
    .withColumn("doctors_confidence",      F.coalesce(F.col("doctors_confidence"), F.lit("not_extracted"))) \
    .withColumn("facility_type_clean",     F.coalesce(F.col("facility_type_clean"), F.lit(""))) \
    .withColumn("operatorTypeId",          F.coalesce(F.col("operatorTypeId"), F.lit(""))) \
    .withColumn("geo_source",              F.coalesce(F.col("geo_source"), F.lit("country_centroid"))) \
    .withColumn("operational_status",      F.coalesce(F.col("operational_status"), F.lit("unknown"))) \
    .withColumn("has_bare_website_domain", F.coalesce(F.col("has_bare_website_domain"), F.lit(False))) \
    .withColumn("organization_category",   F.coalesce(F.col("organization_category"), F.lit(""))) \
    .withColumn("address_type",            F.coalesce(F.col("address_type"), F.lit("unknown"))) \
    .withColumn("address_confidence",      F.coalesce(F.col("address_confidence"), F.lit(0.0))) \
    .withColumn("geo_quality_score",       F.coalesce(F.col("geo_quality_score"), F.lit(0.0)))

df = df \
    .withColumn("row_quality_flags",
        build_quality_flags_udf(
            F.col("name"), F.col("address_city"), F.col("address_stateOrRegion"),
            F.col("region_normalised"), F.col("facilityTypeId"), F.col("facility_type_clean"),
            F.col("organization_type_clean"),
            F.col("has_procedures"), F.col("has_equipment"), F.col("has_capabilities"), F.col("has_specialties"),
            F.col("has_contact"), F.col("data_completeness_score"), F.col("capability_parsed"),
            F.col("year_established_int"), F.col("capacity_int"), F.col("number_doctors_int"),
            F.col("address_countryCode"), F.col("address_country"),
            F.col("latitude"), F.col("longitude"),
            F.col("address_line1"), F.col("address_line2"), F.col("address_line3"),
            F.col("geo_source"), F.col("specialties_parsed"), F.col("facility_type_confidence"),
            F.col("is_operational"), F.col("has_physical_address"),
            F.col("doctors_confidence"), F.col("capacity_confidence"),
            F.col("operatorTypeId"), F.col("source_trust"),
            F.col("has_bare_website_domain"), F.col("operational_status"),
            F.col("organization_category"), F.col("address_type"),
        )) \
    .withColumn("quality_flag_count", F.size(F.col("row_quality_flags")))

# v12-11: Null safety — ensure never NULL (always [] or a populated list)
df = df.withColumn("row_quality_flags",
    F.when(F.col("row_quality_flags").isNull(), F.array().cast(ArrayType(StringType())))
     .otherwise(F.col("row_quality_flags")))

print("Quality flags applied (v12: contradiction flags + null-safe).")
df.select(F.explode_outer("row_quality_flags").alias("flag")) \
  .groupBy("flag").count().orderBy(F.desc("count")).show(70, truncate=False)

# COMMAND ----------
# MAGIC %md ## 24. Re-evaluate RAG Readiness

# COMMAND ----------

df = df.withColumn(
    "is_rag_ready",
    F.col("is_rag_ready") &
    ~F.arrays_overlap(F.col("row_quality_flags"), F.array(
        F.lit("HIGH:no_clinical_data"), F.lit("HIGH:missing_name"),
        F.lit("MEDIUM:capability_contamination_risk"),
        F.lit("MEDIUM:country_centroid_geocode_only"),
        F.lit("HIGH:facility_is_closed_or_defunct"),
    ))
)

rag_final = df.filter(F.col("is_rag_ready")).count()
total     = df.count()
print(f"RAG-ready (post-flag gate): {rag_final:,} / {total:,} ({rag_final/total*100:.1f}%)")

# COMMAND ----------
# MAGIC %md ## 25. Deduplication (v12-8: campus-aware; v12-9: provenance flags; v12-13: source_trust in sort)

# COMMAND ----------

# v12-8: Campus/sub-facility detection — these should NOT trigger canonical override
_SUB_FACILITY_RE = re.compile(
    r"(?i)\b("
    r"ward\s+of|centre\s+of|center\s+of|department\s+of|unit\s+of|wing\s+of"
    r"|section\s+of|annex\s+of|campus\s+of|outpost\s+of"
    r"|national\s+cardiothoracic|reproductive\s+health\s+centre|children['s]*\s+hospital"
    r"|mssi\b|research\s+centre|training\s+centre|specialist\s+centre"
    r")\b"
)

# v12-8: Only EXACT main-name canonical matches (not substring hospital mentions)
_CANONICAL_NAME_MAP_EXACT: Dict[str, str] = {
    "korle bu teaching hospital":     "CANONICAL_KORLE_BU",
    "korle-bu teaching hospital":     "CANONICAL_KORLE_BU",
    "komfo anokye teaching hospital": "CANONICAL_KATH",
    "komfo anokye teaching hospital (kath)": "CANONICAL_KATH",
    "kath":                           "CANONICAL_KATH",
    "accra psychiatric hospital":     "CANONICAL_ACCRA_PSYCH",
    "ridge hospital":                 "CANONICAL_RIDGE",
    "37 military hospital":           "CANONICAL_37_MILITARY",
    "cape coast teaching hospital":   "CANONICAL_CCTH",
    "tamale teaching hospital":       "CANONICAL_TAMALE_TEACH",
    "ho teaching hospital":           "CANONICAL_HO_TEACH",
    "effia nkwanta regional hospital":"CANONICAL_EFFIA_NKWANTA",
    "bolgatanga regional hospital":   "CANONICAL_BOLGA_REGIONAL",
    "wa regional hospital":           "CANONICAL_WA_REGIONAL",
    "koforidua regional hospital":    "CANONICAL_KOF_REGIONAL",
}

_LEGAL_SUFFIX_RE = re.compile(
    r"\b(?:ltd|limited|llc|inc|plc|co\b||||gh?)\b",
    re.I
)

def _geo_grid_cell(lat_s, lon_s, precision=0.002):
    try:
        lat = float(lat_s or ""); lon = float(lon_s or "")
        return f"{round(lat/precision)*precision:.3f}_{round(lon/precision)*precision:.3f}"
    except: return None

def _make_dedup_key_v13(name_val, city_val, addr1_val, addr2_val, org_type_val, phone_val, lat_s, lon_s):
    name_lower = str(name_val or "").strip().lower()

    # v12-8: Skip canonical override for sub-facilities
    if not _SUB_FACILITY_RE.search(name_lower):
        for canonical_name, canonical_key in _CANONICAL_NAME_MAP_EXACT.items():
            # Must be an exact match or very close (starts-with / ends-with)
            if name_lower == canonical_name:
                return f"CANONICAL|{canonical_key}", "canonical_exact_name", canonical_key

    parts = []
    reason_parts = []
    org_prefix = "NGO" if str(org_type_val or "").lower() == "ngo" else "FAC"

    for i, v in enumerate([name_val, city_val, addr1_val, addr2_val]):
        s = str(v or "").strip().lower()
        s = re.sub(r"[^\w\s]"," ",s)
        if i == 0: s = _LEGAL_SUFFIX_RE.sub(" ",s)
        s = re.sub(r"\s+"," ",s).strip()
        if s and s not in ("null","none","nan","","unknown facility"):
            parts.append(s)
            if i == 0: reason_parts.append("name")
            elif i == 1: reason_parts.append("city")
            else: reason_parts.append("address")

    phone_s = str(phone_val or "").strip()
    if phone_s and phone_s.startswith("+") and len(phone_s) >= 10:
        parts.append(f"ph:{phone_s}")
        reason_parts.append("phone")

    geo_cell = _geo_grid_cell(str(lat_s or ""), str(lon_s or ""))
    if geo_cell:
        parts.append(f"geo:{geo_cell}")
        reason_parts.append("geo")

    if not parts:
        return None, None, None
    reason = "+".join(sorted(set(reason_parts))) if reason_parts else "unknown"
    dedup_key = f"{org_prefix}|" + "|".join(parts)
    return dedup_key, reason, None

DEDUP_SCHEMA = StructType([
    StructField("dedup_key", StringType(), True),
    StructField("dedup_merge_reason", StringType(), True),
    StructField("canonical_root_id", StringType(), True),
])

make_dedup_key_udf = F.udf(
    lambda n,c,a1,a2,o,p,lat,lon: _make_dedup_key_v13(n,c,a1,a2,o,p,lat,lon),
    DEDUP_SCHEMA,
)

df = df.withColumn("_dedup",
    make_dedup_key_udf(
        F.col("name"),F.col("city_clean"),F.col("address_line1"),F.col("address_line2"),
        F.col("organization_type_clean"),F.col("official_phone"),
        F.col("latitude"),F.col("longitude"),
    ))
df = df \
    .withColumn("dedup_key", F.col("_dedup.dedup_key")) \
    .withColumn("dedup_merge_reason", F.col("_dedup.dedup_merge_reason")) \
    .withColumn("canonical_root_id", F.col("_dedup.canonical_root_id")) \
    .drop("_dedup")

df = df.withColumn("canonical_root_id",
    F.when(F.col("dedup_key").startswith("CANONICAL|"),
           F.regexp_extract(F.col("dedup_key"), r"CANONICAL\|(.+)$", 1))
     .otherwise(F.md5(F.col("dedup_key"))))

# v12-9: Provenance flags for canonical rows
df = df \
    .withColumn("is_generated_canonical", F.col("dedup_key").startswith("CANONICAL|")) \
    .withColumn("canonical_source_group",
        F.when(F.col("dedup_key").startswith("CANONICAL|"),
               F.regexp_extract(F.col("dedup_key"), r"CANONICAL\|(.+)$", 1))
         .otherwise(F.lit(None).cast(StringType())))

# Cluster size
dedup_cluster_counts = df.groupBy("dedup_key").agg(F.count("*").alias("dedup_cluster_size"))
df = df.join(dedup_cluster_counts, on="dedup_key", how="left")

# v12-13: Survivor ordering — source_trust matters
_source_trust_rank = (
    F.when(F.col("source_trust") == "structured",      0)
     .when(F.col("source_trust") == "text_extracted",  1)
     .when(F.col("source_trust") == "inferred",        2)
     .when(F.col("source_trust") == "social_metadata", 3)
     .otherwise(4)
)

_dedup_window = Window.partitionBy("dedup_key").orderBy(
    F.desc("data_completeness_score"),
    F.asc("quality_flag_count"),
    _source_trust_rank.asc(),   # v12-13: structured beats social_metadata
    F.desc("procedure_count"), F.desc("equipment_count"), F.desc("specialty_count"), F.desc("capability_count"),
    F.when(F.col("geo_source").isin("static_city_dict","geopy_nominatim"), 0)
     .when(F.col("geo_source") == "text_extracted_city", 1)
     .when(F.col("geo_source") == "district_centroid",   2)
     .when(F.col("geo_source") == "region_centroid",     3)
     .otherwise(4).asc(),
    F.asc("row_hash"),
)

df = df \
    .withColumn("_dr",               F.row_number().over(_dedup_window)) \
    .withColumn("is_duplicate_survivor", F.col("_dr") == 1) \
    .drop("_dr")

@F.udf(ArrayType(StringType()))
def _add_duplicate_flag(flags, is_survivor):
    # v12-11: Always return a list, never None
    lst = list(flags) if flags else []
    if not is_survivor: lst = ["HIGH:duplicate_non_survivor"] + lst
    return lst

df = df \
    .withColumn("row_quality_flags", _add_duplicate_flag(F.col("row_quality_flags"), F.col("is_duplicate_survivor"))) \
    .withColumn("quality_flag_count", F.size(F.col("row_quality_flags")))

# v12-14: weak_facility_type threshold moved to < 0.45 (was < 0.40 in v11)
df = df.withColumn("row_quality_flags",
    F.when(F.col("facility_type_confidence") < 0.45,
           F.array_union(F.col("row_quality_flags"), F.array(F.lit("LOW:weak_facility_type"))))
     .otherwise(F.col("row_quality_flags")))

# Final null-safety sweep
df = df.withColumn("row_quality_flags",
    F.when(F.col("row_quality_flags").isNull(), F.array().cast(ArrayType(StringType())))
     .otherwise(F.col("row_quality_flags")))
df = df.withColumn("quality_flag_count", F.size(F.col("row_quality_flags")))

total_rows    = df.count()
survivors     = df.filter(F.col("is_duplicate_survivor")).count()
non_survivors = total_rows - survivors
print(f"Deduplication (v12-8: campus-aware canonical):")
print(f"  Total rows: {total_rows:,} | Unique survivors: {survivors:,} | Non-survivors: {non_survivors:,} ({non_survivors/total_rows*100:.1f}%)")

print("\nResidual clusters > 1:")
df.filter(F.col("dedup_cluster_size") > 1) \
  .select("dedup_key","dedup_cluster_size","name","city_clean","facility_type_clean") \
  .orderBy(F.desc("dedup_cluster_size")).show(15, truncate=80)

# Non-survivors will be split into an audit table at write time

# COMMAND ----------
# MAGIC %md ## 26. Canonical Field Overwrite

# COMMAND ----------

df = df \
    .withColumn("numberDoctors",
        F.coalesce(F.col("number_doctors_int").cast(IntegerType()), F.col("numberDoctors"))) \
    .withColumn("yearestablished",
        F.coalesce(F.col("year_established_int").cast(StringType()), F.col("yearestablished"))) \
    .withColumn("acceptsvolunteers",
        F.coalesce(
            F.when(F.col("accepts_volunteers_bool") == True, F.lit("true"))
             .when(F.col("accepts_volunteers_bool") == False, F.lit("false")),
            F.col("acceptsvolunteers"))) \
    .withColumn("capacity",
        F.coalesce(F.col("capacity_int").cast(StringType()), F.col("capacity")))

print("Canonical fields overwritten.")

# COMMAND ----------
# MAGIC %md ## 27. Contract Validation — Enforce PDF Enums Hard

# COMMAND ----------

df = df.withColumn("facilityTypeId",
    F.when(F.col("facilityTypeId").isin("hospital","clinic","pharmacy","dentist","doctor"), F.col("facilityTypeId"))
     .when(F.col("facilityTypeId") == "chps",                 F.lit("clinic"))
     .when(F.col("facilityTypeId") == "maternity_home",       F.lit("clinic"))
     .when(F.col("facilityTypeId") == "diagnostic_center",    F.lit("clinic"))
     .when(F.col("facilityTypeId") == "eye_clinic",           F.lit("clinic"))
     .when(F.col("facilityTypeId") == "alternative_medicine", F.lit("clinic"))
     .when(F.col("facilityTypeId").isNull(), F.lit("clinic"))
     .otherwise(F.lit("clinic"))
)

df = df.withColumn("operatorTypeId",
    F.when(F.lower(F.col("operatorTypeId")).isin("public","government"), F.lit("public"))
     .when(F.lower(F.col("operatorTypeId")).isin("private","ngo","faith-based","mission","company",""), F.lit("private"))
     .otherwise(F.lit("private"))
)

print("PDF enum enforcement applied.")

# COMMAND ----------
# MAGIC %md ## 28. Metadata + String Schema Enforcement

# COMMAND ----------

df = df \
    .withColumn("ingested_at",        F.to_timestamp(F.lit(datetime.now(timezone.utc).isoformat()))) \
    .withColumn("source_file",        F.lit("bronze_facilities_raw")) \
    .withColumn("dataset_version",    F.lit("v1")) \
    .withColumn("extraction_version", F.lit(cfg.EXTRACTION_VER)) \
    .withColumn("row_hash",
        F.when(F.col("row_hash").isNull(),
               F.md5(F.concat_ws("|", F.col("name"), F.col("address_city"), F.col("address_line1"), F.col("source_url"))))
         .otherwise(F.col("row_hash")))

STRING_ENFORCEMENT_COLS = [
    "phone_numbers","official_phone","pk_unique_id","pk_unique_id_int",
    "officialWebsite","facebooklink","twitterlink","instagramlink",
    "linkedinlink","email","row_hash","source_url","unique_id",
    "website_primary","phone_country_code","org_category_raw","source_type",
]
for col_name in STRING_ENFORCEMENT_COLS:
    if col_name in df.columns:
        df = df.withColumn(col_name, F.col(col_name).cast(StringType()))

print("Metadata + string enforcement done.")

# COMMAND ----------
# MAGIC %md ## 29. Contract Validation Report (v12)

# COMMAND ----------

total_rows = df.count()

print("=" * 70)
print("  PRE-WRITE CONTRACT VALIDATION (v12)")
print("=" * 70)

validation_errors = []

# [1] facilityTypeId
invalid_ftype = df.filter(F.col("facilityTypeId").isNotNull() & ~F.col("facilityTypeId").isin(*list(PDF_FACILITY_TYPE_ENUM))).count()
status = "✅" if invalid_ftype == 0 else f"❌ {invalid_ftype}"
print(f"  [1] facilityTypeId enum: {status}")
if invalid_ftype > 0: validation_errors.append(f"facilityTypeId enum: {invalid_ftype} rows")

# [2] operatorTypeId
invalid_optype = df.filter(F.col("operatorTypeId").isNotNull() & ~F.col("operatorTypeId").isin("public","private")).count()
op_null_pct = df.filter(F.col("operatorTypeId").isNull()).count() / total_rows * 100
status = "✅" if invalid_optype == 0 else f"❌ {invalid_optype}"
print(f"  [2a] operatorTypeId enum: {status}")
print(f"  [2b] operatorTypeId null rate: {op_null_pct:.1f}% (target = 0%)")
if invalid_optype > 0: validation_errors.append(f"operatorTypeId enum: {invalid_optype} rows")

# [3] Specialties
@F.udf(ArrayType(StringType()))
def find_invalid_specs(specs):
    if not specs: return []
    return [s for s in specs if s not in FDR_SPECIALTY_VOCAB]
invalid_spec_rows = df.withColumn("_bad", find_invalid_specs(F.col("specialties_parsed"))).filter(F.size(F.col("_bad")) > 0).count()
print(f"  [3] Specialty FDR taxonomy: {'✅' if invalid_spec_rows==0 else f'⚠️ {invalid_spec_rows} rows'}")

# [4] countryCode
missing_cc = df.filter((F.col("address_country") == "Ghana") & F.col("address_countryCode").isNull()).count()
print(f"  [4] address_countryCode GH: {'✅' if missing_cc==0 else f'⚠️ {missing_cc}'}")

# [5] official_phone E.164
invalid_phone = df.filter(F.col("official_phone").isNotNull() & ~F.col("official_phone").startswith("+")).count()
phone_coverage = df.filter(F.col("official_phone").isNotNull()).count() / total_rows * 100
print(f"  [5a] official_phone E.164: {'✅' if invalid_phone==0 else f'❌ {invalid_phone}'}")
print(f"  [5b] official_phone coverage: {phone_coverage:.1f}% (target ≥ 40%)")
if invalid_phone > 0: validation_errors.append(f"E.164: {invalid_phone} rows")

# [6] Kasoa → Central
kasoa_wrong = df.filter((F.lower(F.col("city_clean")) == "kasoa") & (F.col("region_normalised") != "Central")).count()
print(f"  [6] Kasoa → Central: {'✅' if kasoa_wrong==0 else f'❌ {kasoa_wrong}'}")
if kasoa_wrong > 0: validation_errors.append(f"Kasoa: {kasoa_wrong} rows")

# [7] Hallucination
@F.udf(BooleanType())
def has_hallucination(caps):
    if not caps: return False
    _H = re.compile(r"(?i)(Wait\s*[-–]\s*we\s+should\s+not|As\s+an\s+AI|I\s+am\s+an\s+AI)")
    return any(_H.search(str(c)) for c in caps)
halluc_rows = df.filter(has_hallucination(F.col("capability_parsed"))).count()
print(f"  [7] No hallucination in capabilities: {'✅' if halluc_rows==0 else f'❌ {halluc_rows}'}")
if halluc_rows > 0: validation_errors.append(f"Hallucination: {halluc_rows} rows")

# [8] completeness score overflow
score_overflow = df.filter(F.col("data_completeness_score") > 1.0).count()
print(f"  [8] data_completeness_score ≤ 1.0: {'✅' if score_overflow==0 else f'❌ {score_overflow}'}")
if score_overflow > 0: validation_errors.append(f"Score overflow: {score_overflow}")

# [9] Unknown region
unknown_region_pct = df.filter(F.col("region_normalised") == "Unknown").count() / total_rows * 100
print(f"  [9] Unknown region rate: {unknown_region_pct:.1f}% (target < 10%)")

# [10] Country centroid
cc_count = df.filter(F.col("geo_source") == "country_centroid").count()
print(f"  [10] Country centroid rows: {cc_count} (target < 20; geopy rescue applied)")

# [11] has_physical_address must be discriminative
no_physical = df.filter(~F.col("has_physical_address")).count()
status = "✅" if no_physical > 0 else "❌ ZERO rows without physical address — not discriminative"
print(f"  [11] has_physical_address discriminative: {status} ({no_physical} rows = False)")
if no_physical == 0: validation_errors.append("has_physical_address not discriminative: always True")

# [12] Operational status
op_open    = df.filter(F.col("operational_status") == "open").count()
op_closed  = df.filter(F.col("operational_status") == "closed").count()
op_unknown = df.filter(F.col("operational_status") == "unknown").count()
print(f"  [12] operational_status: open={op_open} closed={op_closed} unknown={op_unknown} (target: open >> unknown)")

# [13] Dedup large clusters
large_clusters = df.filter(F.col("dedup_cluster_size") > 5).count()
print(f"  [13] Large dedup clusters (>5): {'✅' if large_clusters==0 else f'⚠️ {large_clusters}'}")

# [14] Confidence fields populated
conf_null = df.filter(F.col("capacity_confidence").isNull() | (F.col("capacity_confidence") == "")).count()
print(f"  [14] capacity_confidence populated: {'✅' if conf_null==0 else f'⚠️ {conf_null}'}")

# [15] v12 new fields
print(f"  [15] quality_flags null rows: {df.filter(F.col('row_quality_flags').isNull()).count()} (target = 0)")
print(f"  [16] city_clean null rows: {df.filter(F.col('city_clean').isNull()).count()}")
print(f"  [17] org category coverage: {df.filter(F.col('organization_category').isNotNull()).count()}/{total_rows}")
print(f"  [18] address_type coverage: {df.filter(F.col('address_type').isNotNull()).count()}/{total_rows}")
cc_city = df.filter(F.col("is_generated_canonical") == True).count()
print(f"  [19] canonical provenance rows: {cc_city}")

print()
if validation_errors:
    print(f"❌  {len(validation_errors)} HARD violations:")
    for e in validation_errors: print(f"     - {e}")
    assert len(validation_errors) == 0, f"Contract gate failed: {validation_errors}"
else:
    print("✅  All hard contract checks passed. Safe to write.")
print("=" * 70)

# COMMAND ----------
# MAGIC %md ## 30. Final Column Selection & Write

# COMMAND ----------

SILVER_COLUMNS = [
    # Bronze passthrough
    "unique_id","source_url","name","pk_unique_id","mongo_db",
    "specialties","procedure","equipment","capability",
    "organization_type","content_table_id","phone_numbers",
    "email","websites","officialWebsite","yearestablished","acceptsvolunteers",
    "facebooklink","twitterlink","linkedinlink","instagramlink","logo",
    "address_line1","address_line2","address_line3",
    "address_city","address_stateOrRegion","address_zipOrPostcode",
    "address_country","address_countryCode","countries",
    "missionstatement","missionstatementlink","organizationdescription",
    "facilityTypeId","operatorTypeId","affiliationtypeids",
    "description","area","numberDoctors","capacity",
    "ingested_at","source_file","dataset_version","country_scope","row_hash",
    # Parsed arrays
    "specialties_parsed","procedure_parsed","equipment_parsed","capability_parsed",
    "phone_numbers_parsed","affiliationtypeids_parsed","countries_parsed","websites_parsed",
    # Enriched scalar
    "official_phone","area_int",
    "year_established_int","year_confidence",
    "accepts_volunteers_bool","pk_unique_id_int",
    # Raw preserved
    "facility_type_raw","operator_type_raw",
    # Facility type
    "facility_type_clean","facility_type_clean_pdf","facility_type_confidence",
    # Organization
    "organization_type_clean","organization_category","org_category_raw","org_category_confidence",
    "is_ngo","is_faith_based","is_government",
    # Location
    "city_clean","region_normalised","region_source","region_confidence",
    "latitude","longitude","geo_source","geo_confidence","geo_precision_tier","geo_quality_score",
    "postal_address","has_physical_address","address_type","address_confidence",
    # Capacity/doctors
    "capacity_int","capacity_confidence","number_doctors_int","doctors_confidence",
    "capacity_known","number_doctors_known","year_established_known","area_known",
    # Operational
    "operational_status","is_operational",
    # Source quality
    "source_trust","source_type","source_trust_score","has_bare_website_domain",
    # Contact quality
    "phone_country_code","phone_quality_score","email_validity","email_is_generic","email_quality_score",
    "website_primary","website_is_social_only",
    # Boolean flags
    "has_procedures","has_equipment","has_capabilities","has_specialties",
    "is_hospital","is_clinic","is_eye_clinic","is_maternity_home","is_chps","is_diagnostic_center",
    "is_pharmacy","is_dentist","is_alternative_medicine",
    "has_description","has_contact",
    # Counts
    "procedure_count","equipment_count","capability_count","specialty_count",
    # RAG
    "doc_text","doc_text_length","is_rag_ready",
    # Quality
    "citations","row_quality_flags","quality_flag_count","data_completeness_score",
    "contact_completeness","clinical_completeness","geo_completeness","address_completeness","silver_data_quality_score",
    # Dedup
    "dedup_key","dedup_merge_reason","canonical_root_id","dedup_cluster_size","is_duplicate_survivor",
    # v12-9: Provenance
    "is_generated_canonical","canonical_source_group",
    # Version
    "extraction_version",
]

available  = set(df.columns)
missing_c  = [c for c in SILVER_COLUMNS if c not in available]
if missing_c: print(f"⚠️  Missing (will skip): {missing_c}")

final_cols = [c for c in SILVER_COLUMNS if c in available]
df_silver  = df.filter(F.col("is_duplicate_survivor")).select(*final_cols)
df_audit_out = df.filter(~F.col("is_duplicate_survivor")).select(*final_cols)

print(f"Final silver columns : {len(final_cols)}")
print(f"Final silver rows    : {df_silver.count():,}")
print(f"Final audit rows     : {df_audit_out.count():,}")

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .option("delta.enableChangeDataFeed","true")
    .saveAsTable(cfg.SILVER)
)

(
    df_audit_out
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .option("delta.enableChangeDataFeed","true")
    .saveAsTable(cfg.SILVER_AUDIT)
)

print(f"✅  Silver table written: {cfg.SILVER}")
print(f"    Rows: {spark.table(cfg.SILVER).count():,}")
print(f"✅  Silver audit table written: {cfg.SILVER_AUDIT}")
print(f"    Rows: {spark.table(cfg.SILVER_AUDIT).count():,}")

# COMMAND ----------
# MAGIC %md ## 31. Post-Write Validation Report

# COMMAND ----------

sv    = spark.table(cfg.SILVER)
total = sv.count()

print("=" * 70)
print(f"  SILVER LAYER v12 — POST-WRITE VALIDATION REPORT")
print(f"  Table : {cfg.SILVER}  |  Rows : {total:,}")
print("=" * 70)

print("\n[1] facilityTypeId PDF enum compliance")
sv.groupBy("facilityTypeId").count().orderBy(F.desc("count")).show()

print("\n[2] facility_type_clean vs facility_type_clean_pdf")
sv.groupBy("facility_type_clean","facility_type_clean_pdf").count().orderBy(F.desc("count")).show(15)

print("\n[3] facility_type_raw override audit")
sv.groupBy("facility_type_raw","facility_type_clean").count() \
  .filter(F.col("facility_type_raw").isNotNull() & (F.col("facility_type_raw") != F.col("facility_type_clean"))) \
  .orderBy(F.desc("count")).show(20)

print("\n[4] operatorTypeId breakdown (target: 100% populated)")
sv.groupBy("operatorTypeId").count().show()
op_cov = sv.filter(F.col("operatorTypeId").isNotNull()).count()
print(f"    Populated: {op_cov:,}/{total:,} ({op_cov/total*100:.1f}%)")

print("\n[5] operational_status distribution")
sv.groupBy("operational_status").count().show()
op_open = sv.filter(F.col("operational_status") == "open").count()
op_closed = sv.filter(F.col("operational_status") == "closed").count()
op_unk = sv.filter(F.col("operational_status") == "unknown").count()
print(f"    open: {op_open}   closed: {op_closed}   unknown: {op_unk}")

print("\n[6] has_physical_address + address_type breakdown (v12)")
sv.groupBy("has_physical_address","address_type").count().show()
postal_ct   = sv.filter(F.col("postal_address").isNotNull()).count()
no_physical = sv.filter(~F.col("has_physical_address")).count()
print(f"    postal_address populated: {postal_ct} | No physical address: {no_physical}")

print("\n[7] source_trust distribution")
sv.groupBy("source_trust").count().orderBy(F.desc("count")).show()

print("\n[8] Geocoding breakdown (including geopy)")
sv.groupBy("geo_source").count().orderBy(F.desc("count")).show()
cc = sv.filter(F.col("geo_source") == "country_centroid").count()
print(f"    Country centroid remaining: {cc}")

print("\n[9] Numeric field coverage")
for c in ["number_doctors_int","capacity_int","year_established_int","area_int"]:
    cnt = sv.filter(F.col(c).isNotNull()).count()
    print(f"    {c:<28} {cnt:>5,}  ({cnt/total*100:.1f}%)")

print("\n[10] data_completeness_score bands")
max_score = sv.agg(F.max("data_completeness_score")).collect()[0][0]
print(f"    Max score: {max_score}")
sv.select(
    F.when(F.col("data_completeness_score") >= 0.75,"A: ≥0.75")
     .when(F.col("data_completeness_score") >= 0.50,"B: 0.50-0.75")
     .when(F.col("data_completeness_score") >= 0.25,"C: 0.25-0.50")
     .otherwise("D: <0.25").alias("band")
).groupBy("band").count().orderBy("band").show()

print("\n[11] Dedup summary (v12-8: campus-aware canonical)")
survivors_ct = sv.filter(F.col("is_duplicate_survivor")).count()
print(f"    Unique survivors: {survivors_ct:,} | Non-survivors: {total-survivors_ct:,}")
sv.groupBy("dedup_cluster_size").count().orderBy("dedup_cluster_size").show(10)

print("\n[12] RAG readiness")
rag_ct = sv.filter(F.col("is_rag_ready")).count()
print(f"    RAG-ready: {rag_ct:,}  ({rag_ct/total*100:.1f}%)")

print("\n[13] Quality flag distribution")
sv.select(F.explode_outer("row_quality_flags").alias("flag")) \
  .groupBy("flag").count().orderBy(F.desc("count")).show(70, truncate=False)

print("\n[14] organization_category breakdown (v12-3)")
sv.groupBy("organization_category","operatorTypeId").count().orderBy(F.desc("count")).show(20)

print("\n[15] v13 new field verification")
V13_FIELDS = [
    "organization_category","org_category_raw","org_category_confidence","is_ngo","is_faith_based","is_government",
    "facility_type_raw","operator_type_raw","facility_type_clean_pdf",
    "operational_status","source_trust","source_type","source_trust_score","has_bare_website_domain",
    "address_type","address_confidence","geo_precision_tier","geo_quality_score",
    "phone_country_code","phone_quality_score","email_validity","email_is_generic","email_quality_score",
    "website_primary","website_is_social_only",
    "dedup_merge_reason","canonical_root_id","dedup_cluster_size",
    "is_generated_canonical","canonical_source_group",
]
for field in V13_FIELDS:
    status = "✅" if field in sv.columns else "❌ MISSING"
    print(f"    {status}  {field}")
null_flags = sv.filter(F.col("row_quality_flags").isNull()).count()
print(f"    quality_flags null rows: {null_flags} (target = 0)")

print("\n[16] Region coverage")
sv.groupBy("region_normalised").count().orderBy(F.desc("count")).show(20)
unknown_r = sv.filter(F.col("region_normalised") == "Unknown").count()
print(f"    Unknown: {unknown_r:,} ({unknown_r/total*100:.1f}%)")

print("\n[17] Canonical dedup audit (v12-8: sub-facilities should be separate)")
sv.filter(F.col("dedup_key").startswith("CANONICAL|")) \
  .select("name","dedup_key","dedup_cluster_size","is_duplicate_survivor","data_completeness_score","is_generated_canonical") \
  .orderBy("dedup_key","is_duplicate_survivor") \
  .show(25, truncate=80)

print(f"\n{'='*70}")
print(f"  Silver v12 complete.  Extraction: {cfg.EXTRACTION_VER}")
print(f"  Run: {datetime.now(timezone.utc).isoformat()}")
print(f"{'='*70}")

# COMMAND ----------
# MAGIC %md ## 32. Sample Inspection

# COMMAND ----------

print("=== HIGH-COMPLETENESS SURVIVORS ===")
sv.filter(
    F.col("is_duplicate_survivor") & (F.col("data_completeness_score") >= 0.60) &
    F.col("has_procedures") & F.col("has_contact") & F.col("is_operational")
).select(
    "name","city_clean","region_normalised","facility_type_clean","facility_type_clean_pdf",
    "operatorTypeId","organization_category","operational_status","source_trust",
    "official_phone","procedure_count","equipment_count",
    "year_established_int","capacity_int","data_completeness_score","geo_source","is_rag_ready",
).orderBy(F.desc("data_completeness_score")).show(10, truncate=60)

print("\n=== CLOSED FACILITIES ===")
sv.filter(F.col("operational_status") == "closed") \
  .select("name","facility_type_clean","city_clean","is_rag_ready","operational_status") \
  .show(10, truncate=60)

print("\n=== HAS_PHYSICAL_ADDRESS + address_type BREAKDOWN ===")
sv.filter(~F.col("has_physical_address")) \
  .select("name","postal_address","address_line1","city_clean","has_physical_address","address_type") \
  .show(15, truncate=70)

print("\n=== SOURCE TRUST BY FACILITY TYPE ===")
sv.groupBy("facility_type_clean","source_trust").count() \
  .orderBy("facility_type_clean",F.desc("count")).show(30, truncate=40)

print("\n=== ORGANIZATION CATEGORY AUDIT ===")
sv.groupBy("organization_category","operatorTypeId","is_ngo","is_faith_based","is_government") \
  .count().orderBy(F.desc("count")).show(20)

print("\n=== CITY/REGION CONTRADICTION FLAGS ===")
sv.filter(F.array_contains(F.col("row_quality_flags"), "HIGH:city_region_mismatch") |
          F.array_contains(F.col("row_quality_flags"), "HIGH:government_org_but_private_operator")) \
  .select("name","city_clean","region_normalised","organization_category","operatorTypeId","row_quality_flags") \
  .show(10, truncate=80)

print("\n=== v12 complete ✅ ===")


"""
Run: 2026-05-07T15:18:25.273612+00:00
Spark: 4.1.0
Bronze rows    : 987
Bronze columns : 46
Name cleaning done. Rows: 987
JSON parser defined.
Phone normalization defined.
Array parsing complete.
Renames + country + email + URL protocol done.
Organization category done.
+---------------------+-----+
|organization_category|count|
+---------------------+-----+
|                 NULL|  632|
|           government|  131|
|          faith_based|   88|
|                  ngo|   75|
|      private_company|   61|
+---------------------+-----+

P.O. Box detection: 54 postal addresses | 1 rows lack meaningful physical address
+------------+-----+
|address_type|count|
+------------+-----+
|      postal|    1|
|    physical|  933|
|       mixed|   53|
+------------+-----+

Specialty normalization done.
Countries ISO done.
Organization description neutralized.
Junk filter applied.
Initial tri-state detection:
+------------------+-----+
|operational_status|count|
+------------------+-----+
|            closed|   16|
|              open|  139|
|           unknown|  832|
+------------------+-----+

Procedure/equipment mining done.
Numeric extraction done.
  number_doctors_int           non-null: 4  (text extraction only — source fields NULL in v0.3)
  capacity_int                 non-null: 31  (text extraction only — source fields NULL in v0.3)
  year_established_int         non-null: 65  (text extraction only — source fields NULL in v0.3)
  area_int                     non-null: 2  (text extraction only — source fields NULL in v0.3)
Region normalisation done. Unknown: 2.7%
Facility type inference done.
+--------------------+-----+
| facility_type_clean|count|
+--------------------+-----+
|            hospital|  462|
|              clinic|  278|
|                NULL|   64|
|             dentist|   51|
|   diagnostic_center|   44|
|alternative_medicine|   34|
|          eye_clinic|   28|
|      maternity_home|   17|
|            pharmacy|    7|
|                chps|    2|
+--------------------+-----+

operatorTypeId inference complete (v12: organization_category fed in):
+--------------+-----+
|operatorTypeId|count|
+--------------+-----+
|       private|  806|
|        public|  181|
+--------------+-----+

Null operatorTypeId after UDF: 0  (should be 0)
Source trust done:
+---------------+-----+
|   source_trust|count|
+---------------+-----+
| text_extracted|  278|
|       inferred|   12|
|     structured|  361|
|social_metadata|  321|
|        unknown|   15|
+---------------+-----+

Organisation type + city done.
city_clean nulls: 34
Static geocoding done:
+-------------------+-----+
|         geo_source|count|
+-------------------+-----+
|    region_centroid|  238|
|   static_city_dict|  723|
|  district_centroid|    1|
|text_extracted_city|    2|
|   country_centroid|   23|
+-------------------+-----+

⚠️  Geopy fallback skipped: No module named 'geopy'
Geocoding (all sources) summary:
+-------------------+-----+
|         geo_source|count|
+-------------------+-----+
|    region_centroid|  238|
|   static_city_dict|  723|
|  district_centroid|    1|
|text_extracted_city|    2|
|   country_centroid|   23|
+-------------------+-----+

Content flags done.
Operational status after service-evidence upgrade:
+------------------+-----+
|operational_status|count|
+------------------+-----+
|            closed|   16|
|              open|  726|
|           unknown|  245|
+------------------+-----+

RAG-ready (initial): 907 / 987
Quality flags applied (v12: contradiction flags + null-safe).
+------------------------------------------------------------------------------+-----+
|flag                                                                          |count|
+------------------------------------------------------------------------------+-----+
|HIGH:hospital_missing_doctor_count                                            |460  |
|HIGH:hospital_missing_bed_capacity                                            |435  |
|LOW:social_metadata_source                                                    |321  |
|MEDIUM:clinic_missing_doctor_count                                            |278  |
|MEDIUM:missing_contact                                                        |277  |
|MEDIUM:operational_status_unknown                                             |245  |
|MEDIUM:region_centroid_geocode                                                |238  |
|NULL                                                                          |75   |
|MEDIUM:missing_facility_type                                                  |64   |
|LOW:capability_only_no_procedures_or_equipment                                |60   |
|MEDIUM:low_facility_type_confidence                                           |47   |
|MEDIUM:missing_location                                                       |43   |
|MEDIUM:alternative_medicine_facility                                          |34   |
|MEDIUM:unknown_region                                                         |27   |
|HIGH:no_clinical_data                                                         |23   |
|MEDIUM:country_centroid_geocode_only                                          |23   |
|HIGH:facility_is_closed_or_defunct                                            |16   |
|MEDIUM:capability_contamination_risk                                          |14   |
|LOW:low_completeness                                                          |8    |
|HIGH:city_region_mismatch (accra → expected Greater Accra, got Central)       |6    |
|HIGH:government_org_but_private_operator                                      |5    |
|HIGH:ftype_orgtype_mismatch                                                   |3    |
|HIGH:city_region_mismatch (walewale → expected North East, got Northern)      |2    |
|MEDIUM:no_physical_address                                                    |1    |
|HIGH:city_region_mismatch (sefwi wiawso → expected Western, got Western North)|1    |
|HIGH:city_region_mismatch (berekum → expected Bono, got Brong-Ahafo)          |1    |
|HIGH:city_region_mismatch (atebubu → expected Bono East, got Brong-Ahafo)     |1    |
|HIGH:missing_name                                                             |1    |
|LOW:name_was_address_string                                                   |1    |
+------------------------------------------------------------------------------+-----+

RAG-ready (post-flag gate): 895 / 987 (90.7%)
Deduplication (v12-8: campus-aware canonical):
  Total rows: 987 | Unique survivors: 955 | Non-survivors: 32 (3.2%)

Residual clusters > 1:
+--------------------------------------------------------------------------------+------------------+-------------------------------------+----------+-------------------+
|                                                                       dedup_key|dedup_cluster_size|                                 name|city_clean|facility_type_clean|
+--------------------------------------------------------------------------------+------------------+-------------------------------------+----------+-------------------+
|                                                    CANONICAL|CANONICAL_KORLE_BU|                 5|           Korle Bu Teaching Hospital|     Accra|           hospital|
|                                                    CANONICAL|CANONICAL_KORLE_BU|                 5|           Korle Bu Teaching Hospital|     Accra|           hospital|
|                                                        CANONICAL|CANONICAL_KATH|                 5|       Komfo Anokye Teaching Hospital|    Kumasi|           hospital|
|                                                        CANONICAL|CANONICAL_KATH|                 5|       Komfo Anokye Teaching Hospital|    Kumasi|           hospital|
|                                                    CANONICAL|CANONICAL_KORLE_BU|                 5|           Korle Bu Teaching Hospital|     Accra|           hospital|
|                                                        CANONICAL|CANONICAL_KATH|                 5|Komfo Anokye Teaching Hospital (KATH)|    Kumasi|           hospital|
|                                                    CANONICAL|CANONICAL_KORLE_BU|                 5|           Korle Bu Teaching Hospital|     Accra|           hospital|
|                                                        CANONICAL|CANONICAL_KATH|                 5|       Komfo Anokye Teaching Hospital|    Kumasi|           hospital|
|                                                    CANONICAL|CANONICAL_KORLE_BU|                 5|           Korle-Bu Teaching Hospital|     Accra|           hospital|
|                                                        CANONICAL|CANONICAL_KATH|                 5|       Komfo Anokye Teaching Hospital|    Kumasi|           hospital|
|      FAC|accra medical centre|accra|161 ringway estate|osu osu|geo:5.604_-0.188|                 3|                 Accra Medical Centre|     Accra|           hospital|
|      FAC|accra medical centre|accra|161 ringway estate|osu osu|geo:5.604_-0.188|                 3|                 Accra Medical Centre|     Accra|           hospital|
|      FAC|accra medical centre|accra|161 ringway estate|osu osu|geo:5.604_-0.188|                 3|                 Accra Medical Centre|     Accra|           hospital|
|NGO|christian health association of ghana|accra|21 jubilee well street|labone...|                 2|Christian Health Association of Ghana|     Accra|                   |
|NGO|christian health association of ghana|accra|21 jubilee well street|labone...|                 2|Christian Health Association of Ghana|     Accra|                   |
+--------------------------------------------------------------------------------+------------------+-------------------------------------+----------+-------------------+
only showing top 15 rows
Canonical fields overwritten.
PDF enum enforcement applied.
Metadata + string enforcement done.
======================================================================
  PRE-WRITE CONTRACT VALIDATION (v12)
======================================================================
  [1] facilityTypeId enum: ✅
  [2a] operatorTypeId enum: ✅
  [2b] operatorTypeId null rate: 0.0% (target = 0%)
  [3] Specialty FDR taxonomy: ✅
  [4] address_countryCode GH: ✅
  [5a] official_phone E.164: ✅
  [5b] official_phone coverage: 66.2% (target ≥ 40%)
  [6] Kasoa → Central: ✅
  [7] No hallucination in capabilities: ✅
  [8] data_completeness_score ≤ 1.0: ✅
  [9] Unknown region rate: 2.7% (target < 10%)
  [10] Country centroid rows: 23 (target < 20; geopy rescue applied)
  [11] has_physical_address discriminative: ✅ (1 rows = False)
  [12] operational_status: open=726 closed=16 unknown=245 (target: open >> unknown)
  [13] Large dedup clusters (>5): ✅
  [14] capacity_confidence populated: ✅
  [15] quality_flags null rows: 0 (target = 0)
  [16] city_clean null rows: 34
  [17] org category coverage: 987/987
  [18] address_type coverage: 987/987
  [19] canonical provenance rows: 16

✅  All hard contract checks passed. Safe to write.
======================================================================
Final silver columns : 146
Final silver rows    : 955
Final audit rows     : 32
"""

In [0]:
%sql

Select * from virtue_foundation.ghana.silver_facilities_cleaned

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.silver_facilities_cleaned